# Learning Web Technology Fingerprints from Common Crawl
## Multi-Label Stack Detection, Fingerprint Ablation, and Temporal Analysis

**Project type:** applied ML research / passive public-web analysis (OSINT-adjacent, defensive only).

---

### 1. Project overview

This notebook builds an end-to-end machine-learning system that answers a single question:

> **Can a model identify the technologies a website is built with, from publicly archived page evidence alone?**

Given the HTML and page-level metadata of a web page, the system predicts a **technology stack** as a set of
independent probabilities:

```text
React              0.97
Next.js            0.94
Cloudflare         0.91
Google Analytics   0.89
Tailwind CSS       0.82
Stripe             0.76
```

This is a **multi-label classification** problem: a page may use dozens of technologies at once, and the labels are
neither mutually exclusive nor independent (Next.js implies React; WooCommerce implies WordPress).

The pipeline is:

```text
Common Crawl (public archive)
        |
        v  broad, domain-spread automated ingestion (WARC / WAT streaming)
technology evidence extraction  (DOM, scripts, resources, headers, cookies names, classes)
        |
        v  fingerprint registry + confidence-aware weak supervision
multi-label dataset with a trust mask
        |
        v  domain-disjoint train / val / test split
rule baseline -> classical ML -> hybrid neural model
        |
        v  fingerprint-removal ablations (A / B / C)
multi-label evaluation, threshold tuning, calibration, error analysis
        |
        v
technology co-occurrence -> stack clustering -> temporal generalisation
        -> adoption & transition analysis -> HTML-only inference interface
```

**What this project is not.** It is not a search engine, not a retrieval system, and not a document-ranking
pipeline. There is no query interface, no BM25, no NDCG/MRR, and no relevance ranking anywhere in this notebook.
The primary output is a *technology stack*, not a ranked list of documents. See section 38 for an explicit
delta against a prior Common Crawl retrieval project.

**Headline design decisions** (each justified where it appears):

| Decision | Choice | Why |
|---|---|---|
| Data source | Streamed **WARC** (default) with a **WAT** alternative | WARC gives HTTP response headers *and* full HTML in one pass, so DOM features and header features share one extractor |
| Sampling | Stride-sampled archive files + per-domain page caps | Broad domain coverage instead of query-driven relevance sampling |
| Labels | Confidence-aware rules -> `HIGH / MEDIUM / LOW / ABSTAIN` | Only `HIGH` becomes a positive; `MEDIUM`/`LOW` cells are **masked out** of training and evaluation |
| Leakage | Three explicit feature regimes (A/B/C) | Measures how much detection is tautological fingerprint echo vs. learned structure |
| Splitting | Domain-disjoint, greedy balanced | A page from `example.com` can never appear in two splits |

## 2. Research questions

| # | Question | Where it is answered |
|---|---|---|
| **RQ1** | Can ML identify web technologies from Common Crawl evidence? | S23 final test evaluation |
| **RQ2** | How does ML compare with direct fingerprint rules? | S16 rule baseline vs. S23 |
| **RQ3** | How much performance disappears when the obvious fingerprint signals are removed? | S20 ablation A/B/C |
| **RQ4** | Does the model generalise to unseen domains? | S14 domain-disjoint split + S23 |
| **RQ5** | Does it generalise across time? | S27 temporal evaluation |
| **RQ6** | Which technologies are easiest / hardest to identify? | S23 per-technology table, S24 error analysis |
| **RQ7** | Which technologies most commonly co-occur? | S25 co-occurrence, S26 clustering |

A secondary question runs through the whole notebook and is, in some ways, the most interesting one:

> **RQ3b — can server-side and infrastructure technologies (Nginx, Cloudflare, Express, ASP.NET) be predicted from
> client-side HTML structure alone, when the HTTP headers that reveal them are withheld?**

Those labels are derived from response headers, which are *not* present in pasted HTML at inference time. If a
model can recover them from DOM shape and resource graphs, that is genuine learned signal rather than fingerprint
echo. Regime C (structure-only) is designed to measure exactly this.

## 3. Ethical scope and safety posture

This project is **entirely passive**. It analyses an existing public archive; it never touches a live site.

**What this notebook does**

* Reads Common Crawl WARC/WAT archives over HTTPS from `data.commoncrawl.org` (a public, freely licensed corpus).
* Derives structural, lexical and resource-graph features from those archived pages.
* Trains and evaluates classifiers on those derived features.
* Accepts **user-supplied HTML** (pasted or uploaded) at inference time.

**What this notebook explicitly does not do**

| Not performed | Note |
|---|---|
| Live website crawling or fetching | The only hosts contacted are Common Crawl's data endpoint and package registries |
| Port scanning / host discovery | No network scanning code exists in this notebook |
| Vulnerability detection or exploitation | Out of scope; no CVE mapping, no version-to-vuln lookup |
| Authentication bypass / access-control bypass | No credentials are used or tested |
| Brute forcing or credential testing | None |
| Secret, API-key or password discovery | Cookie **values** are discarded at parse time; only cookie *names* are retained |
| Active reconnaissance of a target | The inference UI never visits a URL — it only reads HTML the user provides |

**Data minimisation.** The extractor keeps a bounded, derived summary of each page (counts, tag histograms,
resource domains, truncated text). It deliberately drops: `Set-Cookie` values, `Authorization`/`Cookie` request
headers, query-string values on resource URLs, and any inline script content beyond a short truncated sample used
for framework marker matching. Section 39 restates this as a formal safety report.

**Interpretation discipline.** Common Crawl is a *sample* of the public web with well-known biases. Every
prevalence figure in this notebook is a statement about the sample, never about "the Internet".

## 4. Environment setup

Installs only what is missing, then imports with graceful degradation. The notebook runs on **CPU or GPU**, and
every optional dependency (`torch`, `sentence-transformers`, `gradio`, `warcio`) has a documented fallback so a
missing package downgrades a component rather than breaking the run.

In [1]:
# --- dependency bootstrap -------------------------------------------------
import importlib, subprocess, sys, os, time

REQUIREMENTS = {
    "warcio":               "warcio",              # WARC/WAT record iteration
    "lxml":                 "lxml",                # fast HTML parsing
    "pyarrow":              "pyarrow",             # Parquet IO
    "pandas":               "pandas",
    "numpy":                "numpy",
    "sklearn":              "scikit-learn",
    "scipy":                "scipy",
    "matplotlib":           "matplotlib",
    "networkx":             "networkx",            # co-occurrence graph
    "joblib":               "joblib",
    "tqdm":                 "tqdm",
}
# Optional: absence downgrades a component but never breaks the notebook.
OPTIONAL = {
    "sentence_transformers": "sentence-transformers",
    "gradio":                "gradio",
}

def _ensure(packages: dict, label: str) -> list:
    missing = []
    for mod, pkg in packages.items():
        try:
            importlib.import_module(mod)
        except Exception:
            missing.append(pkg)
    if missing:
        print(f"[setup] installing {label}: {' '.join(missing)}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    return missing

if os.environ.get("CCTECH_SKIP_INSTALL") != "1":
    _ensure(REQUIREMENTS, "required")
    _ensure(OPTIONAL, "optional")
print("[setup] dependency bootstrap complete")

[setup] installing required: warcio
[setup] dependency bootstrap complete


In [2]:
# --- imports and logging --------------------------------------------------
import gc, io, json, hashlib, logging, math, random, re, shutil, threading, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Set, Tuple
from urllib.parse import urlsplit, unquote

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib
matplotlib.use("Agg") if os.environ.get("CCTECH_HEADLESS") == "1" else None
import matplotlib.pyplot as plt

import pyarrow as pa
import pyarrow.parquet as pq
import joblib
import requests

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (average_precision_score, precision_recall_fscore_support,
                             hamming_loss, precision_recall_curve)
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
LOG = logging.getLogger("cc-tech-fp")

# Optional imports -----------------------------------------------------------
try:
    from lxml import html as lxml_html
    from lxml import etree as lxml_etree
    HAVE_LXML = True
except Exception:                                              # pragma: no cover
    HAVE_LXML = False
    LOG.warning("lxml unavailable - falling back to a regex HTML extractor (lower fidelity)")

try:
    from warcio.archiveiterator import ArchiveIterator
    HAVE_WARCIO = True
except Exception:
    HAVE_WARCIO = False
    LOG.warning("warcio unavailable - Common Crawl ingestion is disabled")

try:
    import torch
    import torch.nn as nn
    HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False
    LOG.warning("torch unavailable - the neural head falls back to linear models")

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x=None, **kw):                                     # minimal shim
        return x if x is not None else None

try:
    import networkx as nx
    HAVE_NETWORKX = True
except Exception:
    HAVE_NETWORKX = False

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
LOG.info("environment: colab=%s lxml=%s warcio=%s torch=%s", IN_COLAB, HAVE_LXML, HAVE_WARCIO, HAVE_TORCH)

13:15:19 | INFO    | environment: colab=True lxml=True warcio=True torch=True


## 5. Configuration

**This is the only cell you need to edit.** Everything downstream reads from `CONFIG`.

`DATASET_MODE` selects a tier; the tier fills in page/file/support budgets, which you may still override by
editing the explicit keys after the tier is applied.

| Mode | Target pages | Archive files | Realistic Colab wall-clock |
|---|---|---|---|
| `quick` | ~10,000 | 3 | 8-15 min end to end |
| `standard` | ~100,000 | 12 | 60-110 min end to end |
| `large` | ~500,000 | 60 | 4-7 h, needs a persistent runtime + Drive cache |

Those numbers are measured against Colab's typical ~15-40 MB/s throughput from `data.commoncrawl.org` and
single-machine lxml parsing at roughly 1-3 ms/page with 4 download workers. A Common Crawl WARC file is about
1 GB compressed and yields roughly 25k-45k HTML responses, of which the per-domain cap keeps 50-70%.
`large` is included because it is *achievable*, not because it is comfortable: budget ~60 GB of transfer and use
`RESUME=True` so a disconnect costs you one archive file rather than the whole run.

In [3]:
# =========================== CONFIGURATION ================================
CONFIG: Dict[str, Any] = {
    # ---- identity -------------------------------------------------------
    "PROJECT_NAME":            "common_crawl_technology_fingerprinting",
    "EXPERIMENT_TAG":          "v1",

    # ---- Common Crawl ---------------------------------------------------
    # Pinned so a cached corpus in Drive is reused. "auto" resolves to whatever
    # collinfo.json reports as newest AT RUN TIME - if a newer crawl has shipped since your
    # last run, "auto" finds no cache under that ID and re-downloads the whole corpus.
    # Set back to "auto" for a fresh project, or change the ID to target a specific crawl.
    "CRAWL":                   "CC-MAIN-2026-30",
    "SOURCE_MODE":             "warc",     # "warc" (HTML + headers) | "wat" (metadata only, cheaper parse)
    "MAX_WORKERS":             4,          # parallel archive-file streams

    # ---- dataset scale --------------------------------------------------
    "DATASET_MODE":            "standard", # "quick" | "standard" | "large"
    "MAX_PAGES":               None,       # None -> taken from the tier
    "MAX_ARCHIVE_FILES":       None,
    "MAX_PAGES_PER_DOMAIN":    3,
    "MAX_DOMAINS":             None,       # optional hard cap on distinct registrable domains
    "MIN_TECH_SUPPORT":        None,       # min positive pages for a technology to be modelled

    # ---- splitting ------------------------------------------------------
    "TRAIN_FRACTION":          0.70,
    "VAL_FRACTION":            0.15,
    "TEST_FRACTION":           0.15,
    "RANDOM_SEED":             42,

    # ---- labelling ------------------------------------------------------
    "POSITIVE_CONFIDENCE":     "high",     # confidence level promoted to a supervised positive
    "MASK_UNCERTAIN":          True,       # MEDIUM/LOW cells are excluded from training AND evaluation
    "IMPLICATION_CONFIDENCE":  "medium",   # implied labels (Next.js -> React) stay uncertain by design
    "APPLY_IMPLICATIONS":      True,

    # ---- features / models ----------------------------------------------
    "EMBEDDING_MODEL":         "sentence-transformers/all-MiniLM-L6-v2",
    "EMBED_MAX_ROWS_CPU":      20000,      # above this, CPU runs use the SVD fallback instead of a transformer
    "TFIDF_MAX_FEATURES":      60000,
    "SVD_COMPONENTS":          192,
    "MLP_HIDDEN":              (512, 256),
    "MLP_EPOCHS":              30,
    "MLP_BATCH":               512,
    "MLP_LR":                  1e-3,
    "MAX_TRAIN_ROWS":          400000,     # subsample guard for the largest tier

    # ---- temporal experiment --------------------------------------------
    "ENABLE_TEMPORAL_EVALUATION": True,
    "TEMPORAL_CRAWLS":         "auto",     # "auto" -> spread across available years, or an explicit list
    "TEMPORAL_N_CRAWLS":       4,
    "TEMPORAL_PAGES_PER_CRAWL": None,      # None -> from the tier

    # ---- storage / runtime ----------------------------------------------
    "USE_DRIVE":               True,       # mount Google Drive for a persistent cache when in Colab
    "CACHE_DIR":               None,       # None -> auto (Drive, else /content, else ./)
    "RESUME":                  True,       # skip archive files already recorded in the manifest
    "OFFLINE_DEMO":            False,      # True -> synthetic corpus, no network (pipeline self-test)
    "LAUNCH_GRADIO":           True,
    "GRADIO_SHARE":            True,
    "MAKE_FIGURES":            True,
}

DATASET_TIERS = {
    "quick":    {"MAX_PAGES":  10_000, "MAX_ARCHIVE_FILES":  3, "MIN_TECH_SUPPORT":  25,
                 "TEMPORAL_PAGES_PER_CRAWL":  2_500},
    "standard": {"MAX_PAGES": 100_000, "MAX_ARCHIVE_FILES": 12, "MIN_TECH_SUPPORT": 150,
                 "TEMPORAL_PAGES_PER_CRAWL":  8_000},
    "large":    {"MAX_PAGES": 500_000, "MAX_ARCHIVE_FILES": 60, "MIN_TECH_SUPPORT": 500,
                 "TEMPORAL_PAGES_PER_CRAWL": 20_000},
}

def apply_tier(cfg: Dict[str, Any]) -> Dict[str, Any]:
    """Fill unset budget keys from the selected tier. Explicit values always win."""
    mode = str(cfg["DATASET_MODE"]).lower()
    if mode not in DATASET_TIERS:
        raise ValueError(f"DATASET_MODE must be one of {sorted(DATASET_TIERS)}, got {mode!r}")
    for k, v in DATASET_TIERS[mode].items():
        if cfg.get(k) is None:
            cfg[k] = v
    frac = cfg["TRAIN_FRACTION"] + cfg["VAL_FRACTION"] + cfg["TEST_FRACTION"]
    if abs(frac - 1.0) > 1e-6:
        raise ValueError(f"split fractions must sum to 1.0, got {frac}")
    return cfg

# A tiny self-test mode used by CI / offline validation of the pipeline.
if os.environ.get("CCTECH_SMOKE") == "1":
    CONFIG.update(DATASET_MODE="quick", OFFLINE_DEMO=True, MAX_PAGES=1800, MIN_TECH_SUPPORT=15,
                  LAUNCH_GRADIO=False, USE_DRIVE=False, MLP_EPOCHS=3,
                  TEMPORAL_PAGES_PER_CRAWL=2500, MAX_ARCHIVE_FILES=1, EMBED_MAX_ROWS_CPU=0)

CONFIG = apply_tier(CONFIG)

random.seed(CONFIG["RANDOM_SEED"])
np.random.seed(CONFIG["RANDOM_SEED"])
if HAVE_TORCH:
    torch.manual_seed(CONFIG["RANDOM_SEED"])

print(json.dumps({k: (str(v) if isinstance(v, tuple) else v) for k, v in CONFIG.items()}, indent=2))

{
  "PROJECT_NAME": "common_crawl_technology_fingerprinting",
  "EXPERIMENT_TAG": "v1",
  "CRAWL": "CC-MAIN-2026-30",
  "SOURCE_MODE": "warc",
  "MAX_WORKERS": 4,
  "DATASET_MODE": "standard",
  "MAX_PAGES": 100000,
  "MAX_ARCHIVE_FILES": 12,
  "MAX_PAGES_PER_DOMAIN": 3,
  "MAX_DOMAINS": null,
  "MIN_TECH_SUPPORT": 150,
  "TRAIN_FRACTION": 0.7,
  "VAL_FRACTION": 0.15,
  "TEST_FRACTION": 0.15,
  "RANDOM_SEED": 42,
  "POSITIVE_CONFIDENCE": "high",
  "MASK_UNCERTAIN": true,
  "IMPLICATION_CONFIDENCE": "medium",
  "APPLY_IMPLICATIONS": true,
  "EMBEDDING_MODEL": "sentence-transformers/all-MiniLM-L6-v2",
  "EMBED_MAX_ROWS_CPU": 20000,
  "TFIDF_MAX_FEATURES": 60000,
  "SVD_COMPONENTS": 192,
  "MLP_HIDDEN": "(512, 256)",
  "MLP_EPOCHS": 30,
  "MLP_BATCH": 512,
  "MLP_LR": 0.001,
  "MAX_TRAIN_ROWS": 400000,
  "ENABLE_TEMPORAL_EVALUATION": true,
  "TEMPORAL_CRAWLS": "auto",
  "TEMPORAL_N_CRAWLS": 4,
  "TEMPORAL_PAGES_PER_CRAWL": 8000,
  "USE_DRIVE": true,
  "CACHE_DIR": null,
  "RESUME": true,


In [4]:
# --- storage layout + Drive mount ----------------------------------------
def resolve_cache_root(cfg: Dict[str, Any]) -> Path:
    """Prefer a persistent Drive cache, then Colab local disk, then CWD."""
    if cfg.get("CACHE_DIR"):
        return Path(cfg["CACHE_DIR"])
    if IN_COLAB and cfg.get("USE_DRIVE"):
        try:
            from google.colab import drive  # type: ignore
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
            root = Path("/content/drive/MyDrive") / cfg["PROJECT_NAME"]
            root.mkdir(parents=True, exist_ok=True)
            LOG.info("using Google Drive cache: %s", root)
            return root
        except Exception as exc:
            LOG.warning("Drive mount failed (%s) - falling back to local disk", exc)
    base = Path("/content") if IN_COLAB and os.path.isdir("/content") else Path.cwd()
    return base / cfg["PROJECT_NAME"]

ROOT = resolve_cache_root(CONFIG)
PATHS = {name: ROOT / name for name in
         ["raw", "processed", "datasets", "models", "reports", "figures", "checkpoints"]}
for p in [ROOT, *PATHS.values()]:
    p.mkdir(parents=True, exist_ok=True)

RUN: Dict[str, Any] = {"started_at": time.time(), "root": str(ROOT)}
RESULTS: Dict[str, Any] = {}     # every section writes its findings here; S35 renders the report

def save_json(obj: Any, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str)
    return path

def savefig(fig, name: str) -> Optional[Path]:
    if not CONFIG["MAKE_FIGURES"]:
        plt.close(fig); return None
    out = PATHS["figures"] / f"{name}.png"
    fig.tight_layout(); fig.savefig(out, dpi=140, bbox_inches="tight"); plt.close(fig)
    LOG.info("figure saved: %s", out.name)
    return out

print("cache root :", ROOT)
for k, v in PATHS.items():
    print(f"  {k:<12} {v}")

Mounted at /content/drive


13:15:39 | INFO    | using Google Drive cache: /content/drive/MyDrive/common_crawl_technology_fingerprinting


cache root : /content/drive/MyDrive/common_crawl_technology_fingerprinting
  raw          /content/drive/MyDrive/common_crawl_technology_fingerprinting/raw
  processed    /content/drive/MyDrive/common_crawl_technology_fingerprinting/processed
  datasets     /content/drive/MyDrive/common_crawl_technology_fingerprinting/datasets
  models       /content/drive/MyDrive/common_crawl_technology_fingerprinting/models
  reports      /content/drive/MyDrive/common_crawl_technology_fingerprinting/reports
  figures      /content/drive/MyDrive/common_crawl_technology_fingerprinting/figures
  checkpoints  /content/drive/MyDrive/common_crawl_technology_fingerprinting/checkpoints


In [5]:
# --- hardware detection ---------------------------------------------------
def detect_hardware() -> Dict[str, Any]:
    info: Dict[str, Any] = {"device": "cpu", "gpu_name": None, "gpu_mem_gb": None,
                            "cpu_count": os.cpu_count(), "ram_gb": None}
    try:
        import psutil  # type: ignore
        info["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except Exception:
        try:
            info["ram_gb"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
        except Exception:
            pass
    if HAVE_TORCH and torch.cuda.is_available():
        info.update(device="cuda", gpu_name=torch.cuda.get_device_name(0),
                    gpu_mem_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    return info

HW = detect_hardware()
DEVICE = HW["device"]
RUN["hardware"] = HW
LOG.info("hardware: %s", HW)

def trim_heap() -> None:
    """Ask glibc to return free heap to the OS.

    Freeing millions of small objects leaves the allocator holding fragmented arenas, so
    RSS stays high even though the memory is logically free - and on Colab it is RSS that
    gets you OOM-killed. malloc_trim releases those arenas. No-op off glibc.
    """
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def mem_report(tag: str = "") -> None:
    """Best-effort RSS report; used around the memory-heavy stages."""
    try:
        import psutil  # type: ignore
        rss = psutil.Process().memory_info().rss / 1e9
        LOG.info("[mem] %s rss=%.2f GB", tag, rss)
    except Exception:
        pass

def free(*names: str) -> None:
    """Delete globals BY NAME and collect.

    Takes names, not objects: passing an object into a function only binds a new local
    reference, so `del` inside the function drops that local and leaves the caller's
    global binding — and therefore the memory — completely intact.
    """
    g = globals()
    for n in names:
        if isinstance(n, str) and n in g:
            del g[n]
        else:
            LOG.debug("free() expects a variable NAME as a string, got %r", type(n))
    gc.collect()
    trim_heap()
    if HAVE_TORCH and DEVICE == "cuda":
        torch.cuda.empty_cache()

13:15:39 | INFO    | hardware: {'device': 'cuda', 'gpu_name': 'Tesla T4', 'gpu_mem_gb': 15.6, 'cpu_count': 2, 'ram_gb': 13.6}


## 6. Common Crawl discovery

Crawl identifiers are discovered at runtime from `https://index.commoncrawl.org/collinfo.json`, so the notebook
does not go stale when a new crawl ships. A hard-coded fallback list keeps things running if that endpoint is
unavailable.

For each selected crawl we read `crawl-data/<CRAWL-ID>/warc.paths.gz` (or `wat.paths.gz`), which lists every
archive file in the crawl (typically ~90,000 files across 100 segments).

### Why WARC/WAT and not the columnar index or CDX?

| Mechanism | Content | Verdict for this project |
|---|---|---|
| **CDX API** | URL-level metadata, queried by URL/domain pattern | Rejected as the primary path — it is *targeted retrieval*, requires knowing which domains to ask for, and returns no page content. Retained only as an optional lookup in the temporal transition analysis. |
| **Columnar index** (Parquet on S3) | url, host, mime, status, language, WARC offsets | Excellent for *selecting* pages, but contains **no page evidence** — you still have to fetch WARC byte ranges afterwards, which is one HTTPS round trip per page and is far slower than sequential streaming for six-figure page counts. |
| **WET** | Plain text only | Useless here: strips exactly the scripts, link tags, class attributes and headers that constitute technology evidence. |
| **WAT** | JSON metadata: script/link/img URLs, meta tags, anchor graph, **HTTP response headers** | Strong second choice. Cheap to parse, but no DOM, no class attributes, no inline script markers. |
| **WARC** *(default)* | Raw HTTP response: **headers + full HTML** | Chosen. One sequential stream yields every evidence family at once, and the same extractor can be pointed at user-pasted HTML in the inference UI — the training path and the serving path share code. |

Sequential streaming also means we never issue per-page requests, never touch a live origin server, and read each
archive file exactly once with early termination once the page budget is met.

**Broad coverage strategy.** Archive files are selected by *stride sampling* across the full path list rather than
taking the first N. Common Crawl orders paths by segment, so a stride spreads the sample across the whole crawl
rather than concentrating it in one segment's host partition. Combined with `MAX_PAGES_PER_DOMAIN`, this produces
a dataset that is wide (many domains) rather than deep (many pages per site).

In [6]:
# --- Common Crawl endpoints ----------------------------------------------
CC_DATA_BASE   = "https://data.commoncrawl.org/"
CC_COLLINFO    = "https://index.commoncrawl.org/collinfo.json"
USER_AGENT     = ("cc-technology-fingerprinting-research/1.0 "
                  "(passive Common Crawl analysis; contact: notebook user)")

# Used only if collinfo.json cannot be reached. Deliberately spans several years so the
# temporal experiment still has something to work with offline.
FALLBACK_CRAWLS = [
    "CC-MAIN-2025-08", "CC-MAIN-2024-51", "CC-MAIN-2024-33", "CC-MAIN-2024-10",
    "CC-MAIN-2023-50", "CC-MAIN-2023-23", "CC-MAIN-2022-49", "CC-MAIN-2022-21",
    "CC-MAIN-2021-49", "CC-MAIN-2021-21", "CC-MAIN-2020-45", "CC-MAIN-2019-47",
]

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

def http_get(url: str, *, stream: bool = False, timeout: Tuple[int, int] = (20, 180),
             retries: int = 4, backoff: float = 2.0):
    """GET with exponential backoff. Returns a Response (caller closes streamed ones)."""
    last: Optional[Exception] = None
    for attempt in range(retries):
        try:
            resp = SESSION.get(url, stream=stream, timeout=timeout)
            if resp.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"status {resp.status_code}")
            resp.raise_for_status()
            return resp
        except Exception as exc:                                   # noqa: BLE001
            last = exc
            wait = backoff ** attempt
            LOG.warning("GET failed (%s/%s) %s: %s - retrying in %.0fs",
                        attempt + 1, retries, url.rsplit('/', 1)[-1], exc, wait)
            time.sleep(wait)
    raise RuntimeError(f"GET failed after {retries} attempts: {url}") from last

def crawl_year(crawl_id: str) -> Optional[int]:
    m = re.search(r"CC-MAIN-(\d{4})-(\d{2})", crawl_id)
    return int(m.group(1)) if m else None

def list_available_crawls(force: bool = False) -> List[str]:
    """Discover crawl IDs, newest first. Cached to disk."""
    cache = PATHS["raw"] / "collinfo.json"
    if cache.exists() and not force:
        try:
            data = json.loads(cache.read_text())
            if data:
                return [d["id"] for d in data]
        except Exception:
            pass
    try:
        resp = http_get(CC_COLLINFO, timeout=(15, 60), retries=2)
        data = resp.json()
        save_json(data, cache)
        ids = [d["id"] for d in data]
        LOG.info("discovered %d Common Crawl indexes (newest: %s)", len(ids), ids[0])
        return ids
    except Exception as exc:
        LOG.warning("collinfo.json unavailable (%s) - using the built-in fallback list", exc)
        return list(FALLBACK_CRAWLS)

def resolve_crawl(cfg: Dict[str, Any], available: Sequence[str]) -> str:
    want = cfg.get("CRAWL", "auto")
    if want and want != "auto":
        if want not in available:
            LOG.warning("requested crawl %s is not in the discovered list - using it anyway", want)
        return want
    return available[0]

def resolve_temporal_crawls(cfg: Dict[str, Any], available: Sequence[str], primary: str) -> List[str]:
    """Pick N crawls spread across distinct years, oldest -> newest."""
    want = cfg.get("TEMPORAL_CRAWLS", "auto")
    if isinstance(want, (list, tuple)) and want:
        return list(want)
    n = int(cfg.get("TEMPORAL_N_CRAWLS", 4))
    by_year: Dict[int, str] = {}
    for cid in available:                     # available is newest-first; keep the newest per year
        # Legacy IDs such as CC-MAIN-2008-2009 predate the WARC format and have no
        # warc.paths.gz; requiring the canonical CC-MAIN-YYYY-WW shape filters them out.
        if not re.fullmatch(r"CC-MAIN-\d{4}-\d{2}", cid):
            continue
        y = crawl_year(cid)
        if y is not None and y >= 2013 and y not in by_year:
            by_year[y] = cid
    years = sorted(by_year)
    if len(years) > n:                        # keep the newest year, spread the rest
        idx = np.linspace(0, len(years) - 1, n).round().astype(int)
        years = [years[i] for i in sorted(set(idx.tolist()))]
    chosen = [by_year[y] for y in years]
    if primary not in chosen:
        chosen = chosen[:-1] + [primary] if len(chosen) >= n else chosen + [primary]
    return sorted(set(chosen), key=lambda c: (crawl_year(c) or 0, c))

AVAILABLE_CRAWLS = list_available_crawls()
CRAWL_ID = resolve_crawl(CONFIG, AVAILABLE_CRAWLS)
TEMPORAL_CRAWLS = resolve_temporal_crawls(CONFIG, AVAILABLE_CRAWLS, CRAWL_ID) \
                  if CONFIG["ENABLE_TEMPORAL_EVALUATION"] else []
RUN.update(crawl_id=CRAWL_ID, temporal_crawls=TEMPORAL_CRAWLS,
           n_available_crawls=len(AVAILABLE_CRAWLS))

print(f"available crawls : {len(AVAILABLE_CRAWLS)}  (newest {AVAILABLE_CRAWLS[0]})")
print(f"primary crawl    : {CRAWL_ID}")
print(f"temporal crawls  : {TEMPORAL_CRAWLS}")

available crawls : 126  (newest CC-MAIN-2026-30)
primary crawl    : CC-MAIN-2026-30
temporal crawls  : ['CC-MAIN-2013-48', 'CC-MAIN-2017-51', 'CC-MAIN-2022-49', 'CC-MAIN-2026-30']


In [7]:
# --- archive path listing and stride sampling -----------------------------
def fetch_archive_paths(crawl_id: str, kind: str = "warc") -> List[str]:
    """Return every archive file path in a crawl (cached, gzip-decoded)."""
    cache = PATHS["raw"] / f"{crawl_id}.{kind}.paths.json"
    if cache.exists():
        try:
            paths = json.loads(cache.read_text())
            if paths:
                return paths
        except Exception:
            pass
    url = f"{CC_DATA_BASE}crawl-data/{crawl_id}/{kind}.paths.gz"
    import gzip
    resp = http_get(url, timeout=(20, 120))
    paths = gzip.decompress(resp.content).decode("utf-8").split()
    save_json(paths, cache)
    LOG.info("%s: %d %s files listed", crawl_id, len(paths), kind)
    return paths

def stride_sample(paths: Sequence[str], n: int, seed: int) -> List[str]:
    """Spread the selection across the whole crawl instead of taking a contiguous block."""
    n = max(1, min(int(n), len(paths)))
    if n == len(paths):
        return list(paths)
    stride = len(paths) / n
    offset = (seed % max(1, int(stride))) if stride >= 1 else 0
    picked = []
    for i in range(n):
        idx = min(len(paths) - 1, int(i * stride) + offset)
        picked.append(paths[idx])
    return list(dict.fromkeys(picked))        # dedupe, preserve order

## 7. Automated broad-web ingestion

A thread pool streams several archive files concurrently. Each worker decompresses records on the fly, extracts a
compact evidence record, and hands it to a shared thread-safe writer that flushes Parquet shards every
`FLUSH_ROWS` rows. Nothing is held in RAM beyond the current buffer, and raw HTML is **never** persisted.

Resilience features, all of which matter on a Colab runtime that can vanish at any moment:

* **Manifest checkpointing** — completed archive files are recorded in `raw/<crawl>/manifest.json`; a resumed run
  skips them and continues from the existing page count.
* **Early termination** — a shared stop flag halts every worker the moment the page budget is met.
* **Per-domain caps** — a shared counter enforces `MAX_PAGES_PER_DOMAIN` across all workers.
* **Bounded records** — pages over `MAX_HTML_BYTES` are truncated, oversized ones skipped.
* **Retries with backoff** — the initial request for each archive file retries with exponential backoff. A failure
  *mid-stream* abandons that file and the run continues with the rest; the file is not marked complete, so a later
  resumed run re-reads it, and the rows already written are removed by the deduplication pass.

In [8]:
# --- registrable-domain resolution (offline, no PSL download) -------------
# A curated multi-part suffix set. This is deliberately a snapshot, not the full Public Suffix
# List: it keeps ingestion dependency-free and deterministic. Misses degrade to eTLD+1 on a
# two-label suffix, which only ever makes the domain split *more* conservative.
MULTI_PART_SUFFIXES = {
    "co.uk","org.uk","ac.uk","gov.uk","me.uk","net.uk","sch.uk","ltd.uk","plc.uk",
    "com.au","net.au","org.au","edu.au","gov.au","id.au","asn.au",
    "co.nz","net.nz","org.nz","govt.nz","ac.nz","school.nz",
    "co.jp","ne.jp","or.jp","ac.jp","go.jp","ad.jp","ed.jp","gr.jp","lg.jp",
    "com.br","net.br","org.br","gov.br","edu.br","com.cn","net.cn","org.cn","gov.cn","edu.cn","ac.cn",
    "com.mx","org.mx","gob.mx","com.tr","net.tr","org.tr","gov.tr","edu.tr",
    "co.in","net.in","org.in","gov.in","ac.in","edu.in","co.za","org.za","gov.za","ac.za",
    "co.kr","or.kr","go.kr","ac.kr","com.ar","gob.ar","org.ar","com.sg","edu.sg","gov.sg",
    "com.hk","org.hk","gov.hk","com.tw","org.tw","gov.tw","edu.tw","com.ua","kiev.ua",
    "com.pl","net.pl","org.pl","gov.pl","edu.pl","waw.pl","com.co","com.pe","com.ve","com.ec",
    "com.my","org.my","gov.my","edu.my","co.il","org.il","ac.il","gov.il","com.ng","com.gh",
    "com.pk","com.bd","com.np","com.vn","com.ph","co.id","or.id","ac.id","go.id","com.sa","com.eg",
    "co.th","in.th","ac.th","go.th","com.ru","org.ru","net.ru","com.es","org.es","gob.es",
    "co.at","or.at","ac.at","co.hu","co.ma","com.uy","com.py","com.bo","com.do","com.pa",
    "gov.au","gov.br","edu.mx","net.co","nom.co","org.uk","judiciary.uk","nhs.uk","police.uk",
}

def split_host(url: str) -> str:
    try:
        host = urlsplit(url).netloc.lower()
    except Exception:
        return ""
    host = host.split("@")[-1].split(":")[0].strip(".")
    return host

def registrable_domain(host: str) -> str:
    """eTLD+1 using the curated suffix snapshot; IP literals return unchanged."""
    if not host:
        return ""
    if re.fullmatch(r"[\d.]+", host) or ":" in host:
        return host
    parts = host.split(".")
    if len(parts) <= 2:
        return host
    if ".".join(parts[-2:]) in MULTI_PART_SUFFIXES and len(parts) >= 3:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:])

assert registrable_domain("www.bbc.co.uk") == "bbc.co.uk"
assert registrable_domain("a.b.example.com") == "example.com"
assert registrable_domain("example.com") == "example.com"

In [9]:
# --- evidence record schema ----------------------------------------------
# One row per page. Complex fields are JSON-encoded strings so the Parquet schema stays flat
# and stable across shards written by different workers.
EVIDENCE_COLUMNS = [
    "url", "host", "domain", "crawl_id", "fetch_time", "status", "source_mode",
    "html_bytes", "html_sha1", "title", "text_sample", "text_len",
    "headers_json", "cookie_names_json", "metas_json",
    "scripts_json", "script_ids_json", "inline_js", "links_json",
    "resource_domains_json", "class_tokens_json", "attr_names_json",
    "html_ids_json", "input_names_json", "tag_counts_json",
    "n_elements", "dom_depth", "n_scripts", "n_inline_scripts", "n_external_scripts",
    "n_stylesheets", "n_links_a", "n_images", "n_iframes", "n_forms", "n_inputs",
    "n_metas", "n_resource_domains", "n_external_domains", "resource_entropy",
    "n_class_tokens", "n_unique_class_tokens", "avg_class_tokens",
    "url_len", "url_path_depth", "url_has_query", "url_ext",
]

MAX_HTML_BYTES      = 600_000     # per page, truncated beyond this
SKIP_HTML_BYTES     = 3_000_000   # skip pathological pages entirely
MAX_ELEMENTS        = 25_000      # DOM walk cap
TEXT_SAMPLE_CHARS   = 1_200
INLINE_JS_CHARS     = 2_000
MAX_TRACK_SCRIPTS   = 60
MAX_CLASS_TOKENS    = 80
MAX_IDS             = 120

HEADER_WHITELIST = {
    "server", "x-powered-by", "x-generator", "via", "x-cache", "x-served-by", "x-varnish",
    "cf-ray", "cf-cache-status", "x-amz-cf-id", "x-akamai-transformed", "x-akamai-request-id",
    "x-fastly-request-id", "x-timer", "x-vercel-id", "x-vercel-cache", "x-nf-request-id",
    "x-nextjs-cache", "x-aspnet-version", "x-aspnetmvc-version", "x-runtime", "x-drupal-cache",
    "x-drupal-dynamic-cache", "x-shopify-stage", "x-shopid", "x-sorting-hat-shopid",
    "x-litespeed-cache", "x-turbo-charged-by", "x-wix-request-id", "x-hs-hub-id",
    "x-github-request-id", "x-content-type-options", "content-type", "x-pantheon-styx-hostname",
    "x-sucuri-id", "x-proxy-cache", "x-squarespace-hosting", "x-hubspot-correlation-id",
}
# Deliberately never captured: cookie VALUES, authorization, set-cookie payloads,
# any request-side header, and any header not on the whitelist above.

def blank_record() -> Dict[str, Any]:
    return {c: None for c in EVIDENCE_COLUMNS}

def normalise_headers(raw: Iterable[Tuple[str, str]]) -> Tuple[Dict[str, str], List[str]]:
    """Whitelist response headers and reduce Set-Cookie to bare cookie NAMES."""
    keep: Dict[str, str] = {}
    cookies: List[str] = []
    for name, value in raw or []:
        low = (name or "").lower().strip()
        val = (value or "").strip()
        if low == "set-cookie":
            cname = val.split("=", 1)[0].strip().lower()
            if cname and len(cname) <= 64:
                cookies.append(cname)          # value discarded immediately
        elif low in HEADER_WHITELIST:
            keep[low] = val[:200]
    return keep, sorted(set(cookies))[:25]

def url_features(url: str) -> Dict[str, Any]:
    try:
        parts = urlsplit(url)
    except Exception:
        return {"url_len": len(url or ""), "url_path_depth": 0, "url_has_query": 0, "url_ext": ""}
    path = unquote(parts.path or "")
    seg = [s for s in path.split("/") if s]
    ext = ""
    if seg and "." in seg[-1]:
        cand = seg[-1].rsplit(".", 1)[-1].lower()
        if 1 <= len(cand) <= 6 and cand.isalnum():
            ext = cand
    return {"url_len": len(url or ""), "url_path_depth": len(seg),
            "url_has_query": int(bool(parts.query)), "url_ext": ext}

def shannon_entropy(counts: Iterable[int]) -> float:
    vals = [c for c in counts if c > 0]
    total = float(sum(vals))
    if total <= 0:
        return 0.0
    return float(-sum((c / total) * math.log2(c / total) for c in vals))

In [10]:
# --- the shared evidence extractor ---------------------------------------
# This single function serves THREE call sites: WARC ingestion, the offline demo corpus, and
# the Gradio inference UI. Training-time and serving-time features are therefore identical by
# construction, which removes an entire class of train/serve skew bugs.
_WS_RE = re.compile(r"\s+")
_CLASS_SPLIT = re.compile(r"[\s]+")

def _decode_html(raw: bytes) -> bytes:
    return raw[:MAX_HTML_BYTES]

def extract_evidence_from_html(raw_html: bytes | str, url: str, *,
                               headers: Optional[Iterable[Tuple[str, str]]] = None,
                               crawl_id: str = "", fetch_time: str = "",
                               status: int = 200, source_mode: str = "warc") -> Optional[Dict[str, Any]]:
    """Turn one HTML document into a compact, bounded evidence record."""
    if isinstance(raw_html, str):
        raw_html = raw_html.encode("utf-8", "replace")
    if not raw_html:
        return None
    raw_html = _decode_html(raw_html)

    rec = blank_record()
    host = split_host(url)
    rec.update(url=url[:2000], host=host, domain=registrable_domain(host), crawl_id=crawl_id,
               fetch_time=fetch_time, status=int(status), source_mode=source_mode,
               html_bytes=len(raw_html), html_sha1=hashlib.sha1(raw_html).hexdigest())
    hdrs, cookies = normalise_headers(headers or [])
    rec["headers_json"] = json.dumps(hdrs)
    rec["cookie_names_json"] = json.dumps(cookies)
    rec.update(url_features(url))

    if not HAVE_LXML:
        return _regex_fallback_evidence(rec, raw_html)

    try:
        doc = lxml_html.document_fromstring(raw_html)
    except Exception:
        try:
            doc = lxml_html.document_fromstring(raw_html.decode("utf-8", "replace"))
        except Exception:
            return None

    tag_counts: Counter = Counter()
    class_counter: Counter = Counter()
    attr_names: Counter = Counter()
    ids: List[str] = []
    input_names: List[str] = []
    scripts: List[Dict[str, str]] = []
    script_ids: List[str] = []
    inline_chunks: List[str] = []
    links: List[Dict[str, str]] = []
    resource_urls: List[str] = []
    n_elements = 0
    max_depth = 0
    n_class_tokens = 0

    stack: List[Tuple[Any, int]] = [(doc, 0)]
    while stack:
        el, depth = stack.pop()
        tag = el.tag
        if not isinstance(tag, str):           # comments / PIs
            continue
        n_elements += 1
        if depth > max_depth:
            max_depth = depth
        tag_counts[tag] += 1
        if n_elements < MAX_ELEMENTS:
            attrib = el.attrib
            cls = attrib.get("class")
            if cls:
                toks = [t for t in _CLASS_SPLIT.split(cls.strip()) if t][:40]
                n_class_tokens += len(toks)
                class_counter.update(t[:60].lower() for t in toks)
            eid = attrib.get("id")
            if eid and len(ids) < MAX_IDS:
                ids.append(eid[:60].lower())
            for k in attrib.keys():
                kl = k.lower()
                if (kl.startswith(("data-", "ng-", "v-", "x-", "wire:", "aria-", ":", "@"))
                        or kl in ("ng-version", "v-cloak", "hx-get", "hx-post")):
                    attr_names[kl[:40]] += 1
            if tag == "script":
                src = (attrib.get("src") or "").strip()
                if len(scripts) < MAX_TRACK_SCRIPTS:
                    scripts.append({"src": src[:400], "id": (attrib.get("id") or "")[:60].lower(),
                                    "type": (attrib.get("type") or "")[:60].lower()})
                if attrib.get("id"):
                    script_ids.append(attrib["id"][:60].lower())
                if src:
                    resource_urls.append(src)
                else:
                    txt = (el.text or "")
                    if txt and sum(len(c) for c in inline_chunks) < INLINE_JS_CHARS:
                        inline_chunks.append(txt[:300])
            elif tag == "link":
                href = (attrib.get("href") or "").strip()
                rel = (attrib.get("rel") or "").strip().lower()
                if len(links) < MAX_TRACK_SCRIPTS:
                    links.append({"rel": rel[:40], "href": href[:400]})
                if href and ("stylesheet" in rel or "preload" in rel or "icon" in rel):
                    resource_urls.append(href)
            elif tag in ("img", "iframe", "source", "video", "embed"):
                src = (attrib.get("src") or attrib.get("data-src") or "").strip()
                if src:
                    resource_urls.append(src)
            elif tag == "input":
                nm = (attrib.get("name") or "")[:60]
                if nm and len(input_names) < 50:
                    input_names.append(nm.lower())
        for child in el:
            stack.append((child, depth + 1))

    metas: Dict[str, str] = {}
    for m in doc.iter("meta"):
        key = (m.get("name") or m.get("property") or m.get("http-equiv") or "").lower()[:60]
        if key and key not in metas:
            metas[key] = (m.get("content") or "")[:300]
    if not metas.get("charset"):
        for m in doc.iter("meta"):
            if m.get("charset"):
                metas["charset"] = m.get("charset")[:40]
                break

    title_el = doc.find(".//title")
    title = _WS_RE.sub(" ", (title_el.text or "")).strip()[:300] if title_el is not None else ""

    # resource domain graph (page host excluded -> "external" view)
    dom_counter: Counter = Counter()
    for r in resource_urls:
        rh = split_host(r) if "//" in r else ""
        if rh:
            dom_counter[registrable_domain(rh)] += 1
    ext_domains = {d for d in dom_counter if d and d != rec["domain"]}

    # visible text (scripts/styles stripped first)
    try:
        lxml_etree.strip_elements(doc, "script", "style", "noscript", with_tail=False)
        text = _WS_RE.sub(" ", doc.text_content() or "").strip()
    except Exception:
        text = ""

    rec.update(
        title=title,
        text_sample=text[:TEXT_SAMPLE_CHARS],
        text_len=len(text),
        metas_json=json.dumps(metas),
        scripts_json=json.dumps(scripts),
        script_ids_json=json.dumps(sorted(set(script_ids))[:30]),
        inline_js=("".join(inline_chunks)[:INLINE_JS_CHARS]).lower(),
        links_json=json.dumps(links),
        resource_domains_json=json.dumps(dict(dom_counter.most_common(40))),
        class_tokens_json=json.dumps([t for t, _ in class_counter.most_common(MAX_CLASS_TOKENS)]),
        attr_names_json=json.dumps([a for a, _ in attr_names.most_common(40)]),
        html_ids_json=json.dumps(sorted(set(ids))[:60]),
        input_names_json=json.dumps(sorted(set(input_names))[:30]),
        tag_counts_json=json.dumps(dict(tag_counts.most_common(60))),
        n_elements=n_elements,
        dom_depth=max_depth,
        n_scripts=tag_counts.get("script", 0),
        n_external_scripts=sum(1 for s in scripts if s["src"]),
        n_inline_scripts=max(0, tag_counts.get("script", 0) - sum(1 for s in scripts if s["src"])),
        n_stylesheets=sum(1 for l in links if "stylesheet" in l["rel"]),
        n_links_a=tag_counts.get("a", 0),
        n_images=tag_counts.get("img", 0),
        n_iframes=tag_counts.get("iframe", 0),
        n_forms=tag_counts.get("form", 0),
        n_inputs=tag_counts.get("input", 0),
        n_metas=len(metas),
        n_resource_domains=len(dom_counter),
        n_external_domains=len(ext_domains),
        resource_entropy=round(shannon_entropy(dom_counter.values()), 4),
        n_class_tokens=n_class_tokens,
        n_unique_class_tokens=len(class_counter),
        avg_class_tokens=round(n_class_tokens / max(1, n_elements), 3),
    )
    return rec

def _regex_fallback_evidence(rec: Dict[str, Any], raw: bytes) -> Dict[str, Any]:
    """Minimal extractor used only when lxml is missing. Structural features are degraded."""
    txt = raw.decode("utf-8", "replace")
    low = txt.lower()
    srcs = re.findall(r"<script[^>]+src=[\"']([^\"']+)", low)[:MAX_TRACK_SCRIPTS]
    hrefs = re.findall(r"<link[^>]+href=[\"']([^\"']+)", low)[:MAX_TRACK_SCRIPTS]
    dom_counter = Counter()
    for r in srcs + hrefs:
        rh = split_host(r) if "//" in r else ""
        if rh:
            dom_counter[registrable_domain(rh)] += 1
    tags = Counter(re.findall(r"<([a-z][a-z0-9]*)\b", low))
    rec.update(title="", text_sample=_WS_RE.sub(" ", re.sub(r"<[^>]+>", " ", txt))[:TEXT_SAMPLE_CHARS],
               text_len=len(txt), metas_json="{}", scripts_json=json.dumps([{"src": s, "id": "", "type": ""} for s in srcs]),
               script_ids_json="[]", inline_js=low[:INLINE_JS_CHARS], links_json=json.dumps([{"rel": "", "href": h} for h in hrefs]),
               resource_domains_json=json.dumps(dict(dom_counter.most_common(40))), class_tokens_json="[]",
               attr_names_json="[]", html_ids_json="[]", input_names_json="[]",
               tag_counts_json=json.dumps(dict(tags.most_common(60))),
               n_elements=sum(tags.values()), dom_depth=0, n_scripts=tags.get("script", 0),
               n_external_scripts=len(srcs), n_inline_scripts=max(0, tags.get("script", 0) - len(srcs)),
               n_stylesheets=len(hrefs), n_links_a=tags.get("a", 0), n_images=tags.get("img", 0),
               n_iframes=tags.get("iframe", 0), n_forms=tags.get("form", 0), n_inputs=tags.get("input", 0),
               n_metas=low.count("<meta"), n_resource_domains=len(dom_counter),
               n_external_domains=len(dom_counter), resource_entropy=round(shannon_entropy(dom_counter.values()), 4),
               n_class_tokens=0, n_unique_class_tokens=0, avg_class_tokens=0.0)
    return rec

In [11]:
# --- WAT record adapter ---------------------------------------------------
def extract_evidence_from_wat(payload: bytes, crawl_id: str) -> Optional[Dict[str, Any]]:
    """Map a WAT JSON metadata record onto the same evidence schema.

    WAT carries scripts, link tags, meta tags, the anchor graph and response headers, but no
    DOM and no class attributes - those structural fields are left null and are reported as
    unavailable in the feature audit.
    """
    try:
        obj = json.loads(payload)
    except Exception:
        return None
    env = obj.get("Envelope") or {}
    warc_hdr = env.get("WARC-Header-Metadata") or {}
    if (warc_hdr.get("WARC-Type") or "") != "response":
        return None
    url = warc_hdr.get("WARC-Target-URI") or ""
    if not url:
        return None
    pmeta = (env.get("Payload-Metadata") or {}).get("HTTP-Response-Metadata") or {}
    hmeta = pmeta.get("HTML-Metadata") or {}
    if not hmeta:
        return None
    resp_hdrs = ((env.get("Payload-Metadata") or {}).get("HTTP-Response-Metadata") or {}).get("Headers") or {}
    header_pairs = [(k, str(v)) for k, v in resp_hdrs.items()]

    rec = blank_record()
    host = split_host(url)
    rec.update(url=url[:2000], host=host, domain=registrable_domain(host), crawl_id=crawl_id,
               fetch_time=warc_hdr.get("WARC-Date", ""), status=200, source_mode="wat",
               html_bytes=int(env.get("Payload-Metadata", {}).get("Actual-Content-Length", 0) or 0))
    rec["html_sha1"] = hashlib.sha1(url.encode()).hexdigest()   # no body available in WAT
    hdrs, cookies = normalise_headers(header_pairs)
    rec["headers_json"] = json.dumps(hdrs)
    rec["cookie_names_json"] = json.dumps(cookies)
    rec.update(url_features(url))

    head = hmeta.get("Head") or {}
    metas: Dict[str, str] = {}
    for m in head.get("Metas") or []:
        key = (m.get("name") or m.get("property") or m.get("http-equiv") or "").lower()[:60]
        if key and key not in metas:
            metas[key] = str(m.get("content") or "")[:300]
    scripts = [{"src": str(s.get("url") or "")[:400], "id": "", "type": str(s.get("type") or "")[:60]}
               for s in (head.get("Scripts") or [])][:MAX_TRACK_SCRIPTS]
    links = [{"rel": str(l.get("rel") or "").lower()[:40], "href": str(l.get("url") or l.get("path") or "")[:400]}
             for l in (head.get("Link") or [])][:MAX_TRACK_SCRIPTS]

    dom_counter: Counter = Counter()
    for u in [s["src"] for s in scripts] + [l["href"] for l in links] + \
             [str(l.get("url") or "") for l in (hmeta.get("Links") or [])[:200]]:
        rh = split_host(u) if "//" in u else ""
        if rh:
            dom_counter[registrable_domain(rh)] += 1
    ext = {d for d in dom_counter if d and d != rec["domain"]}
    n_anchor = sum(1 for l in (hmeta.get("Links") or []) if (l.get("path") or "").startswith("A@"))
    n_img = sum(1 for l in (hmeta.get("Links") or []) if (l.get("path") or "").startswith("IMG@"))
    title = str(head.get("Title") or "")[:300]

    rec.update(title=title, text_sample=title, text_len=len(title), metas_json=json.dumps(metas),
               scripts_json=json.dumps(scripts), script_ids_json="[]", inline_js="",
               links_json=json.dumps(links),
               resource_domains_json=json.dumps(dict(dom_counter.most_common(40))),
               class_tokens_json="[]", attr_names_json="[]", html_ids_json="[]", input_names_json="[]",
               tag_counts_json="{}", n_elements=0, dom_depth=0,
               n_scripts=len(scripts), n_external_scripts=len([s for s in scripts if s["src"]]),
               n_inline_scripts=0, n_stylesheets=sum(1 for l in links if "stylesheet" in l["rel"]),
               n_links_a=n_anchor, n_images=n_img, n_iframes=0, n_forms=0, n_inputs=0,
               n_metas=len(metas), n_resource_domains=len(dom_counter), n_external_domains=len(ext),
               resource_entropy=round(shannon_entropy(dom_counter.values()), 4),
               n_class_tokens=0, n_unique_class_tokens=0, avg_class_tokens=0.0)
    return rec

In [12]:
# --- thread-safe shard writer + ingestion driver --------------------------
class ShardWriter:
    """Buffers evidence rows and flushes stable-schema Parquet shards."""

    def __init__(self, out_dir: Path, flush_rows: int = 2500, prefix: str = "shard"):
        self.dir = out_dir; self.dir.mkdir(parents=True, exist_ok=True)
        self.flush_rows = flush_rows; self.prefix = prefix
        self._buf: List[Dict[str, Any]] = []
        self._lock = threading.Lock()
        self._n_written = 0
        existing = sorted(self.dir.glob(f"{prefix}_*.parquet"))
        self._shard_idx = len(existing)

    @property
    def n_written(self) -> int:
        with self._lock:
            return self._n_written

    def add(self, rec: Dict[str, Any]) -> None:
        with self._lock:
            self._buf.append(rec)
            if len(self._buf) >= self.flush_rows:
                self._flush_locked()

    def _flush_locked(self) -> None:
        if not self._buf:
            return
        df = pd.DataFrame(self._buf, columns=EVIDENCE_COLUMNS)
        path = self.dir / f"{self.prefix}_{self._shard_idx:05d}.parquet"
        df.to_parquet(path, index=False, compression="snappy")
        self._n_written += len(df)
        self._shard_idx += 1
        self._buf.clear()
        del df; gc.collect()

    def close(self) -> int:
        with self._lock:
            self._flush_locked()
            return self._n_written

class IngestState:
    """Shared, lock-guarded counters for the worker pool."""

    def __init__(self, max_pages: int, max_per_domain: int, max_domains: Optional[int]):
        self.max_pages = max_pages
        self.max_per_domain = max_per_domain
        self.max_domains = max_domains
        self.domain_counts: Counter = Counter()
        self.kept = 0
        self.seen = 0
        self.stop = threading.Event()
        self._lock = threading.Lock()

    def can_take(self, domain: str) -> bool:
        """Cheap pre-check before the expensive parse. Does not consume budget."""
        with self._lock:
            self.seen += 1
            if self.kept >= self.max_pages:
                self.stop.set(); return False
            if self.domain_counts[domain] >= self.max_per_domain:
                return False
            if (self.max_domains and domain not in self.domain_counts
                    and len(self.domain_counts) >= self.max_domains):
                return False
            return True

    def commit(self, domain: str) -> None:
        """Consume budget only once a record has actually been extracted."""
        with self._lock:
            self.domain_counts[domain] += 1
            self.kept += 1
            if self.kept >= self.max_pages:
                self.stop.set()

def _iter_archive(url: str, state: IngestState):
    """Yield warcio records from a streamed archive file, honouring the stop flag."""
    resp = http_get(url, stream=True, timeout=(20, 300), retries=3)
    try:
        for record in ArchiveIterator(resp.raw):
            if state.stop.is_set():
                break
            yield record
    finally:
        resp.close()

def process_archive_file(path: str, *, crawl_id: str, source_mode: str,
                         writer: ShardWriter, state: IngestState) -> int:
    """Stream one WARC/WAT file and write evidence rows. Returns rows kept."""
    url = CC_DATA_BASE + path
    kept = 0
    for record in _iter_archive(url, state):
        try:
            if source_mode == "warc":
                if record.rec_type != "response":
                    continue
                http = record.http_headers
                if http is None:
                    continue
                ctype = (http.get_header("Content-Type") or "").lower()
                if "html" not in ctype:
                    continue
                try:
                    status = int(http.get_statuscode() or 0)
                except Exception:
                    status = 0
                if status != 200:
                    continue
                target = record.rec_headers.get_header("WARC-Target-URI") or ""
                domain = registrable_domain(split_host(target))
                if not domain or not state.can_take(domain):
                    continue
                body = record.content_stream().read(SKIP_HTML_BYTES)
                if not body or len(body) < 200:
                    continue
                rec = extract_evidence_from_html(
                    body, target, headers=http.headers, crawl_id=crawl_id,
                    fetch_time=record.rec_headers.get_header("WARC-Date") or "",
                    status=status, source_mode="warc")
            else:                                                   # WAT
                if record.rec_type != "metadata":
                    continue
                payload = record.content_stream().read()
                rec = extract_evidence_from_wat(payload, crawl_id)
                if rec is None:
                    continue
                domain = rec["domain"]
                if not domain or not state.can_take(domain):
                    continue
            if rec is not None:
                state.commit(rec["domain"])          # budget consumed only on success
                writer.add(rec)
                kept += 1
        except Exception as exc:                                    # one bad record must not kill the file
            LOG.debug("record skipped: %s", exc)
            continue
    return kept

def ingest_crawl(crawl_id: str, *, max_pages: int, max_archive_files: int,
                 max_pages_per_domain: int, max_domains: Optional[int] = None,
                 source_mode: str = "warc", workers: int = 4,
                 resume: bool = True, tag: str = "main") -> Path:
    """Fully automated ingestion for one crawl. Idempotent and resumable."""
    out_dir = PATHS["raw"] / f"{crawl_id}_{tag}"
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "manifest.json"
    manifest = {"crawl_id": crawl_id, "source_mode": source_mode, "done_files": [],
                "pages": 0, "config": {"max_pages": max_pages,
                                       "max_pages_per_domain": max_pages_per_domain}}
    if resume and manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text())
        except Exception:
            pass

    already = int(manifest.get("pages", 0))
    if already >= max_pages:
        LOG.info("[%s] cached corpus already holds %d pages - skipping ingestion", crawl_id, already)
        return out_dir
    if not HAVE_WARCIO:
        raise RuntimeError("warcio is required for Common Crawl ingestion")

    kind = "warc" if source_mode == "warc" else "wat"
    all_paths = fetch_archive_paths(crawl_id, kind)
    selected = stride_sample(all_paths, max_archive_files, CONFIG["RANDOM_SEED"])
    done: Set[str] = set(manifest.get("done_files", []))
    todo = [p for p in selected if p not in done]
    LOG.info("[%s] %d/%d archive files remaining (target %d pages, %d already cached)",
             crawl_id, len(todo), len(selected), max_pages, already)
    if not todo:
        return out_dir

    writer = ShardWriter(out_dir, flush_rows=2500)
    state = IngestState(max_pages - already, max_pages_per_domain, max_domains)
    # Pre-seed domain counts so a resumed run keeps honouring the per-domain cap.
    if already and (out_dir / "domain_counts.json").exists():
        try:
            state.domain_counts.update(json.loads((out_dir / "domain_counts.json").read_text()))
        except Exception:
            pass

    t0 = time.time()
    from concurrent.futures import ThreadPoolExecutor, as_completed
    lock = threading.Lock()
    with ThreadPoolExecutor(max_workers=max(1, workers)) as pool:
        futures = {pool.submit(process_archive_file, p, crawl_id=crawl_id, source_mode=source_mode,
                               writer=writer, state=state): p for p in todo}
        for fut in as_completed(futures):
            p = futures[fut]
            try:
                n = fut.result()
                with lock:
                    manifest["done_files"] = sorted(set(manifest["done_files"]) | {p})
                LOG.info("[%s] %s -> +%d rows (total kept %d, seen %d, %.0fs)",
                         crawl_id, p.rsplit('/', 1)[-1], n, state.kept, state.seen, time.time() - t0)
            except Exception as exc:
                LOG.warning("[%s] archive file failed permanently: %s (%s)", crawl_id, p, exc)
            if state.stop.is_set():
                for f in futures:
                    f.cancel()
                break

    written = writer.close()
    manifest["pages"] = already + written
    manifest["domains"] = len(state.domain_counts)
    manifest["updated_at"] = time.strftime("%Y-%m-%dT%H:%M:%S")
    save_json(manifest, manifest_path)
    save_json(dict(state.domain_counts), out_dir / "domain_counts.json")
    LOG.info("[%s] ingestion complete: %d new rows, %d total, %d domains, %.1f min",
             crawl_id, written, manifest["pages"], len(state.domain_counts), (time.time() - t0) / 60)
    mem_report("post-ingest")
    return out_dir

In [13]:
# --- offline synthetic corpus (pipeline self-test only) -------------------
# Used when OFFLINE_DEMO=True or when the network is unavailable. It generates HTML pages from
# templated technology stacks so the entire notebook can be validated end to end without
# touching the network. It is NOT research data and every downstream report says so.
SYNTH_STACKS = [
    ("wordpress", ["WordPress", "jQuery", "Google Analytics", "Nginx"]),
    ("wp_shop",   ["WordPress", "WooCommerce", "jQuery", "PayPal", "Apache", "Google Fonts"]),
    ("next_saas", ["Next.js", "Tailwind CSS", "Stripe", "Cloudflare", "Google Tag Manager", "Vercel"]),
    ("react_spa", ["React", "Bootstrap", "Google Analytics", "Cloudflare"]),
    ("vue_app",   ["Vue.js", "Nginx", "Matomo"]),
    ("nuxt_app",  ["Nuxt.js", "Tailwind CSS", "Cloudflare"]),
    ("shopify",   ["Shopify", "Google Analytics", "Cloudflare", "Stripe"]),
    ("drupal",    ["Drupal", "jQuery", "Apache", "Google Fonts"]),
    ("aspnet",    ["ASP.NET", "Microsoft IIS", "Bootstrap", "jQuery"]),
    ("laravel",   ["Laravel", "PHP", "Bootstrap", "Nginx", "Google Analytics"]),
    ("static",    ["Google Fonts", "Cloudflare", "Tailwind CSS"]),
    ("angular",   ["Angular", "Nginx", "Google Tag Manager"]),
]

def _synth_page(stack_name: str, techs: List[str], rng: random.Random, i: int) -> Tuple[bytes, List[Tuple[str, str]], str]:
    head, body, hdrs = [], [], []
    t = set(techs)
    if "WordPress" in t:
        head.append('<meta name="generator" content="WordPress 6.5.2">')
        head.append('<link rel="stylesheet" href="/wp-content/themes/twenty/style.css">')
        body.append('<script src="/wp-includes/js/wp-emoji-release.min.js"></script>')
        hdrs.append(("Set-Cookie", "wordpress_test_cookie=WP+Cookie+check; path=/"))
    if "WooCommerce" in t:
        head.append('<link rel="stylesheet" href="/wp-content/plugins/woocommerce/assets/css/woocommerce.css">')
    if "Drupal" in t:
        head.append('<meta name="generator" content="Drupal 10 (https://www.drupal.org)">')
        body.append('<script src="/core/misc/drupal.js"></script>')
        body.append('<div data-drupal-selector="edit-form">x</div>')
        hdrs.append(("X-Generator", "Drupal 10 (https://www.drupal.org)"))
    if "Shopify" in t:
        head.append('<link rel="stylesheet" href="https://cdn.shopify.com/s/files/1/theme.css">')
        body.append('<script>var Shopify = Shopify || {}; Shopify.shop = "demo.myshopify.com";</script>')
        hdrs.append(("X-Shopid", "12345"))
    if "Next.js" in t:
        body.append('<script id="__NEXT_DATA__" type="application/json">{"props":{}}</script>')
        body.append('<script src="/_next/static/chunks/main.js"></script>')
        body.append('<div id="__next"><span>app</span></div>')
        hdrs.append(("X-Powered-By", "Next.js"))
    if "React" in t and "Next.js" not in t:
        body.append('<div id="root" data-reactroot=""><span>app</span></div>')
        body.append('<script src="https://unpkg.com/react-dom@18/umd/react-dom.production.min.js"></script>')
    if "Vue.js" in t and "Nuxt.js" not in t:
        body.append('<div id="app" data-v-7ba5bd90><span data-v-7ba5bd90>x</span></div>')
        body.append('<script src="https://cdn.jsdelivr.net/npm/vue@3/dist/vue.global.js"></script>')
    if "Nuxt.js" in t:
        body.append('<script>window.__NUXT__=(function(a){return {data:a}})([]);</script>')
        body.append('<script src="/_nuxt/entry.abc123.js"></script><div id="__nuxt"></div>')
    if "Angular" in t:
        body.append('<app-root ng-version="17.1.0"><div _ngcontent-abc>a</div></app-root>')
    if "Laravel" in t:
        hdrs.append(("Set-Cookie", "laravel_session=abc; path=/"))
    if "PHP" in t:
        hdrs.append(("X-Powered-By", "PHP/8.1.2"))
    if "ASP.NET" in t:
        hdrs.append(("X-AspNet-Version", "4.0.30319"))
        body.append('<input type="hidden" name="__VIEWSTATE" value="x">')
    if "Bootstrap" in t:
        head.append('<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">')
        body.append('<div class="container"><div class="row"><div class="col-md-6"><button class="btn btn-primary" data-bs-toggle="modal">go</button></div></div></div>')
    if "Tailwind CSS" in t:
        body.append('<div class="flex items-center justify-between px-4 py-2 text-sm font-medium bg-white rounded-lg shadow-md md:px-6 lg:text-base hover:bg-gray-50"><p class="mt-2 mb-4 text-gray-700">hi</p></div>')
    if "jQuery" in t:
        body.append('<script src="https://code.jquery.com/jquery-3.6.0.min.js"></script>')
    if "Google Analytics" in t:
        body.append('<script async src="https://www.googletagmanager.com/gtag/js?id=G-XYZ"></script>')
        body.append('<script>window.dataLayer=window.dataLayer||[];function gtag(){dataLayer.push(arguments);}gtag("js",new Date());</script>')
    if "Google Tag Manager" in t:
        body.append('<script async src="https://www.googletagmanager.com/gtm.js?id=GTM-ABCD"></script>')
    if "Matomo" in t:
        body.append('<script src="https://analytics.example.org/matomo.js"></script><script>var _paq=window._paq=window._paq||[];_paq.push(["trackPageView"]);</script>')
    if "Stripe" in t:
        body.append('<script src="https://js.stripe.com/v3/"></script>')
    if "PayPal" in t:
        body.append('<script src="https://www.paypal.com/sdk/js?client-id=x"></script>')
    if "Google Fonts" in t:
        head.append('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Inter">')
    if "Cloudflare" in t:
        hdrs.append(("Server", "cloudflare")); hdrs.append(("CF-Ray", "8abc123-LHR"))
        body.append('<script src="/cdn-cgi/scripts/beacon.min.js"></script>')
    if "Vercel" in t:
        hdrs.append(("X-Vercel-Id", "lhr1::abc"))
    if "Nginx" in t:
        hdrs.append(("Server", "nginx/1.24.0"))
    if "Apache" in t:
        hdrs.append(("Server", "Apache/2.4.52 (Ubuntu)"))
    if "Microsoft IIS" in t:
        hdrs.append(("Server", "Microsoft-IIS/10.0"))

    filler = " ".join(rng.choice(
        ["about", "services", "contact", "products", "the", "team", "pricing", "blog",
         "news", "support", "docs", "careers", "privacy", "shipping", "reviews"])
        for _ in range(rng.randint(60, 200)))
    for _ in range(rng.randint(3, 25)):
        body.append(f'<a href="/{rng.choice(["a","b","c","d"])}/{rng.randint(1,99)}">link</a>')
    for _ in range(rng.randint(0, 8)):
        body.append('<img src="/img/pic.png" alt="p">')
    hdrs.append(("Content-Type", "text/html; charset=utf-8"))
    html = (f'<!doctype html><html lang="en"><head><meta charset="utf-8">'
            f'<meta name="viewport" content="width=device-width, initial-scale=1">'
            f'<title>{stack_name} demo page {i}</title>{"".join(head)}</head>'
            f'<body><header><nav>menu</nav></header><main><h1>{stack_name}</h1>'
            f'<p>{filler}</p>{"".join(body)}</main><footer>footer</footer></body></html>')
    return html.encode("utf-8"), hdrs, stack_name

def build_synthetic_corpus(n_pages: int, crawl_id: str, seed: int = 0, tag: str = "main") -> Path:
    """Deterministic synthetic corpus with realistic stack co-occurrence for offline runs."""
    out_dir = PATHS["raw"] / f"{crawl_id}_{tag}"
    if (out_dir / "manifest.json").exists() and CONFIG["RESUME"]:
        try:
            if json.loads((out_dir / "manifest.json").read_text()).get("pages", 0) >= n_pages:
                LOG.info("[%s] synthetic corpus already cached", crawl_id); return out_dir
        except Exception:
            pass
    rng = random.Random(seed + (crawl_year(crawl_id) or 0))
    writer = ShardWriter(out_dir, flush_rows=1000)
    # Stack mix drifts with crawl year so the temporal analysis has a real signal to find.
    year = crawl_year(crawl_id) or 2024
    modern = min(0.85, max(0.05, (year - 2018) / 8.0))
    n_domains = max(20, n_pages // 3)
    for i in range(n_pages):
        dom_idx = rng.randrange(n_domains)
        drng = random.Random(seed * 7919 + dom_idx)          # stack is a property of the domain
        pool = SYNTH_STACKS
        pick = drng.random()
        if pick < modern:
            cand = [s for s in pool if s[0] in ("next_saas", "react_spa", "vue_app", "nuxt_app", "static", "shopify")]
        else:
            cand = [s for s in pool if s[0] in ("wordpress", "wp_shop", "drupal", "aspnet", "laravel", "angular")]
        name, techs = drng.choice(cand or pool)
        html, hdrs, _ = _synth_page(name, techs, rng, i)
        tld = ("com", "org", "net", "co.uk", "de", "io")[dom_idx % 6]
        url = f"https://www.demo-site-{dom_idx:05d}.{tld}/{name}/page{i}"
        rec = extract_evidence_from_html(html, url, headers=hdrs, crawl_id=crawl_id,
                                         fetch_time=f"{year}-06-01T00:00:00Z", source_mode="synthetic")
        if rec:
            writer.add(rec)
    n = writer.close()
    save_json({"crawl_id": crawl_id, "source_mode": "synthetic", "pages": n,
               "done_files": ["synthetic"], "synthetic": True}, out_dir / "manifest.json")
    LOG.warning("[%s] SYNTHETIC corpus generated (%d pages) - not research data", crawl_id, n)
    return out_dir

In [14]:
# --- run the primary ingestion -------------------------------------------
def run_ingestion(crawl_id: str, max_pages: int, tag: str = "main") -> Path:
    """Try Common Crawl, fall back to the synthetic corpus only if explicitly allowed."""
    if CONFIG["OFFLINE_DEMO"]:
        return build_synthetic_corpus(max_pages, crawl_id, CONFIG["RANDOM_SEED"], tag=tag)
    try:
        return ingest_crawl(
            crawl_id,
            max_pages=max_pages,
            max_archive_files=CONFIG["MAX_ARCHIVE_FILES"],
            max_pages_per_domain=CONFIG["MAX_PAGES_PER_DOMAIN"],
            max_domains=CONFIG["MAX_DOMAINS"],
            source_mode=CONFIG["SOURCE_MODE"],
            workers=CONFIG["MAX_WORKERS"],
            resume=CONFIG["RESUME"],
            tag=tag,
        )
    except Exception as exc:
        LOG.error("Common Crawl ingestion failed: %s", exc)
        LOG.error("Set CONFIG['OFFLINE_DEMO']=True to validate the pipeline without network access.")
        raise

RAW_DIR = run_ingestion(CRAWL_ID, CONFIG["MAX_PAGES"], tag="main")
print("raw evidence directory:", RAW_DIR)

13:15:42 | INFO    | [CC-MAIN-2026-30] cached corpus already holds 100003 pages - skipping ingestion


raw evidence directory: /content/drive/MyDrive/common_crawl_technology_fingerprinting/raw/CC-MAIN-2026-30_main


## 8. Raw data extraction and loading

Shards are read back with a projection so unused columns never enter RAM, and the JSON-encoded evidence fields
are parsed once into object columns for reuse by the labeller and the feature builders.

In [15]:
# --- load evidence shards -------------------------------------------------
JSON_FIELDS = {
    "headers_json": "headers", "cookie_names_json": "cookie_names", "metas_json": "metas",
    "scripts_json": "scripts", "script_ids_json": "script_ids", "links_json": "links",
    "resource_domains_json": "resource_domains", "class_tokens_json": "class_tokens",
    "attr_names_json": "attr_names", "html_ids_json": "html_ids",
    "input_names_json": "input_names", "tag_counts_json": "tag_counts",
}

def _safe_json(s: Any, default):
    if isinstance(s, (dict, list)):
        return s
    if not s:
        return default
    try:
        return json.loads(s)
    except Exception:
        return default

def load_evidence(raw_dir: Path, limit: Optional[int] = None) -> pd.DataFrame:
    shards = sorted(Path(raw_dir).glob("shard_*.parquet"))
    if not shards:
        raise FileNotFoundError(f"no evidence shards in {raw_dir}")
    frames, total = [], 0
    for s in shards:
        df = pd.read_parquet(s)
        frames.append(df); total += len(df)
        if limit and total >= limit:
            break
    df = pd.concat(frames, ignore_index=True)
    if limit:
        df = df.iloc[:limit].copy()
    del frames; gc.collect()
    for col in ["inline_js", "title", "text_sample", "url", "host", "domain", "url_ext"]:
        df[col] = df[col].fillna("").astype(str)
    num_cols = [c for c in EVIDENCE_COLUMNS if c.startswith(("n_", "url_len", "url_path",
                "url_has", "dom_depth", "html_bytes", "text_len", "resource_entropy", "avg_class"))]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    LOG.info("loaded %d evidence rows from %d shards", len(df), len(shards))
    return df

def materialise_json(df: pd.DataFrame) -> pd.DataFrame:
    """Parse JSON evidence columns once into object columns."""
    for src, dst in JSON_FIELDS.items():
        if src not in df.columns:
            continue          # slimmed frames retain only the JSON columns still needed
        default = {} if src.endswith(("headers_json", "metas_json", "resource_domains_json",
                                      "tag_counts_json")) else []
        df[dst] = df[src].map(lambda v, d=default: _safe_json(v, d))
    return df

def parsed_chunks(df: pd.DataFrame, chunk_size: int = 4000):
    """Yield (offset, chunk) with JSON evidence parsed for that chunk only.

    Materialising all 12 object columns for the whole corpus builds tens of millions of
    small dicts. Freeing them afterwards does NOT return the memory: the allocator keeps the
    fragmented arenas and RSS stays pinned near the ceiling. Parsing a chunk at a time keeps
    the live dict count bounded by chunk_size, so peak RSS never gets there.
    """
    for start in range(0, len(df), chunk_size):
        chunk = materialise_json(df.iloc[start:start + chunk_size].copy())
        yield start, chunk
        del chunk
    gc.collect()
    trim_heap()

# NOTE: the frame deliberately keeps the raw *_json strings; parsing happens per chunk.
EV = load_evidence(RAW_DIR)
RUN["ingested_pages"] = int(len(EV))
RUN["ingested_domains"] = int(EV["domain"].nunique())
mem_report("post-load")

print(f"pages   : {len(EV):,}")
print(f"domains : {EV['domain'].nunique():,}")
print(f"crawl   : {EV['crawl_id'].iloc[0] if len(EV) else 'n/a'}")
print(f"source  : {EV['source_mode'].iloc[0] if len(EV) else 'n/a'}")
display(EV[["url", "domain", "n_scripts", "n_stylesheets", "n_elements", "dom_depth",
            "n_external_domains", "title"]].head(8))

13:17:02 | INFO    | loaded 100003 evidence rows from 41 shards
13:17:02 | INFO    | [mem] post-load rss=2.68 GB


pages   : 100,003
domains : 88,981
crawl   : CC-MAIN-2026-30
source  : warc


,url,domain,n_scripts,n_stylesheets,n_elements,dom_depth,n_external_domains,title
0,http://0006xxx.com/11/186536/,0006xxx.com,13,1,215,8,1,"狼友阁,AI-杨幂-将按摩师请到家里进行全身舒适按摩【久热中文字幕第一页】高清电影完整版免费..."
1,http://01master.ru/products/literature/magazin...,01master.ru,10,8,241,12,2,ООО «ПОЖАРНЫЙ МАСТЕР» - Интернет-магазин - Жур...
2,http://053koshopco.com/shop/list.php?cate=139013,053koshopco.com,8,2,2838,25,2,전자상거래의 모든 것! koSHOPco!
3,http://057vv.com/list/index2_0.html,057vv.com,2,0,4,2,1,
4,http://093ff.com/play/index18651-0-0.html,093ff.com,17,2,161,7,4,私拍泄密极品御姐啪啪性爱私拍流出各式制服爆操完美露脸
5,http://1-220-modellbahn.de/,1-220-modellbahn.de,0,0,7,2,0,Juergens Z Page
6,http://1000box.ru/sumki_dlya_instrumentov_jetech,1000box.ru,14,5,251,13,3,Сумки для инструментов JeTech
7,http://1006pw.com/Article_List.asp?ID=4838,1006pw.com,12,2,362,12,4,Lyn:队友太给力 不太喜欢赛制-电子竞技-长沙新闻网


## 9. Technology fingerprint registry

The registry is the project's ground-truth generator, so it is written as **data, not code**: an explicit,
versioned, extensible dictionary of technologies, each with multiple independent evidence rules.

Every rule declares:

| Field | Meaning |
|---|---|
| `kind` | which evidence family it reads (`header`, `cookie`, `script_src`, `link_href`, `resource_domain`, `meta`, `attr`, `class_token`, `inline_js`, `url_path`, `script_id`, `html_id`, `input_name`, `custom`) |
| `pattern` | a regex (case-insensitive) applied to that family |
| `strength` | `strong` (1.0) or `weak` (0.4) — how diagnostic the signal is on its own |
| `sufficient` | if `True`, a single match is enough for `HIGH` confidence (used only for signals that are effectively unique to one vendor, e.g. `js.stripe.com`) |

**Design rules followed throughout:**

1. *Never label on a bare word.* "react", "vue" or "stripe" appearing in page text is not evidence. Every rule
   targets a structured location (a URL path, a header, an attribute name, a script id).
2. *Prefer vendor-unique artefacts.* `/_next/static/`, `data-drupal-selector`, `cf-ray`, `wp-content` are
   near-impossible to produce accidentally; generic ids like `#app` or `#root` are only ever `weak`.
3. *Multiple families beat multiple rules in one family.* Confidence rises fastest when independent evidence
   families agree (a header **and** a script URL **and** a cookie name).
4. *Document what cannot be labelled.* Several important technologies are simply not reliably observable in
   archived HTML — see the limitations table below the registry.

To extend the taxonomy, add an entry to `TECHNOLOGY_FINGERPRINTS` and bump `REGISTRY_VERSION`. Nothing else in
the notebook needs to change: label generation, feature redaction and the leakage ablations all read from the
registry.

In [16]:
# ===================== TECHNOLOGY FINGERPRINT REGISTRY ====================
REGISTRY_VERSION = "1.0.0"

def R(kind: str, pattern: str, strength: str = "strong",
      sufficient: bool = False, note: str = "") -> Dict[str, Any]:
    return {"kind": kind, "pattern": pattern, "strength": strength,
            "sufficient": sufficient, "note": note}

TECHNOLOGY_FINGERPRINTS: Dict[str, Dict[str, Any]] = {
    # ------------------------------------------------------------ CMS
    "WordPress": {"category": "CMS", "rules": [
        R("meta", r"generator=wordpress", sufficient=True),
        R("script_src", r"/wp-(content|includes)/", sufficient=True),
        R("link_href", r"/wp-(content|includes)/", sufficient=True),
        R("link_href", r"/wp-json/"),
        R("script_src", r"wp-emoji-release"),
        R("cookie", r"^wordpress"),
        R("url_path", r"/wp-content/", strength="weak"),
        R("html_id", r"^wp-block-", strength="weak"),
    ]},
    "WooCommerce": {"category": "ecommerce", "rules": [
        R("link_href", r"/plugins/woocommerce/", sufficient=True),
        R("script_src", r"/plugins/woocommerce/", sufficient=True),
        R("class_token", r"^woocommerce(-page|-js)?$"),
        R("cookie", r"^woocommerce_"),
    ]},
    "Drupal": {"category": "CMS", "rules": [
        R("meta", r"generator=drupal", sufficient=True),
        R("header", r"x-generator=drupal", sufficient=True),
        R("script_src", r"/core/misc/drupal\.js", sufficient=True),
        R("attr", r"^data-drupal-", sufficient=True),
        R("header", r"x-drupal-cache=.+"),
        R("script_src", r"/sites/(all|default)/(modules|themes|files)/"),
        R("class_token", r"^(node|region|block-block)-\d+$", strength="weak"),
    ]},
    "Joomla": {"category": "CMS", "rules": [
        R("meta", r"generator=joomla", sufficient=True),
        R("script_src", r"/media/(jui|system)/js/", sufficient=True),
        R("url_path", r"option=com_", strength="weak"),
        R("link_href", r"/templates/[^/]+/css/template"),
    ]},
    "Shopify": {"category": "ecommerce", "rules": [
        R("resource_domain", r"^(cdn\.)?shopify(cdn)?\.(com|net)$", sufficient=True),
        R("header", r"x-shopid=\d+", sufficient=True),
        R("header", r"x-shopify-stage=", sufficient=True),
        R("inline_js", r"shopify\.(shop|theme|currency)", sufficient=True),
        R("cookie", r"^_shopify_"),
        R("url_path", r"/cdn/shop/", strength="weak"),
    ]},
    "Wix": {"category": "site-builder", "rules": [
        R("resource_domain", r"^parastorage\.com$", sufficient=True),
        R("resource_domain", r"^wixstatic\.com$", sufficient=True),
        R("meta", r"generator=wix\.com", sufficient=True),
        R("html_id", r"^wix-warmup-data$"),
    ]},
    "Squarespace": {"category": "site-builder", "rules": [
        R("resource_domain", r"^squarespace(-cdn)?\.com$", sufficient=True),
        R("inline_js", r"static\.squarespace_context", sufficient=True),
        R("meta", r"generator=squarespace"),
    ]},
    "Webflow": {"category": "site-builder", "rules": [
        R("resource_domain", r"^website-files\.com$", sufficient=True),
        R("meta", r"generator=webflow", sufficient=True),
        R("attr", r"^data-wf-(page|site)$", sufficient=True),
    ]},
    # ------------------------------------------------------- frontend
    "React": {"category": "frontend", "rules": [
        R("attr", r"^data-react(root|id)$", sufficient=True),
        R("script_src", r"/react(-dom)?[@\-.][\d.]*(\.production|\.development)?(\.min)?\.js", sufficient=True),
        R("inline_js", r"__react_devtools_global_hook__|_reactrootcontainer"),
        R("class_token", r"^jsx-\d+$"),
        R("html_id", r"^(root|react-root|app-root)$", strength="weak"),
        R("script_src", r"/static/js/(main|bundle)\.[a-f0-9]{6,}\.js", strength="weak"),
    ]},
    "Vue.js": {"category": "frontend", "rules": [
        R("attr", r"^data-v-[a-f0-9]{6,8}$", sufficient=True),
        R("script_src", r"/vue[@\-.][\d.]*(\.global|\.runtime)?(\.min|\.prod)?\.js", sufficient=True),
        R("inline_js", r"vue\.createapp|new vue\("),
        R("attr", r"^v-(cloak|if|for|bind|model|show)$"),
        R("html_id", r"^app$", strength="weak"),
    ]},
    "Angular": {"category": "frontend", "rules": [
        R("attr", r"^ng-version$", sufficient=True),
        R("attr", r"^_ng(host|content)-", sufficient=True),
        R("custom", r"tag:app-root", sufficient=True),
        R("attr", r"^ng-(app|controller|repeat)$", note="AngularJS 1.x"),
        R("script_src", r"/(runtime|polyfills|main)\.[a-f0-9]{8,}\.js", strength="weak"),
    ]},
    "Svelte": {"category": "frontend", "rules": [
        R("class_token", r"^svelte-[a-z0-9]{5,}$", sufficient=True),
        R("script_src", r"/_app/immutable/", sufficient=True),
        R("inline_js", r"__sveltekit_"),
    ]},
    "jQuery": {"category": "frontend", "rules": [
        R("script_src", r"/jquery[.\-@][\d.]*(\.min|\.slim)?\.js", sufficient=True),
        R("script_src", r"jquery(-migrate|-ui)?[.\-]", strength="weak"),
        R("inline_js", r"jquery\(document\)\.ready|\$\(document\)\.ready"),
    ]},
    # -------------------------------------------------- meta-framework
    "Next.js": {"category": "meta-framework", "rules": [
        R("script_id", r"^__next_data__$", sufficient=True),
        R("script_src", r"/_next/static/", sufficient=True),
        R("header", r"x-powered-by=next\.js", sufficient=True),
        R("header", r"x-nextjs-cache="),
        R("html_id", r"^__next$"),
    ]},
    "Nuxt.js": {"category": "meta-framework", "rules": [
        R("script_src", r"/_nuxt/", sufficient=True),
        R("inline_js", r"window\.__nuxt__", sufficient=True),
        R("html_id", r"^__nuxt$"),
        R("attr", r"^data-nuxt-", strength="weak"),
    ]},
    "Gatsby": {"category": "meta-framework", "rules": [
        R("html_id", r"^___gatsby$", sufficient=True),
        R("script_src", r"/(webpack-runtime|app)-[a-f0-9]{16,}\.js"),
        R("inline_js", r"window\.___gatsby|___loader"),
        R("link_href", r"/page-data/"),
    ]},
    # -------------------------------------------------------- backend
    "PHP": {"category": "backend", "rules": [
        R("header", r"x-powered-by=php", sufficient=True),
        R("cookie", r"^phpsessid$", sufficient=True),
        R("url_path", r"\.php($|\?)"),
    ]},
    "Laravel": {"category": "backend", "rules": [
        R("cookie", r"^laravel_session$", sufficient=True),
        R("cookie", r"^xsrf-token$", strength="weak"),
        R("meta", r"csrf-token=.{20,}", strength="weak"),
    ]},
    "Django": {"category": "backend", "rules": [
        R("cookie", r"^csrftoken$"),
        R("cookie", r"^django_language$", sufficient=True),
        R("input_name", r"^csrfmiddlewaretoken$", sufficient=True),
        R("url_path", r"/static/admin/", strength="weak"),
    ]},
    "Ruby on Rails": {"category": "backend", "rules": [
        R("meta", r"csrf-param=authenticity_token", sufficient=True),
        R("header", r"x-runtime=[\d.]+"),
        R("cookie", r"^_[a-z0-9_]+_session$", strength="weak"),
        R("script_src", r"/assets/application-[a-f0-9]{16,}\.js"),
    ]},
    "Express": {"category": "backend", "rules": [
        R("header", r"x-powered-by=express", sufficient=True),
    ]},
    "ASP.NET": {"category": "backend", "rules": [
        R("header", r"x-aspnet(mvc)?-version=", sufficient=True),
        R("header", r"x-powered-by=asp\.net", sufficient=True),
        R("input_name", r"^__viewstate$", sufficient=True),
        R("cookie", r"^asp\.net_sessionid$", sufficient=True),
        R("url_path", r"\.aspx($|\?)"),
    ]},
    # --------------------------------------------------- CDN / server
    "Cloudflare": {"category": "CDN", "rules": [
        R("header", r"server=cloudflare", sufficient=True),
        R("header", r"cf-ray=", sufficient=True),
        R("script_src", r"/cdn-cgi/(scripts|challenge)"),
        R("header", r"cf-cache-status="),
    ]},
    "Fastly": {"category": "CDN", "rules": [
        R("header", r"x-fastly-request-id=", sufficient=True),
        R("header", r"x-served-by=cache-[a-z]{3}\d+", sufficient=True),
        R("header", r"x-timer=s\d"),
    ]},
    "Akamai": {"category": "CDN", "rules": [
        R("header", r"x-akamai-(transformed|request-id)=", sufficient=True),
        R("header", r"server=akamaighost", sufficient=True),
        R("resource_domain", r"^akamai(zed|hd)\.net$"),
    ]},
    "Amazon CloudFront": {"category": "CDN", "rules": [
        R("header", r"x-amz-cf-id=", sufficient=True),
        R("header", r"server=cloudfront", sufficient=True),
        R("resource_domain", r"^cloudfront\.net$"),
    ]},
    "Vercel": {"category": "hosting", "rules": [
        R("header", r"x-vercel-(id|cache)=", sufficient=True),
        R("header", r"server=vercel", sufficient=True),
    ]},
    "Netlify": {"category": "hosting", "rules": [
        R("header", r"x-nf-request-id=", sufficient=True),
        R("header", r"server=netlify", sufficient=True),
    ]},
    "Nginx": {"category": "server", "rules": [
        R("header", r"server=nginx", sufficient=True),
    ]},
    "Apache": {"category": "server", "rules": [
        R("header", r"server=apache", sufficient=True),
    ]},
    "LiteSpeed": {"category": "server", "rules": [
        R("header", r"server=litespeed", sufficient=True),
        R("header", r"x-litespeed-cache="),
    ]},
    "Microsoft IIS": {"category": "server", "rules": [
        R("header", r"server=microsoft-iis", sufficient=True),
    ]},
    # ------------------------------------------------------ analytics
    "Google Analytics": {"category": "analytics", "rules": [
        R("resource_domain", r"^google-analytics\.com$", sufficient=True),
        R("script_src", r"googletagmanager\.com/gtag/js", sufficient=True),
        R("inline_js", r"ga\('create'|_gaq\.push|gtag\('config'"),
    ]},
    "Google Tag Manager": {"category": "analytics", "rules": [
        R("script_src", r"googletagmanager\.com/gtm\.js", sufficient=True),
        R("inline_js", r"gtm-[a-z0-9]{4,9}"),
        R("custom", r"iframe_gtm", strength="weak"),
    ]},
    "Matomo": {"category": "analytics", "rules": [
        R("script_src", r"/(matomo|piwik)\.js", sufficient=True),
        R("inline_js", r"_paq\.push"),
    ]},
    "Meta Pixel": {"category": "analytics", "rules": [
        R("script_src", r"connect\.facebook\.net/.*/fbevents\.js", sufficient=True),
        R("inline_js", r"fbq\('init'"),
    ]},
    "Hotjar": {"category": "analytics", "rules": [
        R("resource_domain", r"^hotjar\.com$", sufficient=True),
        R("inline_js", r"_hjsettings"),
    ]},
    "HubSpot": {"category": "marketing", "rules": [
        R("resource_domain", r"^(hs-scripts|hubspot|hsforms)\.(com|net)$", sufficient=True),
        R("header", r"x-hs-hub-id="),
    ]},
    # ------------------------------------------------------- payments
    "Stripe": {"category": "payments", "rules": [
        R("resource_domain", r"^stripe\.(com|network)$", sufficient=True),
        R("script_src", r"js\.stripe\.com", sufficient=True),
    ]},
    "PayPal": {"category": "payments", "rules": [
        R("resource_domain", r"^paypal(objects)?\.com$", sufficient=True),
        R("script_src", r"paypal\.com/sdk/js", sufficient=True),
    ]},
    # --------------------------------------------------------- CSS/UI
    "Bootstrap": {"category": "CSS/UI", "rules": [
        R("link_href", r"bootstrap[.\-@][\d.]*(/dist/css)?/?[a-z.]*\.min\.css", sufficient=True),
        R("link_href", r"/bootstrap(\.min)?\.css", sufficient=True),
        R("attr", r"^data-bs-(toggle|target|ride)$", sufficient=True),
        R("class_token", r"^col-(xs|sm|md|lg|xl)-\d+$"),
        R("class_token", r"^(navbar-nav|btn-primary|form-control|card-body)$"),
    ]},
    "Tailwind CSS": {"category": "CSS/UI", "rules": [
        R("script_src", r"cdn\.tailwindcss\.com", sufficient=True),
        R("link_href", r"tailwind(\.min)?\.css", sufficient=True),
        R("custom", r"tailwind_utility_density", sufficient=True,
          note=">=8 distinct Tailwind-shaped utility classes on one page"),
    ]},
    "Font Awesome": {"category": "CSS/UI", "rules": [
        R("link_href", r"font-?awesome", sufficient=True),
        R("class_token", r"^fa-[a-z0-9-]+$"),
    ]},
    "Google Fonts": {"category": "CSS/UI", "rules": [
        R("resource_domain", r"^(googleapis|gstatic)\.com$", strength="weak"),
        R("link_href", r"fonts\.googleapis\.com", sufficient=True),
    ]},
    "reCAPTCHA": {"category": "security", "rules": [
        R("script_src", r"(google\.com|recaptcha\.net)/recaptcha/", sufficient=True),
        R("class_token", r"^g-recaptcha$"),
    ]},
}

# Implication edges. These are recorded at MEDIUM confidence on purpose (see S10) so they
# never become free supervised positives and never inflate the co-occurrence analysis.
TECH_IMPLICATIONS = {
    "Next.js": ["React"], "Gatsby": ["React"], "Nuxt.js": ["Vue.js"],
    "WooCommerce": ["WordPress", "PHP"], "Laravel": ["PHP"], "Drupal": ["PHP"],
    "Joomla": ["PHP"],
}

TECH_CATEGORY = {t: v["category"] for t, v in TECHNOLOGY_FINGERPRINTS.items()}
STRENGTH_WEIGHT = {"strong": 1.0, "weak": 0.4}

print(f"registry v{REGISTRY_VERSION}: {len(TECHNOLOGY_FINGERPRINTS)} technologies, "
      f"{sum(len(v['rules']) for v in TECHNOLOGY_FINGERPRINTS.values())} rules, "
      f"{len(set(TECH_CATEGORY.values()))} categories")
pd.Series(Counter(TECH_CATEGORY.values())).sort_values(ascending=False).to_frame("technologies").T

registry v1.0.0: 45 technologies, 148 rules, 14 categories


,backend,frontend,analytics,CSS/UI,CDN,server,CMS,meta-framework,site-builder,ecommerce,hosting,payments,marketing,security
technologies,6,5,5,4,4,4,3,3,3,2,2,2,1,1


### 9b. Technologies that this evidence source cannot label reliably

Rather than fabricating labels, these are documented as out of scope. They are excluded from the taxonomy or, where
included, flagged in the limitations section.

| Technology / class | Why archived HTML is insufficient |
|---|---|
| Bundled SPA frameworks with no runtime marker | A production React/Vue/Svelte build compiled through Webpack/Vite often emits no framework-specific attribute at all. Detection therefore skews toward CDN-loaded or SSR-hydrated deployments, so **React recall is structurally limited** and its measured prevalence is a floor, not an estimate. |
| Server-side languages behind a reverse proxy | Modern deployments suppress `X-Powered-By`. Absence of a PHP header is not absence of PHP. |
| Databases, queues, caches | Never observable client-side. Deliberately absent from the taxonomy. |
| Server versions / patch levels | Version strings are frequently redacted, and inferring them invites exactly the vulnerability-mapping use this project excludes. Versions are not parsed or stored. |
| A/B-tested or per-visit-injected tags | Common Crawl captures one visit; tags loaded conditionally may be missing. |
| Technologies injected purely at runtime by JS | Common Crawl does **not** execute JavaScript. Anything that only appears after hydration is invisible. This is the single largest source of false negatives in the whole project. |

## 10. Automatic label construction

Labels come from weak supervision with an explicit confidence ladder, and the ladder feeds a **trust mask** that
governs both training and evaluation.

```text
score        = sum over matched rules of STRENGTH_WEIGHT[strength]
n_strong     = number of matched rules with strength == "strong"
n_families   = number of distinct evidence kinds that matched

HIGH     any sufficient rule matched
         OR (score >= 1.4 AND n_families >= 2)
         OR n_strong >= 2
MEDIUM   score >= 1.0                      (one strong rule, nothing corroborating)
LOW      0 < score < 1.0                   (weak evidence only)
ABSTAIN  score == 0                        (no evidence at all)
```

**How the ladder becomes supervision:**

| Confidence | `y` | `mask` | Rationale |
|---|---|---|---|
| `HIGH` | 1 | 1 | Trusted positive |
| `MEDIUM` / `LOW` | 1 / 0 | **0** | Ambiguous — excluded from loss *and* from metrics rather than guessed at |
| `ABSTAIN` | 0 | 1 | Trusted negative — no evidence of any kind was found |

Masking is what makes the weak labels defensible. A page with one weak jQuery hint is neither a clean positive nor
a clean negative, and forcing it to be either would inject noise into training and then *hide* that noise inside
the metrics. Every score reported in this notebook is computed over trusted cells only, and the masked fraction is
reported alongside so the reader can see how much of the label matrix was set aside.

The obvious residual risk — that "trusted negative" really means "evidence was not visible" — is real and is
quantified in the limitations section. It biases recall estimates optimistically for technologies whose markers
are easy to strip.

In [17]:
# --- compile the registry into fast matchers ------------------------------
CONF_ORDER = {"abstain": 0, "low": 1, "medium": 2, "high": 3}

TAILWIND_UTILITY = re.compile(
    r"^(?:(?:sm|md|lg|xl|2xl|hover|focus|group-hover|dark|active|disabled):)*"
    r"(?:-?(?:m|p)[xytrbl]?-\d|-?(?:m|p)[xytrbl]?-(?:px|auto|\d+(?:\.5)?)"
    r"|text-(?:xs|sm|base|lg|xl|\dxl|left|center|right|gray-\d{2,3}|white|black)"
    r"|bg-(?:white|black|transparent|gray-\d{2,3}|blue-\d{2,3}|red-\d{2,3}|green-\d{2,3})"
    r"|flex(?:-(?:col|row|wrap|1|none))?|grid(?:-cols-\d+)?|items-(?:center|start|end|stretch)"
    r"|justify-(?:center|between|start|end|around)|gap-\d+|space-[xy]-\d+"
    r"|w-(?:full|screen|\d+(?:/\d+)?)|h-(?:full|screen|\d+)|max-w-(?:xs|sm|md|lg|xl|\dxl|full)"
    r"|rounded(?:-(?:sm|md|lg|xl|full))?|shadow(?:-(?:sm|md|lg|xl))?|border(?:-\d)?"
    r"|font-(?:thin|light|normal|medium|semibold|bold|extrabold)|leading-(?:none|tight|normal|relaxed)"
    r"|opacity-\d{1,3}|z-\d{1,2}|overflow-(?:hidden|auto|scroll)|absolute|relative|sticky|hidden|block|inline-block"
    r")$")

@dataclass
class CompiledRule:
    tech: str
    kind: str
    pattern: str
    regex: Any
    strength: str
    sufficient: bool
    rule_id: str
    note: str = ""

def compile_registry(registry: Dict[str, Dict[str, Any]]) -> List[CompiledRule]:
    out: List[CompiledRule] = []
    for tech, spec in registry.items():
        for i, r in enumerate(spec["rules"]):
            rid = f"{tech}::{r['kind']}::{i}"
            rx = None if r["kind"] == "custom" else re.compile(r["pattern"], re.I)
            out.append(CompiledRule(tech=tech, kind=r["kind"], pattern=r["pattern"], regex=rx,
                                    strength=r["strength"], sufficient=r["sufficient"],
                                    rule_id=rid, note=r.get("note", "")))
    return out

COMPILED_RULES = compile_registry(TECHNOLOGY_FINGERPRINTS)
RULES_BY_TECH: Dict[str, List[CompiledRule]] = defaultdict(list)
for cr in COMPILED_RULES:
    RULES_BY_TECH[cr.tech].append(cr)
RULE_IDS = [cr.rule_id for cr in COMPILED_RULES]
RULE_INDEX = {rid: i for i, rid in enumerate(RULE_IDS)}
LOG.info("compiled %d rules across %d technologies", len(COMPILED_RULES), len(RULES_BY_TECH))

13:17:02 | INFO    | compiled 148 rules across 45 technologies


In [18]:
# --- evidence view: the strings each rule kind is allowed to read ---------
def evidence_view(row: Any) -> Dict[str, List[str]]:
    """Flatten one evidence row into per-kind lists of lowercase strings."""
    headers = row["headers"] if isinstance(row["headers"], dict) else {}
    metas = row["metas"] if isinstance(row["metas"], dict) else {}
    scripts = row["scripts"] if isinstance(row["scripts"], list) else []
    links = row["links"] if isinstance(row["links"], list) else []
    resources = row["resource_domains"] if isinstance(row["resource_domains"], dict) else {}
    tags = row["tag_counts"] if isinstance(row["tag_counts"], dict) else {}
    classes = row["class_tokens"] if isinstance(row["class_tokens"], list) else []

    view = {
        "header":          [f"{k}={v}".lower() for k, v in headers.items()],
        "cookie":          [str(c).lower() for c in (row["cookie_names"] or [])],
        "meta":            [f"{k}={v}".lower() for k, v in metas.items()],
        "script_src":      [str(s.get("src", "")).lower() for s in scripts if s.get("src")],
        "script_id":       [str(s.get("id", "")).lower() for s in scripts if s.get("id")]
                           + [str(x).lower() for x in (row["script_ids"] or [])],
        "link_href":       [str(l.get("href", "")).lower() for l in links if l.get("href")],
        "resource_domain": [str(d).lower() for d in resources.keys()],
        "class_token":     [str(c).lower() for c in classes],
        "attr":            [str(a).lower() for a in (row["attr_names"] or [])],
        "html_id":         [str(i).lower() for i in (row["html_ids"] or [])],
        "input_name":      [str(i).lower() for i in (row["input_names"] or [])],
        "inline_js":       [str(row["inline_js"] or "")],
        "url_path":        [str(row["url"] or "").lower()],
        "custom":          [],
    }
    # Resource domains are stored as eTLD+1; also expose the bare label so
    # patterns like ^stripe\.(com)$ match cdn hosts consistently.
    view["custom_state"] = {"tags": tags, "classes": classes}
    return view

def _custom_match(pattern: str, row: Any, view: Dict[str, Any]) -> bool:
    """Structural predicates that are not a single regex over one string list."""
    if pattern == "tag:app-root":
        return int((view["custom_state"]["tags"] or {}).get("app-root", 0)) > 0
    if pattern == "tailwind_utility_density":
        toks = view["custom_state"]["classes"] or []
        hits = {t for t in toks if TAILWIND_UTILITY.match(t)}
        return len(hits) >= 8
    if pattern == "iframe_gtm":
        return int(row.get("n_iframes", 0) or 0) > 0
    return False

def match_rules(row: Any) -> Dict[str, List[CompiledRule]]:
    """Return matched rules grouped by technology."""
    view = evidence_view(row)
    hits: Dict[str, List[CompiledRule]] = defaultdict(list)
    for cr in COMPILED_RULES:
        if cr.kind == "custom":
            if _custom_match(cr.pattern, row, view):
                hits[cr.tech].append(cr)
            continue
        for s in view.get(cr.kind, ()):
            if s and cr.regex.search(s):
                hits[cr.tech].append(cr)
                break
    return hits

def score_technology(rules: List[CompiledRule]) -> Tuple[str, float, int, int]:
    """Aggregate matched rules into a confidence level."""
    if not rules:
        return "abstain", 0.0, 0, 0
    score = sum(STRENGTH_WEIGHT[r.strength] for r in rules)
    n_strong = sum(1 for r in rules if r.strength == "strong")
    families = len({r.kind for r in rules})
    if any(r.sufficient for r in rules):
        return "high", score, n_strong, families
    if (score >= 1.4 and families >= 2) or n_strong >= 2:
        return "high", score, n_strong, families
    if score >= 1.0:
        return "medium", score, n_strong, families
    return "low", score, n_strong, families

In [19]:
# --- label the corpus -----------------------------------------------------
def label_corpus(df: pd.DataFrame, apply_implications: bool = True) -> Tuple[pd.DataFrame, sp.csr_matrix]:
    """Produce a per-page confidence table plus a sparse rule-hit matrix.

    Returns
    -------
    conf_df : DataFrame [n_pages x n_tech] of confidence strings
    hits    : csr_matrix [n_pages x n_rules] binary rule-activation matrix
              (this is the raw material for the fingerprint feature channel and,
               crucially, for the redaction sets used by the ablations)
    """
    techs = sorted(TECHNOLOGY_FINGERPRINTS)
    tech_idx = {t: i for i, t in enumerate(techs)}
    n = len(df)
    conf = np.zeros((n, len(techs)), dtype=np.int8)          # CONF_ORDER codes
    rows_i: List[int] = []; cols_i: List[int] = []
    imp_conf = CONF_ORDER[CONFIG["IMPLICATION_CONFIDENCE"]]

    pbar = tqdm(total=n, desc="labelling")
    for offset, chunk in parsed_chunks(df):
        for k, row in enumerate(chunk.to_dict("records")):
            i = offset + k
            hits = match_rules(row)
            for tech, rules in hits.items():
                level, _, _, _ = score_technology(rules)
                conf[i, tech_idx[tech]] = max(conf[i, tech_idx[tech]], CONF_ORDER[level])
                for r in rules:
                    rows_i.append(i); cols_i.append(RULE_INDEX[r.rule_id])
            if apply_implications:
                for src, targets in TECH_IMPLICATIONS.items():
                    if conf[i, tech_idx[src]] == CONF_ORDER["high"]:
                        for tgt in targets:
                            j = tech_idx.get(tgt)
                            if j is not None and conf[i, j] < imp_conf:
                                conf[i, j] = imp_conf      # never overrides a HIGH direct label
        pbar.update(len(chunk))
    pbar.close()
    inv = {v: k for k, v in CONF_ORDER.items()}
    conf_df = pd.DataFrame(np.vectorize(inv.get)(conf), columns=techs, index=df.index)
    hits_mat = sp.csr_matrix((np.ones(len(rows_i), dtype=np.int8), (rows_i, cols_i)),
                             shape=(n, len(RULE_IDS)))
    return conf_df, hits_mat

t0 = time.time()
CONF_DF, RULE_HITS = label_corpus(EV, CONFIG["APPLY_IMPLICATIONS"])
LOG.info("labelled %d pages in %.1fs", len(EV), time.time() - t0)
RUN["labelling_seconds"] = round(time.time() - t0, 1)
CONF_DF.head(5)

labelling:   0%|          | 0/100003 [00:00<?, ?it/s]

13:19:13 | INFO    | labelled 100003 pages in 131.0s


,ASP.NET,Akamai,Amazon CloudFront,Angular,Apache,Bootstrap,Cloudflare,Django,Drupal,Express,...,Svelte,Tailwind CSS,Vercel,Vue.js,Webflow,Wix,WooCommerce,WordPress,jQuery,reCAPTCHA
0,abstain,abstain,abstain,abstain,abstain,medium,abstain,abstain,abstain,abstain,...,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,low,abstain
1,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,...,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,low,abstain
2,abstain,abstain,abstain,abstain,high,abstain,abstain,abstain,abstain,abstain,...,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,high,abstain
3,abstain,abstain,abstain,abstain,abstain,abstain,high,abstain,abstain,abstain,...,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain
4,abstain,abstain,abstain,abstain,abstain,abstain,high,abstain,abstain,abstain,...,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain,abstain


## 11. Label quality analysis

Before a single model is trained, the label matrix is audited: how many trusted positives exist per technology,
how much of the matrix is masked, and which rules are actually doing the work. A technology whose positives come
from exactly one rule is fragile — its "detection" is that rule, and its ablation score will collapse.

In [20]:
# --- confidence tallies and label matrices --------------------------------
def build_label_matrices(conf_df: pd.DataFrame,
                         positive_level: str = "high",
                         mask_uncertain: bool = True) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Map confidence strings to (y, mask). mask=0 -> cell excluded from loss and metrics."""
    techs = list(conf_df.columns)
    codes = conf_df.replace(CONF_ORDER).to_numpy(dtype=np.int8)
    pos = CONF_ORDER[positive_level]
    y = (codes >= pos).astype(np.int8)
    if mask_uncertain:
        uncertain = (codes > CONF_ORDER["abstain"]) & (codes < pos)
        mask = (~uncertain).astype(np.uint8)
    else:
        mask = np.ones_like(y, dtype=np.uint8)
    return y, mask, techs

Y_ALL, M_ALL, TECHS_ALL = build_label_matrices(
    CONF_DF, CONFIG["POSITIVE_CONFIDENCE"], CONFIG["MASK_UNCERTAIN"])

def label_quality_table(conf_df: pd.DataFrame, y: np.ndarray, m: np.ndarray,
                        hits: sp.csr_matrix) -> pd.DataFrame:
    rows = []
    hits_csc = hits.tocsc()
    for j, tech in enumerate(conf_df.columns):
        col = conf_df[tech]
        counts = col.value_counts()
        trusted = int(m[:, j].sum())
        pos = int(((y[:, j] == 1) & (m[:, j] == 1)).sum())
        # which rules produced this technology's evidence?
        rule_positions = [RULE_INDEX[r.rule_id] for r in RULES_BY_TECH[tech]]
        rule_counts = {RULE_IDS[p].split("::", 1)[1]: int(hits_csc[:, p].sum()) for p in rule_positions}
        active = {k: v for k, v in rule_counts.items() if v > 0}
        top_rule, top_n = (max(active.items(), key=lambda kv: kv[1]) if active else ("-", 0))
        rows.append({
            "technology": tech, "category": TECH_CATEGORY[tech],
            "high": int(counts.get("high", 0)), "medium": int(counts.get("medium", 0)),
            "low": int(counts.get("low", 0)), "abstain": int(counts.get("abstain", 0)),
            "positives": pos, "negatives": trusted - pos,
            "trusted_frac": round(trusted / max(1, len(conf_df)), 4),
            "prevalence": round(pos / max(1, trusted), 5),
            "n_active_rules": len(active),
            "dominant_rule": top_rule,
            "dominant_rule_share": round(top_n / max(1, pos), 3) if pos else 0.0,
        })
    return pd.DataFrame(rows).sort_values("positives", ascending=False).reset_index(drop=True)

LABEL_QUALITY = label_quality_table(CONF_DF, Y_ALL, M_ALL, RULE_HITS)
LABEL_QUALITY.to_csv(PATHS["reports"] / "label_quality.csv", index=False)
RESULTS["label_quality"] = LABEL_QUALITY.to_dict("records")

print(f"label matrix       : {Y_ALL.shape[0]:,} pages x {Y_ALL.shape[1]} technologies")
print(f"trusted cells      : {M_ALL.mean():.3%}")
print(f"masked (uncertain) : {1 - M_ALL.mean():.3%}")
print(f"positive rate      : {(Y_ALL[M_ALL == 1] == 1).mean():.3%} of trusted cells")
display(LABEL_QUALITY.head(25))

label matrix       : 100,003 pages x 45 technologies
trusted cells      : 97.300%
masked (uncertain) : 2.700%
positive rate      : 7.475% of trusted cells


,technology,category,high,medium,low,abstain,positives,negatives,trusted_frac,prevalence,n_active_rules,dominant_rule,dominant_rule_share
0,WordPress,CMS,40385,82,55,59481,40385,59481,0.9986,0.40439,8,link_href::2,0.990
1,Google Fonts,CSS/UI,35355,0,4967,59681,35355,59681,0.9503,0.37202,2,resource_domain::0,1.068
2,Font Awesome,CSS/UI,28283,7872,0,63848,28283,63848,0.9213,0.30699,2,link_href::0,1.000
3,Google Analytics,analytics,26289,2048,0,71666,26289,71666,0.9795,0.26838,3,script_src::1,0.985
4,PHP,backend,26287,8150,0,65566,26287,65566,0.9185,0.28619,3,header::0,0.675
5,Nginx,server,26242,0,0,73761,26242,73761,1.0000,0.26241,1,header::0,1.000
6,Bootstrap,CSS/UI,21924,8236,0,69843,21924,69843,0.9176,0.23891,5,class_token::3,0.990
7,Apache,server,21881,0,0,78122,21881,78122,1.0000,0.21880,1,header::0,1.000
8,jQuery,frontend,21754,1810,38142,38297,21754,38297,0.6005,0.36226,3,script_src::1,2.753
9,Cloudflare,CDN,17994,38,0,81971,17994,81971,0.9996,0.18000,4,header::1,1.000


In [21]:
# --- label quality figures ------------------------------------------------
if CONFIG["MAKE_FIGURES"] and len(LABEL_QUALITY):
    top = LABEL_QUALITY.head(28).iloc[::-1]
    fig, axes = plt.subplots(1, 2, figsize=(15, 9))
    ax = axes[0]
    ax.barh(top["technology"], top["high"], label="HIGH (trusted positive)", color="#2b6cb0")
    ax.barh(top["technology"], top["medium"], left=top["high"], label="MEDIUM (masked)", color="#dd9b3c")
    ax.barh(top["technology"], top["low"], left=top["high"] + top["medium"], label="LOW (masked)", color="#c05050")
    ax.set_xlabel("pages"); ax.set_title("Evidence confidence by technology"); ax.legend(loc="lower right")
    ax = axes[1]
    ax.barh(top["technology"], top["dominant_rule_share"], color="#4a5568")
    ax.axvline(1.0, ls="--", c="k", lw=1)
    ax.set_xlim(0, 1.05); ax.set_xlabel("share of positives from the single most active rule")
    ax.set_title("Label fragility (1.0 = one rule explains every positive)")
    savefig(fig, "01_label_quality")

    fig, ax = plt.subplots(figsize=(9, 5))
    prev = LABEL_QUALITY[LABEL_QUALITY["positives"] > 0].sort_values("prevalence", ascending=False)
    ax.bar(range(len(prev)), prev["prevalence"], color="#2b6cb0")
    ax.set_yscale("log"); ax.set_xticks(range(len(prev)))
    ax.set_xticklabels(prev["technology"], rotation=90, fontsize=7)
    ax.set_ylabel("prevalence in sample (log)")
    ax.set_title("Technology prevalence - long tail drives the class-imbalance strategy")
    savefig(fig, "02_prevalence")
print("label-quality figures written")

13:19:23 | INFO    | figure saved: 01_label_quality.png
13:19:24 | INFO    | figure saved: 02_prevalence.png


label-quality figures written


## 12. Deduplication

Common Crawl contains the same page many times over: mirrors, session-parameter variants, syndicated templates,
and re-captures of a URL within one crawl. Left alone, these inflate every metric — a duplicated page in train and
test is test-set contamination even when the domains differ.

Four passes, cheapest first:

1. **Exact URL** — same string.
2. **Normalised URL** — scheme/`www.`/trailing-slash/index-file/tracking-parameter stripped, so
   `http://www.a.com/x/?utm_source=z` collapses into `a.com/x`.
3. **Exact content** — SHA-1 of the captured HTML bytes.
4. **Near-duplicate** — 64-bit SimHash over token shingles of the structural signature (tag histogram + resource
   domains + text), banded into 4x16-bit buckets, Hamming distance <= 3 within a bucket.

SimHash on the *structural* signature rather than raw text is deliberate: it catches template farms and mass-hosted
sites that share a stack but differ in copy, which are exactly the pages that would otherwise let a model memorise
one template and score well on hundreds of near-identical test pages.

In [22]:
# --- multi-level deduplication --------------------------------------------
_TRACKING_PARAMS = re.compile(r"(utm_[a-z]+|gclid|fbclid|mc_[a-z]+|ref|sessionid|phpsessid|sid)=", re.I)

def normalise_url(url: str) -> str:
    try:
        p = urlsplit(url)
    except Exception:
        return (url or "").lower()
    host = p.netloc.lower().split(":")[0]
    if host.startswith("www."):
        host = host[4:]
    path = re.sub(r"/(index|default|home)\.(html?|php|aspx?)$", "/", p.path or "/")
    path = re.sub(r"//+", "/", path).rstrip("/") or "/"
    query = "&".join(sorted(q for q in (p.query or "").split("&")
                            if q and not _TRACKING_PARAMS.match(q)))
    return f"{host}{path}?{query}" if query else f"{host}{path}"

def simhash64(tokens: Sequence[str]) -> int:
    if not tokens:
        return 0
    acc = np.zeros(64, dtype=np.int32)
    for tok in tokens:
        h = int.from_bytes(hashlib.blake2b(tok.encode("utf-8", "ignore"), digest_size=8).digest(), "big")
        for b in range(64):
            acc[b] += 1 if (h >> b) & 1 else -1
    out = 0
    for b in range(64):
        if acc[b] > 0:
            out |= (1 << b)
    return out

def structural_tokens(row: Any) -> List[str]:
    tags = row["tag_counts"] if isinstance(row["tag_counts"], dict) else {}
    res = row["resource_domains"] if isinstance(row["resource_domains"], dict) else {}
    toks = [f"t:{k}:{min(int(v), 50)}" for k, v in tags.items()]
    toks += [f"r:{k}" for k in res.keys()]
    toks += [f"w:{w}" for w in str(row.get("text_sample") or "").lower().split()[:80]]
    toks += [f"c:{c}" for c in (row["class_tokens"] or [])[:30]]
    return toks

def deduplicate(df: pd.DataFrame, near_dup_hamming: int = 3) -> Tuple[pd.DataFrame, Dict[str, int]]:
    stats = {"records_before": len(df), "domains_before": int(df["domain"].nunique())}
    keep = pd.Series(True, index=df.index)

    dup_url = df.duplicated(subset=["url"], keep="first")
    keep &= ~dup_url
    stats["exact_url_duplicates"] = int(dup_url.sum())

    norm = df["url"].map(normalise_url)
    dup_norm = norm.duplicated(keep="first") & keep
    keep &= ~dup_norm
    stats["normalised_url_duplicates"] = int(dup_norm.sum())

    dup_hash = df["html_sha1"].duplicated(keep="first") & keep
    keep &= ~dup_hash
    stats["exact_content_duplicates"] = int(dup_hash.sum())

    sub = df[keep]
    hash_list: List[int] = []
    pbar = tqdm(total=len(sub), desc="simhash")
    for _, chunk in parsed_chunks(sub):
        hash_list.extend(simhash64(structural_tokens(r)) for r in chunk.to_dict("records"))
        pbar.update(len(chunk))
    pbar.close()
    hashes = np.array(hash_list, dtype=np.uint64)
    del hash_list
    seen_bands: Dict[Tuple[int, int], List[int]] = defaultdict(list)
    near_dup_positions: Set[int] = set()
    for pos, h in enumerate(hashes):
        hi = int(h)
        if pos in near_dup_positions:
            continue
        is_dup = False
        bands = [(b, (hi >> (16 * b)) & 0xFFFF) for b in range(4)]
        for band in bands:
            for other in seen_bands[band]:
                if bin(hi ^ int(hashes[other])).count("1") <= near_dup_hamming:
                    is_dup = True; break
            if is_dup:
                break
        if is_dup:
            near_dup_positions.add(pos)
        else:
            for band in bands:
                seen_bands[band].append(pos)
    near_idx = sub.index[sorted(near_dup_positions)]
    keep.loc[near_idx] = False
    stats["near_duplicates"] = int(len(near_idx))

    out = df[keep].copy()
    stats["records_after"] = len(out)
    stats["domains_after"] = int(out["domain"].nunique())
    stats["removed_total"] = stats["records_before"] - stats["records_after"]
    stats["removed_pct"] = round(100 * stats["removed_total"] / max(1, stats["records_before"]), 2)
    return out, stats

t0 = time.time()
EV_DEDUP, DEDUP_STATS = deduplicate(EV)
KEEP_POS = EV.index.get_indexer(EV_DEDUP.index)          # row positions kept, for Y/M/hits alignment
Y_ALL, M_ALL = Y_ALL[KEEP_POS], M_ALL[KEEP_POS]
RULE_HITS = RULE_HITS[KEEP_POS]
CONF_DF = CONF_DF.loc[EV_DEDUP.index]
EV_DEDUP = EV_DEDUP.reset_index(drop=True)
CONF_DF = CONF_DF.reset_index(drop=True)

DEDUP_STATS["seconds"] = round(time.time() - t0, 1)
RESULTS["deduplication"] = DEDUP_STATS
save_json(DEDUP_STATS, PATHS["reports"] / "deduplication.json")
free("EV")   # release the pre-dedup frame (~4 GB at 100k pages)
for k, v in DEDUP_STATS.items():
    print(f"{k:<28} {v}")

simhash:   0%|          | 0/99451 [00:00<?, ?it/s]

records_before               100003
domains_before               88981
exact_url_duplicates         3
normalised_url_duplicates    17
exact_content_duplicates     532
near_duplicates              2964
records_after                96487
domains_after                86554
removed_total                3516
removed_pct                  3.52
seconds                      444.5


## 13. Dataset construction

Technologies below `MIN_TECH_SUPPORT` trusted positives are dropped: a label with 12 positives cannot be
meaningfully split three ways, tuned on a validation set, and then evaluated. Dropping them is a modelling
decision, not a claim that they are absent — the label-quality table above retains their counts, and the report
lists exactly which technologies were excluded and why.

In [23]:
# --- filter the label space and freeze the modelling dataset --------------
def select_technologies(y: np.ndarray, m: np.ndarray, techs: List[str],
                        min_support: int) -> Tuple[List[str], pd.DataFrame]:
    rows, keep = [], []
    for j, t in enumerate(techs):
        trusted = int(m[:, j].sum())
        pos = int(((y[:, j] == 1) & (m[:, j] == 1)).sum())
        neg = trusted - pos
        ok = pos >= min_support and neg >= min_support
        rows.append({"technology": t, "positives": pos, "negatives": neg,
                     "kept": ok, "reason": "" if ok else
                     ("too few positives" if pos < min_support else "too few negatives")})
        if ok:
            keep.append(t)
    return keep, pd.DataFrame(rows)

MIN_SUPPORT = int(CONFIG["MIN_TECH_SUPPORT"])
TECHS, SUPPORT_TABLE = select_technologies(Y_ALL, M_ALL, TECHS_ALL, MIN_SUPPORT)
if not TECHS:
    raise RuntimeError(f"no technology reached MIN_TECH_SUPPORT={MIN_SUPPORT}. "
                       f"Lower it or ingest more pages.")
KEEP_COLS = [TECHS_ALL.index(t) for t in TECHS]
Y = Y_ALL[:, KEEP_COLS]
M = M_ALL[:, KEEP_COLS]

SUPPORT_TABLE.to_csv(PATHS["reports"] / "technology_support.csv", index=False)
RESULTS["dropped_technologies"] = SUPPORT_TABLE.loc[~SUPPORT_TABLE["kept"], "technology"].tolist()
RESULTS["modelled_technologies"] = TECHS

print(f"min support     : {MIN_SUPPORT}")
print(f"modelled techs  : {len(TECHS)} / {len(TECHS_ALL)}")
print(f"dropped         : {', '.join(RESULTS['dropped_technologies']) or 'none'}")
print(f"label matrix    : {Y.shape}")
print(f"labels per page : mean {Y[M == 1].mean() * Y.shape[1]:.2f} "
      f"(mean trusted positives/page = {(Y * M).sum(1).mean():.2f})")
display(SUPPORT_TABLE[~SUPPORT_TABLE["kept"]].head(20))

min support     : 150
modelled techs  : 39 / 45
dropped         : Gatsby, Hotjar, Matomo, Meta Pixel, Squarespace, Svelte
label matrix    : (96487, 39)
labels per page : mean 3.45 (mean trusted positives/page = 3.34)


,technology,positives,negatives,kept,reason
12,Gatsby,90,96330,False,too few positives
16,Hotjar,4,96151,False,too few positives
21,Matomo,69,94875,False,too few positives
22,Meta Pixel,8,96421,False,too few positives
33,Squarespace,17,96470,False,too few positives
35,Svelte,90,96360,False,too few positives


## 14. Domain-disjoint train / validation / test split

Random page-level splitting would be fatal here. Sites are internally consistent — every page on `example.com`
shares one stack, one CDN and one template — so a random split lets the model memorise a domain in training and
recognise it in test. The measured score would be *page* recognition dressed up as *technology* detection.

The split is therefore over **registrable domains**, assigned greedily in shuffled order to whichever split is
furthest below its page-count target. This keeps the page-level proportions close to 70/15/15 even though domains
vary in size, and disjointness is asserted programmatically before anything is trained.

In [24]:
# --- greedy balanced, domain-disjoint split -------------------------------
def domain_disjoint_split(domains: pd.Series, fractions: Tuple[float, float, float],
                          seed: int) -> pd.Series:
    """Assign whole domains to splits, balancing PAGE counts against the target fractions."""
    sizes = domains.value_counts()
    order = list(sizes.index)
    rng = random.Random(seed)
    rng.shuffle(order)
    names = ["train", "val", "test"]
    targets = np.array(fractions, dtype=float) * len(domains)
    filled = np.zeros(3, dtype=float)
    assignment: Dict[str, str] = {}
    for dom in order:
        deficit = (targets - filled) / np.maximum(targets, 1.0)
        k = int(np.argmax(deficit))
        assignment[dom] = names[k]
        filled[k] += sizes[dom]
    return domains.map(assignment)

SPLIT = domain_disjoint_split(
    EV_DEDUP["domain"],
    (CONFIG["TRAIN_FRACTION"], CONFIG["VAL_FRACTION"], CONFIG["TEST_FRACTION"]),
    CONFIG["RANDOM_SEED"])
EV_DEDUP["split"] = SPLIT.values

train_domains = set(EV_DEDUP.loc[EV_DEDUP.split == "train", "domain"])
val_domains   = set(EV_DEDUP.loc[EV_DEDUP.split == "val",   "domain"])
test_domains  = set(EV_DEDUP.loc[EV_DEDUP.split == "test",  "domain"])

assert train_domains.isdisjoint(val_domains),  "train/val domain overlap"
assert train_domains.isdisjoint(test_domains), "train/test domain overlap"
assert val_domains.isdisjoint(test_domains),   "val/test domain overlap"
assert len(train_domains | val_domains | test_domains) == EV_DEDUP["domain"].nunique()

IDX = {s: np.where(EV_DEDUP["split"].values == s)[0] for s in ("train", "val", "test")}
SPLIT_STATS = pd.DataFrame([{
    "split": s, "pages": len(IDX[s]), "page_pct": round(100 * len(IDX[s]) / len(EV_DEDUP), 2),
    "domains": len({"train": train_domains, "val": val_domains, "test": test_domains}[s]),
    "pages_per_domain": round(len(IDX[s]) / max(1, len({"train": train_domains, "val": val_domains,
                                                        "test": test_domains}[s])), 2),
} for s in ("train", "val", "test")])
RESULTS["split_stats"] = SPLIT_STATS.to_dict("records")

print("DOMAIN DISJOINTNESS VERIFIED")
print(f"  train n val  : {len(train_domains & val_domains)} shared domains")
print(f"  train n test : {len(train_domains & test_domains)} shared domains")
print(f"  val   n test : {len(val_domains & test_domains)} shared domains")
print()
display(SPLIT_STATS)

# per-split positive rates: a sanity check that no split is degenerate
prev = pd.DataFrame({
    s: [round(float(((Y[IDX[s], j] == 1) & (M[IDX[s], j] == 1)).sum() /
                     max(1, M[IDX[s], j].sum())), 4) for j in range(len(TECHS))]
    for s in ("train", "val", "test")}, index=TECHS)
display(prev.sort_values("train", ascending=False).head(15))

DOMAIN DISJOINTNESS VERIFIED
  train n val  : 0 shared domains
  train n test : 0 shared domains
  val   n test : 0 shared domains



,split,pages,page_pct,domains,pages_per_domain
0,train,67541,70.0,60578,1.11
1,val,14473,15.0,12975,1.12
2,test,14473,15.0,13001,1.11


,train,val,test
WordPress,0.4156,0.4094,0.4290
Google Fonts,0.3845,0.3780,0.3773
jQuery,0.3784,0.3774,0.3790
Font Awesome,0.3155,0.3124,0.3149
PHP,0.2947,0.2901,0.2932
Google Analytics,0.2747,0.2783,0.2705
Nginx,0.2631,0.2682,0.2719
Bootstrap,0.2454,0.2450,0.2390
Apache,0.2216,0.2239,0.2161
Cloudflare,0.1814,0.1828,0.1834


## 15. Exploratory data analysis

A short look at what the corpus actually contains before modelling: page-level structural distributions, the most
common third-party resource domains, and how many technologies co-occur on a typical page.

In [25]:
# --- EDA ------------------------------------------------------------------
n_tech_per_page = (Y * M).sum(axis=1)
res_domains = Counter()
for _, _chunk in parsed_chunks(EV_DEDUP):        # evidence is parsed per chunk, not corpus-wide
    for d in _chunk["resource_domains"]:
        if isinstance(d, dict):
            res_domains.update(d.keys())

EDA = {
    "pages": int(len(EV_DEDUP)),
    "domains": int(EV_DEDUP["domain"].nunique()),
    "hosts": int(EV_DEDUP["host"].nunique()),
    "median_html_kb": round(float(EV_DEDUP["html_bytes"].median()) / 1024, 1),
    "median_elements": float(EV_DEDUP["n_elements"].median()),
    "median_dom_depth": float(EV_DEDUP["dom_depth"].median()),
    "median_scripts": float(EV_DEDUP["n_scripts"].median()),
    "median_external_domains": float(EV_DEDUP["n_external_domains"].median()),
    "mean_techs_per_page": round(float(n_tech_per_page.mean()), 2),
    "pages_with_no_trusted_tech": int((n_tech_per_page == 0).sum()),
}
RESULTS["eda"] = EDA
for k, v in EDA.items():
    print(f"{k:<28} {v}")

if CONFIG["MAKE_FIGURES"]:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    specs = [("n_elements", "DOM elements", True), ("dom_depth", "DOM depth", False),
             ("n_scripts", "script tags", False), ("n_external_domains", "external resource domains", False),
             ("html_bytes", "HTML bytes", True), ("resource_entropy", "resource-domain entropy", False)]
    for ax, (col, title, logx) in zip(axes.ravel(), specs):
        vals = pd.to_numeric(EV_DEDUP[col], errors="coerce").dropna()
        vals = vals[vals > 0] if logx else vals
        ax.hist(np.log10(vals + 1) if logx else vals, bins=50, color="#2b6cb0")
        ax.set_title(("log10 " if logx else "") + title, fontsize=10)
    savefig(fig, "03_structural_distributions")

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    top_res = pd.Series(dict(res_domains.most_common(20))).sort_values()
    axes[0].barh(top_res.index, top_res.values / len(EV_DEDUP), color="#4a5568")
    axes[0].set_xlabel("share of pages"); axes[0].set_title("Most common third-party resource domains")
    axes[1].hist(n_tech_per_page, bins=range(0, int(n_tech_per_page.max()) + 2), color="#2b6cb0")
    axes[1].set_xlabel("trusted technologies detected on a page"); axes[1].set_ylabel("pages")
    axes[1].set_title("Stack size distribution")
    savefig(fig, "04_eda_resources")
print("EDA complete")

pages                        96487
domains                      86554
hosts                        88731
median_html_kb               93.2
median_elements              520.0
median_dom_depth             16.0
median_scripts               21.0
median_external_domains      2.0
mean_techs_per_page          3.34
pages_with_no_trusted_tech   2661


13:27:06 | INFO    | figure saved: 03_structural_distributions.png
13:27:07 | INFO    | figure saved: 04_eda_resources.png


EDA complete


## 15b. Feature engineering and the three leakage regimes

This is the architectural heart of the project, so it is worth being precise about the problem it solves.

**The tautology.** A page is labelled `WordPress` because its HTML contains `/wp-content/`. If `/wp-content/` is
then handed to the model as a feature, the model has not learned to detect WordPress — it has learned to copy the
labelling function, and its 0.99 F1 measures nothing but the reliability of a regex. Any technology-detection
paper that reports scores under those conditions is reporting a circular result.

The response is not to hide the problem but to **measure it**, by building the same models over three nested
feature regimes:

| Regime | Channels | What survives | What it measures |
|---|---|---|---|
| **A — Full evidence** | text, resource/URL tokens, structure, **fingerprint rule hits** | everything observable | Ceiling. Includes the label-generating signal, so it is expected to be near-perfect and is reported as an upper bound, not a result. |
| **B — Fingerprint-reduced** | text (redacted), resource/URL tokens (filtered), structure | everything *except* the strings the labeller matched on | The honest operating point. Can the model detect the technology from the rest of the page once its signature is stripped? |
| **C — Structure-only** | numeric structure and resource-graph statistics only | no lexical content whatsoever — no URLs, no domains, no text, no headers | Pure learned structure. Any score above the prior here is genuine generalisation. |

**How redaction actually works.** Every rule in the registry that reads a lexical family (`script_src`,
`link_href`, `resource_domain`, `url_path`, `class_token`, `inline_js`, `attr`, `script_id`, `html_id`, `meta`,
`cookie`, `header`) contributes its regex to a redaction set. In regime B:

* the header and cookie channels are dropped in their entirety (they are pure label material);
* text is passed through `redact_text()`, which replaces every regex hit with a neutral token;
* resource-domain tokens matching any fingerprint domain pattern are removed;
* after TF-IDF is fitted, the vocabulary is filtered — any term matching a redaction pattern, or containing a
  known vendor domain root, is dropped from the matrix.

**This is approximate and the limitation is stated rather than glossed.** Redaction removes the signature, not
every correlate of it: a WordPress page still links to `/wp-includes/`-shaped paths through *other* URLs, and
`/2024/03/post-title/` permalink structure remains. Regime B should be read as "signature removed", not
"information-theoretically clean". Regime C is the clean condition, and that is exactly why it is included.

In [26]:
# --- redaction sets derived from the registry -----------------------------
LEXICAL_KINDS = {"script_src", "link_href", "resource_domain", "url_path", "class_token",
                 "inline_js", "attr", "script_id", "html_id", "meta", "input_name"}
HEADER_KINDS  = {"header", "cookie"}

REDACTION_PATTERNS = [cr.regex for cr in COMPILED_RULES
                      if cr.kind in LEXICAL_KINDS and cr.regex is not None]
REDACTION_UNION = re.compile("|".join(f"(?:{cr.pattern})" for cr in COMPILED_RULES
                                      if cr.kind in LEXICAL_KINDS and cr.regex is not None), re.I)

# Vendor domain roots that a fingerprint rule keys on. Removed from the resource channel in B.
FINGERPRINT_DOMAIN_PATTERNS = [cr.regex for cr in COMPILED_RULES if cr.kind == "resource_domain"]
EXTRA_VENDOR_ROOTS = {
    "shopify", "shopifycdn", "parastorage", "wixstatic", "squarespace", "website-files",
    "stripe", "paypal", "paypalobjects", "google-analytics", "googletagmanager", "hotjar",
    "hs-scripts", "hubspot", "hsforms", "cloudfront", "akamaized", "akamaihd", "jquery",
    "bootstrapcdn", "tailwindcss", "fontawesome", "recaptcha", "matomo", "piwik", "nuxt", "next",
}

def is_leaky_token(token: str) -> bool:
    """True if a vocabulary term echoes a label-generating signature."""
    t = token.lower()
    if REDACTION_UNION.search(t):
        return True
    return any(root in t for root in EXTRA_VENDOR_ROOTS)

def redact_text(text: str) -> str:
    """Neutralise fingerprint substrings inside a free-text field."""
    if not text:
        return ""
    return REDACTION_UNION.sub(" xredactedx ", text)

def is_fingerprint_domain(domain: str) -> bool:
    d = (domain or "").lower()
    return any(rx.search(d) for rx in FINGERPRINT_DOMAIN_PATTERNS) or \
           any(root in d for root in EXTRA_VENDOR_ROOTS)

print(f"redaction set: {len(REDACTION_PATTERNS)} lexical patterns, "
      f"{len(FINGERPRINT_DOMAIN_PATTERNS)} resource-domain patterns, "
      f"{len(EXTRA_VENDOR_ROOTS)} vendor roots")
print("sample redaction:", redact_text("loaded /wp-content/themes/x.css and js.stripe.com/v3/")[:90])

redaction set: 104 lexical patterns, 13 resource-domain patterns, 27 vendor roots
sample redaction: loaded  xredactedx themes/x.css and  xredactedx /v3/


In [27]:
# --- channel builders -----------------------------------------------------
STRUCT_TAGS = ["div", "span", "a", "p", "img", "li", "ul", "ol", "script", "link", "meta",
               "input", "form", "button", "table", "tr", "td", "th", "h1", "h2", "h3", "h4",
               "section", "article", "header", "footer", "nav", "aside", "svg", "path",
               "iframe", "video", "source", "style", "label", "select", "option", "br",
               "hr", "picture", "canvas", "main", "figure", "strong", "em", "noscript", "body"]

STRUCT_NUMERIC = ["html_bytes", "text_len", "n_elements", "dom_depth", "n_scripts",
                  "n_inline_scripts", "n_external_scripts", "n_stylesheets", "n_links_a",
                  "n_images", "n_iframes", "n_forms", "n_inputs", "n_metas",
                  "n_resource_domains", "n_external_domains", "resource_entropy",
                  "n_class_tokens", "n_unique_class_tokens", "avg_class_tokens",
                  "url_len", "url_path_depth", "url_has_query"]

STRUCT_COLUMNS = (STRUCT_NUMERIC
                  + [f"tagfrac_{t}" for t in STRUCT_TAGS]
                  + ["text_ratio", "script_per_kb", "elem_per_kb", "depth_per_elem",
                     "ext_dom_ratio", "img_link_ratio", "inline_script_ratio",
                     "has_dom_features"])

STRUCT_CACHE: Dict[str, Any] = {}

def struct_matrix(df: pd.DataFrame) -> np.ndarray:
    """Structural features, served from cache once `tag_counts` has been dropped.

    Sub-frames are always produced by `.iloc[...]` on the cached frame, so their index
    labels are positions into the cached matrix.
    """
    if "tag_counts" in df.columns:                       # inference / already-parsed chunk
        return _compute_struct_matrix(df)
    if "tag_counts_json" in df.columns:                  # temporal frame: parse on demand
        return _compute_struct_matrix(materialise_json(df.copy()))
    ent = STRUCT_CACHE.get("main")                       # slimmed main frame: served from cache
    if ent is None:
        raise RuntimeError("struct cache missing and tag_counts dropped - re-run from S8")
    idx = df.index.to_numpy()
    if idx.max(initial=-1) >= len(ent["matrix"]):
        raise RuntimeError("frame index does not align with the struct cache")
    return ent["matrix"][idx]

def _compute_struct_matrix(df: pd.DataFrame) -> np.ndarray:
    """Purely structural, technology-agnostic numeric features.

    Contains no vendor strings of any kind - this is what makes regime C a clean test.
    """
    n = len(df)
    out = np.zeros((n, len(STRUCT_COLUMNS)), dtype=np.float32)
    col_idx = {c: i for i, c in enumerate(STRUCT_COLUMNS)}
    for c in STRUCT_NUMERIC:
        out[:, col_idx[c]] = pd.to_numeric(df[c], errors="coerce").fillna(0).to_numpy(dtype=np.float32)
    tag_dicts = df["tag_counts"].tolist()
    for i, td in enumerate(tag_dicts):
        if not isinstance(td, dict) or not td:
            continue
        total = float(sum(td.values())) or 1.0
        for t in STRUCT_TAGS:
            v = td.get(t)
            if v:
                out[i, col_idx[f"tagfrac_{t}"]] = v / total
    kb = np.maximum(out[:, col_idx["html_bytes"]] / 1024.0, 1e-3)
    elems = np.maximum(out[:, col_idx["n_elements"]], 1.0)
    out[:, col_idx["text_ratio"]] = out[:, col_idx["text_len"]] / np.maximum(out[:, col_idx["html_bytes"]], 1.0)
    out[:, col_idx["script_per_kb"]] = out[:, col_idx["n_scripts"]] / kb
    out[:, col_idx["elem_per_kb"]] = elems / kb
    out[:, col_idx["depth_per_elem"]] = out[:, col_idx["dom_depth"]] / elems
    out[:, col_idx["ext_dom_ratio"]] = out[:, col_idx["n_external_domains"]] / \
                                       np.maximum(out[:, col_idx["n_resource_domains"]], 1.0)
    out[:, col_idx["img_link_ratio"]] = out[:, col_idx["n_images"]] / \
                                        np.maximum(out[:, col_idx["n_links_a"]], 1.0)
    out[:, col_idx["inline_script_ratio"]] = out[:, col_idx["n_inline_scripts"]] / \
                                             np.maximum(out[:, col_idx["n_scripts"]], 1.0)
    out[:, col_idx["has_dom_features"]] = (out[:, col_idx["n_elements"]] > 0).astype(np.float32)
    # log-compress heavy-tailed magnitudes
    for c in ["html_bytes", "text_len", "n_elements", "n_links_a", "n_images", "url_len"]:
        out[:, col_idx[c]] = np.log1p(np.maximum(out[:, col_idx[c]], 0))
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

DOC_COLUMNS = {("text", True): "_doc_text_red", ("text", False): "_doc_text_raw",
               ("res", True): "_doc_res_filt", ("res", False): "_doc_res_raw"}

def text_field(df: pd.DataFrame, redacted: bool) -> List[str]:
    """Cached accessor - falls through to computation for frames without the cache
    (the single-row frames built at inference time)."""
    col = DOC_COLUMNS[("text", redacted)]
    if col in df.columns:
        return df[col].tolist()
    return _compute_text_field(df, redacted)

def resource_field(df: pd.DataFrame, filtered: bool) -> List[str]:
    col = DOC_COLUMNS[("res", filtered)]
    if col in df.columns:
        return df[col].tolist()
    return _compute_resource_field(df, filtered)

def _compute_text_field(df: pd.DataFrame, redacted: bool) -> List[str]:
    """Title + description + visible text. Redacted variant used by regime B."""
    metas = df["metas"].tolist()
    titles = df["title"].tolist()
    texts = df["text_sample"].tolist()
    out = []
    for t, m, x in zip(titles, metas, texts):
        desc = ""
        if isinstance(m, dict):
            desc = " ".join(str(m.get(k, "")) for k in ("description", "og:title", "og:description",
                                                        "keywords") if m.get(k))
        s = f"{t} {desc} {x}"
        out.append(redact_text(s) if redacted else s)
    return out

_TOKEN_SPLIT = re.compile(r"[^a-z0-9]+")

def _compute_resource_field(df: pd.DataFrame, filtered: bool) -> List[str]:
    """Resource-graph document: URL path tokens, script/link filenames, resource domains.

    `filtered=True` (regime B) drops any token that echoes a labelling signature.
    """
    out = []
    for url, scripts, links, res, cls in zip(df["url"], df["scripts"], df["links"],
                                             df["resource_domains"], df["class_tokens"]):
        toks: List[str] = []
        try:
            path = urlsplit(str(url)).path.lower()
        except Exception:
            path = ""
        toks += [f"p_{t}" for t in _TOKEN_SPLIT.split(path) if 1 < len(t) < 25]
        for s in (scripts or [])[:30]:
            src = str(s.get("src", "")).lower()
            if not src:
                continue
            if filtered and REDACTION_UNION.search(src):
                continue
            tail = src.rsplit("/", 1)[-1].split("?")[0]
            toks += [f"s_{t}" for t in _TOKEN_SPLIT.split(tail) if 1 < len(t) < 25]
        for l in (links or [])[:30]:
            href = str(l.get("href", "")).lower()
            if not href:
                continue
            if filtered and REDACTION_UNION.search(href):
                continue
            tail = href.rsplit("/", 1)[-1].split("?")[0]
            toks += [f"l_{t}" for t in _TOKEN_SPLIT.split(tail) if 1 < len(t) < 25]
        for d in (res or {}):
            d = str(d).lower()
            if filtered and is_fingerprint_domain(d):
                continue
            toks.append(f"d_{d.replace('.', '_')}")
        for c in (cls or [])[:40]:
            c = str(c).lower()
            if filtered and REDACTION_UNION.search(c):
                continue
            toks.append(f"c_{_TOKEN_SPLIT.sub('_', c)[:25]}")
        out.append(" ".join(toks))
    return out

In [28]:
# --- filtered TF-IDF and the FeatureSpace ---------------------------------
class FilteredTfidf:
    """TF-IDF whose vocabulary can be pruned of label-echoing terms after fitting."""

    def __init__(self, filter_leaky: bool, **kw):
        self.filter_leaky = filter_leaky
        self.vec = TfidfVectorizer(**kw)
        self.keep_: Optional[np.ndarray] = None
        self.n_dropped_ = 0

    def fit(self, docs: Sequence[str]):
        self.vec.fit(docs)
        names = self.vec.get_feature_names_out()
        if self.filter_leaky:
            keep = np.array([not is_leaky_token(t) for t in names])
            if keep.sum() == 0:
                keep = np.ones(len(names), dtype=bool)
            self.keep_ = keep
            self.n_dropped_ = int((~keep).sum())
        else:
            self.keep_ = np.ones(len(names), dtype=bool)
        return self

    def transform(self, docs: Sequence[str]):
        X = self.vec.transform(docs)
        return X[:, self.keep_] if self.keep_ is not None else X

    def fit_transform(self, docs: Sequence[str]):
        return self.fit(docs).transform(docs)

    @property
    def feature_names(self) -> np.ndarray:
        names = self.vec.get_feature_names_out()
        return names[self.keep_] if self.keep_ is not None else names

REGIME_SPECS = {
    "A_full":      {"text": True,  "resource": True,  "struct": True,  "fingerprint": True,
                    "redact": False, "label": "A - full evidence (upper bound)"},
    "B_reduced":   {"text": True,  "resource": True,  "struct": True,  "fingerprint": False,
                    "redact": True,  "label": "B - fingerprint-reduced"},
    "C_structure": {"text": False, "resource": False, "struct": True,  "fingerprint": False,
                    "redact": True,  "label": "C - structure only"},
    # channel-isolation regimes for the hybrid comparison (S19)
    "text_only":   {"text": True,  "resource": False, "struct": False, "fingerprint": False,
                    "redact": True,  "label": "text only (redacted)"},
    "resource_only": {"text": False, "resource": True, "struct": False, "fingerprint": False,
                    "redact": True,  "label": "resource graph only (filtered)"},
    "fingerprint_only": {"text": False, "resource": False, "struct": False, "fingerprint": True,
                    "redact": False, "label": "fingerprint rule hits only"},
    "hybrid":      {"text": True,  "resource": True,  "struct": True,  "fingerprint": False,
                    "redact": True,  "label": "hybrid (text + resource + structure)"},
}

class FeatureSpace:
    """Fits every enabled channel on TRAIN ONLY and transforms any split identically."""

    def __init__(self, regime: str, cfg: Dict[str, Any]):
        if regime not in REGIME_SPECS:
            raise ValueError(f"unknown regime {regime}")
        self.regime = regime
        self.spec = REGIME_SPECS[regime]
        self.cfg = cfg
        self.text_vec: Optional[FilteredTfidf] = None
        self.res_vec: Optional[FilteredTfidf] = None
        self.scaler: Optional[StandardScaler] = None
        self.rule_cols_: Optional[np.ndarray] = None
        self.blocks_: List[Tuple[str, int]] = []

    def fit(self, df: pd.DataFrame, hits: Optional[sp.csr_matrix] = None) -> "FeatureSpace":
        s, cfg = self.spec, self.cfg
        if s["text"]:
            self.text_vec = FilteredTfidf(filter_leaky=s["redact"], max_features=cfg["TFIDF_MAX_FEATURES"],
                                          ngram_range=(1, 2), min_df=3, max_df=0.9,
                                          sublinear_tf=True, strip_accents="unicode",
                                          lowercase=True, dtype=np.float32)
            self.text_vec.fit(text_field(df, s["redact"]))
        if s["resource"]:
            self.res_vec = FilteredTfidf(filter_leaky=s["redact"], max_features=cfg["TFIDF_MAX_FEATURES"],
                                         ngram_range=(1, 1), min_df=3, sublinear_tf=True,
                                         token_pattern=r"\S+", lowercase=True, dtype=np.float32)
            self.res_vec.fit(resource_field(df, s["redact"]))
        if s["struct"]:
            self.scaler = StandardScaler().fit(struct_matrix(df))
        if s["fingerprint"] and hits is not None:
            active = np.asarray(hits.sum(axis=0)).ravel() > 0
            self.rule_cols_ = np.where(active)[0]
        return self

    def transform(self, df: pd.DataFrame, hits: Optional[sp.csr_matrix] = None) -> sp.csr_matrix:
        s = self.spec
        parts: List[sp.spmatrix] = []
        blocks: List[Tuple[str, int]] = []
        if s["text"] and self.text_vec is not None:
            X = self.text_vec.transform(text_field(df, s["redact"]))
            parts.append(sp.csr_matrix(X)); blocks.append(("text", X.shape[1]))
        if s["resource"] and self.res_vec is not None:
            X = self.res_vec.transform(resource_field(df, s["redact"]))
            parts.append(sp.csr_matrix(X)); blocks.append(("resource", X.shape[1]))
        if s["struct"] and self.scaler is not None:
            X = self.scaler.transform(struct_matrix(df)).astype(np.float32)
            parts.append(sp.csr_matrix(np.nan_to_num(X))); blocks.append(("struct", X.shape[1]))
        if s["fingerprint"] and hits is not None and self.rule_cols_ is not None:
            X = hits[:, self.rule_cols_].astype(np.float32)
            parts.append(sp.csr_matrix(X)); blocks.append(("fingerprint", X.shape[1]))
        if not parts:
            raise RuntimeError(f"regime {self.regime} produced no channels")
        self.blocks_ = blocks
        return sp.hstack(parts, format="csr")

    def describe(self) -> str:
        return f"{self.regime}: " + ", ".join(f"{n}={d}" for n, d in self.blocks_)

def build_regime(regime: str, df: pd.DataFrame, hits: sp.csr_matrix,
                 idx: Dict[str, np.ndarray]) -> Tuple[FeatureSpace, Dict[str, sp.csr_matrix]]:
    """Fit on train rows only, then transform all three splits."""
    fs = FeatureSpace(regime, CONFIG).fit(df.iloc[idx["train"]], hits[idx["train"]])
    mats = {s: fs.transform(df.iloc[idx[s]], hits[idx[s]]) for s in ("train", "val", "test")}
    LOG.info("%s -> %s | train %s", regime, fs.describe(), mats["train"].shape)
    return fs, mats

### 15c. Document precomputation and frame slimming

Two memory/time optimisations that the first full run made unavoidable.

**Documents are built once.** `text_field` and `resource_field` are Python loops with regex redaction over every
row; each pass costs ~90s at 100k pages and the notebook needs about ten of them across the regimes. All four
variants (text raw/redacted, resource raw/filtered) are materialised once into cached columns and reused.

**The evidence frame is then slimmed.** The materialised JSON columns — `scripts`, `metas`, `links`,
`resource_domains`, `class_tokens` and friends — are millions of small Python dicts and lists, and they dominate
resident memory at ~4-5 GB. They are needed for exactly two things: rule matching (finished in S10) and document
construction (finished above). Once both are done they are dead weight, and holding them is what pushes a 13 GB
Colab runtime into an OOM kill partway through the ablations.

If you need to re-run an earlier cell after this point, re-run from the evidence-loading cell (S8) — the dropped
columns are not recoverable in place.

In [29]:
# --- precompute documents, then drop what is no longer needed --------------
def precompute_documents(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray]:
    """Build all four document variants and the struct matrix in ONE chunked pass.

    One pass instead of five: the JSON evidence is parsed once per chunk and every consumer
    reads from that same parse, which is both faster and the reason peak memory stays flat.
    """
    t0 = time.time()
    docs: Dict[str, List[str]] = {col: [] for col in DOC_COLUMNS.values()}
    mats: List[np.ndarray] = []
    done = 0
    for _, chunk in parsed_chunks(df):
        for (kind, flag), col in DOC_COLUMNS.items():
            docs[col].extend(_compute_text_field(chunk, flag) if kind == "text"
                             else _compute_resource_field(chunk, flag))
        mats.append(_compute_struct_matrix(chunk))
        done += len(chunk)
        if done % 20000 < len(chunk):
            LOG.info("precompute: %d/%d rows (%.0fs)", done, len(df), time.time() - t0)
    for col, vals in docs.items():
        df[col] = pd.array(vals, dtype="string")
    del docs
    struct = np.vstack(mats)
    del mats
    gc.collect(); trim_heap()
    LOG.info("precompute complete: 4 document variants + struct %s (%.0fs)",
             struct.shape, time.time() - t0)
    return df, struct

# Columns needed only by rule matching (S10), document construction and struct features -
# all of which are complete by the time this runs.
HEAVY_OBJECT_COLUMNS = ["headers", "cookie_names", "metas", "scripts", "script_ids", "links",
                        "resource_domains", "class_tokens", "attr_names", "html_ids",
                        "input_names", "tag_counts"]
HEAVY_JSON_COLUMNS = [c for c in EVIDENCE_COLUMNS if c.endswith("_json")] + ["inline_js",
                                                                             "text_sample"]

def slim_frame(df: pd.DataFrame, keep_extra: Sequence[str] = ()) -> pd.DataFrame:
    """Return a frame that genuinely releases the dropped objects.

    `df.drop(columns=...)` is NOT sufficient: pandas stores every object column in one
    consolidated 2-D block, and drop returns a VIEW of that block, so the discarded dicts
    stay reachable and nothing is actually freed. Re-selecting the survivors and forcing a
    copy builds a fresh block holding only the kept references; the caller then rebinds and
    the original frame - with its millions of dicts - becomes collectable.
    """
    drop = [c for c in HEAVY_OBJECT_COLUMNS + HEAVY_JSON_COLUMNS
            if c in df.columns and c not in set(keep_extra)]
    keep = [c for c in df.columns if c not in drop]
    out = df[keep].copy()
    LOG.info("released %d evidence columns (%d retained)", len(drop), len(keep))
    return out

mem_report("pre-precompute")
EV_DEDUP, _struct_all = precompute_documents(EV_DEDUP)
STRUCT_CACHE["main"] = {"matrix": _struct_all}
del _struct_all
mem_report("post-documents")

_old_frame = EV_DEDUP
EV_DEDUP = slim_frame(EV_DEDUP)
del _old_frame                     # last reference to the raw-JSON block
gc.collect()
trim_heap()
mem_report("post-slim")

if HW.get("ram_gb") and HW["ram_gb"] < 20:
    print(f"NOTE: {HW['ram_gb']} GB RAM. Feature matrices are evicted as soon as each regime "
          f"is finished with; if you still hit an OOM, lower CONFIG['TFIDF_MAX_FEATURES'] "
          f"to 30000 and re-run from this cell.")

13:27:07 | INFO    | [mem] pre-precompute rss=3.27 GB
13:28:38 | INFO    | precompute: 20000/96487 rows (90s)
13:30:15 | INFO    | precompute: 40000/96487 rows (187s)
13:32:02 | INFO    | precompute: 60000/96487 rows (295s)
13:33:47 | INFO    | precompute: 80000/96487 rows (400s)
13:35:04 | INFO    | precompute complete: 4 document variants + struct (96487, 78) (477s)
13:35:04 | INFO    | [mem] post-documents rss=4.04 GB
13:35:04 | INFO    | released 14 evidence columns (38 retained)
13:35:05 | INFO    | [mem] post-slim rss=2.89 GB


NOTE: 13.6 GB RAM. Feature matrices are evicted as soon as each regime is finished with; if you still hit an OOM, lower CONFIG['TFIDF_MAX_FEATURES'] to 30000 and re-run from this cell.


In [30]:
# --- multi-label metrics (mask-aware) -------------------------------------
def masked_metrics(y: np.ndarray, prob: np.ndarray, mask: np.ndarray,
                   thresholds: np.ndarray, labels: Sequence[str]) -> Dict[str, Any]:
    """Every metric is computed over TRUSTED cells only (mask == 1)."""
    y = np.asarray(y); prob = np.asarray(prob); mask = np.asarray(mask).astype(bool)
    pred = (prob >= thresholds[None, :]).astype(np.int8)

    yt, pt = y[mask], pred[mask]
    tp = float(((yt == 1) & (pt == 1)).sum())
    fp = float(((yt == 0) & (pt == 1)).sum())
    fn = float(((yt == 1) & (pt == 0)).sum())
    micro_p = tp / max(tp + fp, 1e-9)
    micro_r = tp / max(tp + fn, 1e-9)
    micro_f1 = 2 * micro_p * micro_r / max(micro_p + micro_r, 1e-9)

    per: List[Dict[str, Any]] = []
    for j, lab in enumerate(labels):
        m = mask[:, j]
        yj, pj, sj = y[m, j], pred[m, j], prob[m, j]
        sup = int(yj.sum())
        if sup == 0 or m.sum() == 0:
            per.append({"technology": lab, "precision": np.nan, "recall": np.nan, "f1": np.nan,
                        "pr_auc": np.nan, "support": sup, "trusted": int(m.sum()),
                        "threshold": float(thresholds[j]), "predicted_positive": int(pj.sum())})
            continue
        p, r, f, _ = precision_recall_fscore_support(yj, pj, average="binary", zero_division=0)
        try:
            ap = float(average_precision_score(yj, sj)) if 0 < sup < len(yj) else np.nan
        except Exception:
            ap = np.nan
        per.append({"technology": lab, "precision": float(p), "recall": float(r), "f1": float(f),
                    "pr_auc": ap, "support": sup, "trusted": int(m.sum()),
                    "threshold": float(thresholds[j]), "predicted_positive": int(pj.sum())})
    per_df = pd.DataFrame(per)
    valid = per_df.dropna(subset=["f1"])
    w = valid["support"].to_numpy(dtype=float)

    full_rows = mask.all(axis=1)
    subset_acc = float((pred[full_rows] == y[full_rows]).all(axis=1).mean()) if full_rows.any() else np.nan
    try:
        micro_ap = float(average_precision_score(yt.ravel(), prob[mask].ravel()))
    except Exception:
        micro_ap = np.nan

    return {
        "micro_precision": micro_p, "micro_recall": micro_r, "micro_f1": micro_f1,
        "macro_precision": float(valid["precision"].mean()) if len(valid) else np.nan,
        "macro_recall": float(valid["recall"].mean()) if len(valid) else np.nan,
        "macro_f1": float(valid["f1"].mean()) if len(valid) else np.nan,
        "weighted_f1": float(np.average(valid["f1"], weights=w)) if w.sum() else np.nan,
        "macro_pr_auc": float(valid["pr_auc"].mean(skipna=True)) if len(valid) else np.nan,
        "micro_pr_auc": micro_ap,
        "hamming_loss": float((yt != pt).mean()),
        "subset_accuracy": subset_acc,
        "n_labels_scored": int(len(valid)),
        "trusted_cell_frac": float(mask.mean()),
        "per_technology": per_df,
    }

def summarise(name: str, m: Dict[str, Any]) -> Dict[str, Any]:
    keys = ["micro_f1", "macro_f1", "weighted_f1", "micro_precision", "micro_recall",
            "macro_precision", "macro_recall", "macro_pr_auc", "micro_pr_auc",
            "hamming_loss", "subset_accuracy"]
    row = {"model": name}
    row.update({k: (round(float(m[k]), 4) if m.get(k) is not None and
                    not (isinstance(m.get(k), float) and math.isnan(m[k])) else np.nan) for k in keys})
    return row

def tune_thresholds(y: np.ndarray, prob: np.ndarray, mask: np.ndarray,
                    grid: np.ndarray = np.linspace(0.05, 0.95, 91),
                    min_pos: int = 5) -> np.ndarray:
    """Per-label F1-maximising thresholds, fitted on VALIDATION only."""
    k = y.shape[1]
    out = np.full(k, 0.5, dtype=np.float32)
    for j in range(k):
        m = mask[:, j].astype(bool)
        yj, sj = y[m, j], prob[m, j]
        if yj.sum() < min_pos:
            continue
        best_f, best_t = -1.0, 0.5
        for t in grid:
            pj = (sj >= t)
            tp = float((yj[pj] == 1).sum()); fp = float((yj[pj] == 0).sum())
            fn = float((yj[~pj] == 1).sum())
            f1 = 2 * tp / max(2 * tp + fp + fn, 1e-9)
            if f1 > best_f:
                best_f, best_t = f1, float(t)
        out[j] = best_t
    return out

DEFAULT_THRESHOLDS = np.full(len(TECHS), 0.5, dtype=np.float32)

## 16. Baseline 1 — fingerprint rules alone

The rule baseline represents the incumbent approach: this is what Wappalyzer-style detectors do. It is also the
labelling function, which makes it a subtle thing to evaluate honestly.

Two variants are reported:

* **`rules_full`** — the complete registry. Against labels this same registry produced, it is **tautological by
  construction** and will score near 1.0. It is included precisely so the number is visible and cannot be mistaken
  for a research finding. It is the reference point for "how much of regime A is circular".
* **`rules_reduced`** — the registry with every `sufficient` rule removed. This is a real baseline: it asks how
  well the *remaining* corroborating evidence performs when the decisive signature is gone, which is the same
  handicap regime B imposes on the ML models. Comparing `rules_reduced` against ML-in-regime-B is the
  apples-to-apples rule-vs-learning comparison (RQ2).

In [31]:
# --- rule baselines -------------------------------------------------------
def rule_predictions(hits: sp.csr_matrix, techs: Sequence[str],
                     drop_sufficient: bool = False,
                     min_confidence: str = "high") -> np.ndarray:
    """Re-derive predictions from rule hits, optionally with the decisive rules removed."""
    hits = hits.tocsr()
    n = hits.shape[0]
    out = np.zeros((n, len(techs)), dtype=np.float32)
    need = CONF_ORDER[min_confidence]
    for j, tech in enumerate(techs):
        rules = [r for r in RULES_BY_TECH[tech] if not (drop_sufficient and r.sufficient)]
        if not rules:
            continue
        cols = [RULE_INDEX[r.rule_id] for r in rules]
        sub = hits[:, cols].toarray().astype(bool)
        weights = np.array([STRENGTH_WEIGHT[r.strength] for r in rules], dtype=np.float32)
        strong = np.array([r.strength == "strong" for r in rules])
        suff = np.array([r.sufficient and not drop_sufficient for r in rules])
        kinds = np.array([r.kind for r in rules])
        score = sub @ weights
        n_strong = sub[:, strong].sum(axis=1) if strong.any() else np.zeros(n)
        has_suff = sub[:, suff].any(axis=1) if suff.any() else np.zeros(n, dtype=bool)
        families = np.zeros(n, dtype=np.int16)
        for kind in np.unique(kinds):
            families += sub[:, kinds == kind].any(axis=1).astype(np.int16)
        level = np.where(has_suff | ((score >= 1.4) & (families >= 2)) | (n_strong >= 2),
                         CONF_ORDER["high"],
                         np.where(score >= 1.0, CONF_ORDER["medium"],
                                  np.where(score > 0, CONF_ORDER["low"], CONF_ORDER["abstain"])))
        out[:, j] = (level >= need).astype(np.float32)
    return out

def frequency_baseline(y_train: np.ndarray, m_train: np.ndarray, n_rows: int) -> np.ndarray:
    """Predict each label's training prevalence for every row (a proper probabilistic prior)."""
    prior = np.array([((y_train[:, j] == 1) & (m_train[:, j] == 1)).sum() /
                      max(1, m_train[:, j].sum()) for j in range(y_train.shape[1])], dtype=np.float32)
    return np.repeat(prior[None, :], n_rows, axis=0)

Y_TR, Y_VA, Y_TE = Y[IDX["train"]], Y[IDX["val"]], Y[IDX["test"]]
M_TR, M_VA, M_TE = M[IDX["train"]], M[IDX["val"]], M[IDX["test"]]
HITS_TR, HITS_VA, HITS_TE = RULE_HITS[IDX["train"]], RULE_HITS[IDX["val"]], RULE_HITS[IDX["test"]]

BASELINES: Dict[str, Dict[str, Any]] = {}

p_rules_full = rule_predictions(HITS_VA, TECHS, drop_sufficient=False)
BASELINES["rules_full"] = masked_metrics(Y_VA, p_rules_full, M_VA, DEFAULT_THRESHOLDS, TECHS)

p_rules_red = rule_predictions(HITS_VA, TECHS, drop_sufficient=True)
BASELINES["rules_reduced"] = masked_metrics(Y_VA, p_rules_red, M_VA, DEFAULT_THRESHOLDS, TECHS)

# Two prior baselines, because the naive one is degenerate here. Every technology has
# prevalence < 0.5, so a constant-prior prediction thresholded at 0.5 predicts nothing and
# scores exactly 0.0 micro-F1 - a floor that no model can fail to clear, and therefore a
# useless reference. The tuned variant gets the same per-label threshold treatment as every
# real model, which turns it into the honest "predict by base rate alone" floor.
p_freq = frequency_baseline(Y_TR, M_TR, len(IDX["val"]))
BASELINES["majority_class"] = masked_metrics(Y_VA, p_freq, M_VA, DEFAULT_THRESHOLDS, TECHS)
THR_PRIOR = tune_thresholds(Y_VA, p_freq, M_VA)
BASELINES["frequency_prior"] = masked_metrics(Y_VA, p_freq, M_VA, THR_PRIOR, TECHS)

base_tbl = pd.DataFrame([summarise(k, v) for k, v in BASELINES.items()])
RESULTS["baselines_validation"] = base_tbl.to_dict("records")
print("Validation-set baselines (rules_full is CIRCULAR by construction - reference only)")
display(base_tbl)

Validation-set baselines (rules_full is CIRCULAR by construction - reference only)


,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,macro_precision,macro_recall,macro_pr_auc,micro_pr_auc,hamming_loss,subset_accuracy
0,rules_full,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,1.0000
1,rules_reduced,0.2219,0.1412,0.1754,1.0000,0.1248,0.3077,0.1034,0.1820,0.2019,0.0771,0.0543
2,majority_class,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0919,0.3304,0.0881,0.0417
3,frequency_prior,0.3674,0.1318,0.3975,0.2299,0.9146,0.0845,0.3590,0.0919,0.3304,0.2775,0.0000


## 17. Classical ML

`MaskedOvR` trains one binary classifier per technology, and — this is the point — each classifier sees only the
rows where *its own* label is trusted. A page whose WordPress evidence was ambiguous is dropped from the WordPress
model while still training the Stripe model. Standard `OneVsRestClassifier` cannot express that, which is why the
wrapper exists.

Degenerate labels (a split where one class vanished) fall back to a constant prior rather than raising, so a rare
technology cannot crash the run.

* **Baseline 3** — TF-IDF text + one-vs-rest logistic regression, `class_weight="balanced"`.
* **Baseline 4** — structural features + histogram gradient boosting, the strongest non-lexical classical option.

In [32]:
# --- masked one-vs-rest wrapper -------------------------------------------
class MaskedOvR:
    """One binary estimator per label, trained only on that label's trusted rows."""

    def __init__(self, factory: Callable[[], Any], labels: Sequence[str], name: str = "ovr",
                 dense: bool = False, max_rows: Optional[int] = None, seed: int = 42):
        self.factory = factory; self.labels = list(labels); self.name = name
        self.dense = dense; self.max_rows = max_rows; self.seed = seed
        self.models_: List[Any] = []
        self.priors_: List[float] = []

    def fit(self, X, y: np.ndarray, m: np.ndarray) -> "MaskedOvR":
        rng = np.random.default_rng(self.seed)
        self.models_, self.priors_ = [], []
        t_start = time.time()
        n_rows_total = X.shape[0]
        # Densify once rather than 39 times; guarded so this can never fire on a wide matrix.
        X_dense = np.asarray(X.todense()) if (self.dense and sp.issparse(X)
                                              and X.shape[1] <= 2000) else None
        for j, lab in enumerate(self.labels):
            rows = np.where(m[:, j] == 1)[0]
            yj = y[rows, j]
            prior = float(yj.mean()) if len(yj) else 0.0
            self.priors_.append(prior)
            if len(rows) < 20 or yj.sum() < 5 or yj.sum() == len(yj):
                self.models_.append(None)                      # constant-prior fallback
                continue
            if self.max_rows and len(rows) > self.max_rows:
                rows = rng.choice(rows, self.max_rows, replace=False)
                yj = y[rows, j]
            # Slicing copies the whole matrix; at 120k features that is ~300 MB per label
            # and it dominated the hybrid fit. Most labels have a complete mask, so skip it.
            base = X_dense if X_dense is not None else X
            Xi = base if len(rows) == n_rows_total else base[rows]
            if self.dense and sp.issparse(Xi):
                Xi = np.asarray(Xi.todense())
            try:
                mdl = self.factory()
                mdl.fit(Xi, yj)
                self.models_.append(mdl)
            except Exception as exc:
                LOG.warning("[%s] label %s failed to fit (%s) - using prior", self.name, lab, exc)
                self.models_.append(None)
            finally:
                del Xi                       # release the per-label row slice immediately
            if (j + 1) % 5 == 0 or j == len(self.labels) - 1:
                LOG.info("[%s] fitted %d/%d labels (%.0fs elapsed)",
                         self.name, j + 1, len(self.labels), time.time() - t_start)
        return self

    def predict_proba(self, X) -> np.ndarray:
        n = X.shape[0]
        out = np.zeros((n, len(self.labels)), dtype=np.float32)
        for j, mdl in enumerate(self.models_):
            if mdl is None:
                out[:, j] = self.priors_[j]; continue
            Xi = np.asarray(X.todense()) if (self.dense and sp.issparse(X)) else X
            try:
                p = mdl.predict_proba(Xi)
                out[:, j] = p[:, 1] if p.ndim == 2 and p.shape[1] == 2 else p.ravel()
            except Exception:
                out[:, j] = self.priors_[j]
        return out

def logreg_factory():
    """L2 logistic regression tuned for WIDE SPARSE input.

    `liblinear` solves the primal by coordinate descent, so its cost scales with
    n_features - at ~120k concatenated features that becomes the dominant cost in the whole
    notebook (hours, not minutes). `lbfgs` is gradient-based: each iteration is two sparse
    matvecs costing O(nnz), which is an order of magnitude cheaper at this width.
    """
    return LogisticRegression(max_iter=200, C=2.0, solver="lbfgs", tol=1e-3,
                              class_weight="balanced")

def hgb_factory():
    return HistGradientBoostingClassifier(max_iter=120, learning_rate=0.12, max_depth=None,
                                          early_stopping=True, validation_fraction=0.12,
                                          random_state=CONFIG["RANDOM_SEED"])

In [33]:
# --- fit the classical baselines on regime B (the honest condition) -------
FEATURE_CACHE: Dict[str, Tuple[FeatureSpace, Dict[str, sp.csr_matrix]]] = {}

# The "hybrid" channel combination (text + resource + structure, redacted) is exactly the
# B_reduced regime - same channels, same redaction flag. Building and fitting it twice cost
# ~30 minutes of pure duplicated work, so it is aliased to a single cache entry.
REGIME_ALIASES = {"hybrid": "B_reduced"}

def get_regime(regime: str):
    key = REGIME_ALIASES.get(regime, regime)
    if key not in FEATURE_CACHE:
        FEATURE_CACHE[key] = build_regime(key, EV_DEDUP, RULE_HITS, IDX)
    return FEATURE_CACHE[key]

def evict_regimes(*regimes: str) -> None:
    """Drop cached feature matrices once a regime is finished with.

    Each cached regime holds three sparse matrices (train/val/test) that run to ~1 GB at
    100k pages. Holding all of them at once is what pushes a 13 GB Colab runtime into
    allocator thrashing partway through the ablations.
    """
    for r in regimes:
        key = REGIME_ALIASES.get(r, r)
        if FEATURE_CACHE.pop(key, None) is not None:
            LOG.info("evicted cached features for regime %s", key)
    gc.collect()
    trim_heap()
    mem_report("post-evict")

t0 = time.time()
fs_text, X_text = get_regime("text_only")
clf_tfidf = MaskedOvR(logreg_factory, TECHS, "tfidf_logreg",
                      max_rows=120_000, seed=CONFIG["RANDOM_SEED"])
clf_tfidf.fit(X_text["train"], Y_TR, M_TR)
p_tfidf_va = clf_tfidf.predict_proba(X_text["val"])
BASELINES["tfidf_logreg"] = masked_metrics(Y_VA, p_tfidf_va, M_VA, DEFAULT_THRESHOLDS, TECHS)
LOG.info("TF-IDF + OvR logistic regression fitted in %.1fs", time.time() - t0)

t0 = time.time()
fs_struct, X_struct = get_regime("C_structure")
clf_struct = MaskedOvR(hgb_factory, TECHS, "struct_hgb", dense=True,
                       max_rows=60_000, seed=CONFIG["RANDOM_SEED"])
clf_struct.fit(X_struct["train"], Y_TR, M_TR)
p_struct_va = clf_struct.predict_proba(X_struct["val"])
BASELINES["struct_gbdt"] = masked_metrics(Y_VA, p_struct_va, M_VA, DEFAULT_THRESHOLDS, TECHS)
LOG.info("structural GBDT fitted in %.1fs", time.time() - t0)

base_tbl = pd.DataFrame([summarise(k, v) for k, v in BASELINES.items()])
RESULTS["baselines_validation"] = base_tbl.to_dict("records")
display(base_tbl)
mem_report("post-classical")

13:37:01 | INFO    | text_only -> text_only: text=59970 | train (67541, 59970)
13:37:07 | INFO    | [tfidf_logreg] fitted 5/39 labels (6s elapsed)
13:37:15 | INFO    | [tfidf_logreg] fitted 10/39 labels (13s elapsed)
13:37:22 | INFO    | [tfidf_logreg] fitted 15/39 labels (21s elapsed)
13:37:31 | INFO    | [tfidf_logreg] fitted 20/39 labels (30s elapsed)
13:37:36 | INFO    | [tfidf_logreg] fitted 25/39 labels (35s elapsed)
13:37:40 | INFO    | [tfidf_logreg] fitted 30/39 labels (39s elapsed)
13:37:50 | INFO    | [tfidf_logreg] fitted 35/39 labels (49s elapsed)
13:37:54 | INFO    | [tfidf_logreg] fitted 39/39 labels (53s elapsed)
13:37:55 | INFO    | TF-IDF + OvR logistic regression fitted in 167.3s
13:37:55 | INFO    | C_structure -> C_structure: struct=78 | train (67541, 78)
13:38:13 | INFO    | [struct_hgb] fitted 5/39 labels (18s elapsed)
13:38:32 | INFO    | [struct_hgb] fitted 10/39 labels (37s elapsed)
13:38:51 | INFO    | [struct_hgb] fitted 15/39 labels (55s elapsed)
13:39:07 |

,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,macro_precision,macro_recall,macro_pr_auc,micro_pr_auc,hamming_loss,subset_accuracy
0,rules_full,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,1.0000
1,rules_reduced,0.2219,0.1412,0.1754,1.0000,0.1248,0.3077,0.1034,0.1820,0.2019,0.0771,0.0543
2,majority_class,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0919,0.3304,0.0881,0.0417
3,frequency_prior,0.3674,0.1318,0.3975,0.2299,0.9146,0.0845,0.3590,0.0919,0.3304,0.2775,0.0000
4,tfidf_logreg,0.4766,0.2987,0.5059,0.3903,0.6119,0.2420,0.4459,0.3079,0.4626,0.1184,0.0272
5,struct_gbdt,0.6038,0.4082,0.5500,0.7724,0.4956,0.5774,0.3626,0.4645,0.7230,0.0573,0.1363


13:40:21 | INFO    | [mem] post-classical rss=3.96 GB


## 18. Advanced model — hybrid semantic + structural network

The advanced model is a **hybrid multi-label network**, chosen for feasibility rather than for looking impressive:

```text
   redacted page text  --> sentence embeddings (MiniLM, 384-d)   \
   resource-graph TF-IDF --> TruncatedSVD (192-d)                  >-- concat --> MLP --> k sigmoids
   structural features (~75-d, standardised)                      /
```

**Why this and not a document transformer over raw HTML?** Fine-tuning a transformer on 100k HTML documents is a
multi-GPU-hour job, and HTML blows past a 512-token window before the `<body>` even opens — the tokens that matter
(script URLs, class attributes) would be truncated away. Frozen sentence embeddings over the *text* channel, fused
with explicit structural and resource features, captures the same information at a fraction of the cost and keeps
the whole notebook inside a free Colab session.

**Adaptive degradation, in order of preference:**

1. GPU available -> MiniLM embeddings for the text channel.
2. CPU with fewer than `EMBED_MAX_ROWS_CPU` rows -> MiniLM on CPU.
3. Otherwise -> TF-IDF + SVD as the semantic channel, which is a genuine latent-semantic representation rather
   than a stub, and costs seconds instead of an hour.

The head is trained with **masked BCE**: the loss for cell (i, j) is multiplied by `mask[i, j]`, so untrusted
labels contribute exactly zero gradient. Per-label `pos_weight` (capped at 50) counteracts the extreme class
imbalance, and early stopping watches validation macro-PR-AUC — not loss, and never anything from the test set.

In [34]:
# --- semantic embedding channel with graceful degradation -----------------
class SemanticEncoder:
    """MiniLM if it is affordable here, otherwise TF-IDF + SVD. Same interface either way."""

    def __init__(self, cfg: Dict[str, Any], n_train_rows: int):
        self.cfg = cfg
        self.mode = "none"
        self.model = None
        self.tfidf: Optional[FilteredTfidf] = None
        self.svd: Optional[TruncatedSVD] = None
        want_st = False
        try:
            from sentence_transformers import SentenceTransformer  # noqa: F401
            want_st = True
        except Exception:
            want_st = False
        affordable = (DEVICE == "cuda") or (n_train_rows <= int(cfg["EMBED_MAX_ROWS_CPU"]))
        if want_st and affordable:
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer(cfg["EMBEDDING_MODEL"], device=DEVICE)
                self.mode = "sentence_transformer"
            except Exception as exc:
                LOG.warning("sentence-transformer load failed (%s) - falling back to SVD", exc)
        if self.mode == "none":
            self.mode = "tfidf_svd"
        LOG.info("semantic encoder: %s (device=%s, train_rows=%d)", self.mode, DEVICE, n_train_rows)

    def fit(self, docs: Sequence[str]) -> "SemanticEncoder":
        if self.mode == "tfidf_svd":
            self.tfidf = FilteredTfidf(filter_leaky=True, max_features=40000, ngram_range=(1, 2),
                                       min_df=3, sublinear_tf=True, dtype=np.float32)
            Xt = self.tfidf.fit_transform(docs)
            comps = int(min(self.cfg["SVD_COMPONENTS"], max(2, Xt.shape[1] - 1), max(2, Xt.shape[0] - 1)))
            self.svd = TruncatedSVD(n_components=comps, random_state=self.cfg["RANDOM_SEED"])
            self.svd.fit(Xt)
        return self

    def encode(self, docs: Sequence[str], batch: int = 256) -> np.ndarray:
        if self.mode == "sentence_transformer":
            return np.asarray(self.model.encode(list(docs), batch_size=batch,
                                                show_progress_bar=True, convert_to_numpy=True,
                                                normalize_embeddings=True), dtype=np.float32)
        Xt = self.tfidf.transform(docs)
        return np.asarray(self.svd.transform(Xt), dtype=np.float32)

def build_hybrid_features(df: pd.DataFrame, idx: Dict[str, np.ndarray]) -> Tuple[Dict[str, np.ndarray], Dict[str, Any]]:
    """Semantic(text) + SVD(resource graph) + structural, all fitted on train only."""
    docs_all = text_field(df, redacted=True)
    enc = SemanticEncoder(CONFIG, len(idx["train"]))
    enc.fit([docs_all[i] for i in idx["train"]])

    res_all = resource_field(df, filtered=True)
    res_vec = FilteredTfidf(filter_leaky=True, max_features=CONFIG["TFIDF_MAX_FEATURES"],
                            ngram_range=(1, 1), min_df=3, sublinear_tf=True,
                            token_pattern=r"\S+", dtype=np.float32)
    res_tr = res_vec.fit_transform([res_all[i] for i in idx["train"]])
    comps = int(min(CONFIG["SVD_COMPONENTS"], max(2, res_tr.shape[1] - 1), max(2, res_tr.shape[0] - 1)))
    res_svd = TruncatedSVD(n_components=comps, random_state=CONFIG["RANDOM_SEED"]).fit(res_tr)

    struct_all = struct_matrix(df)
    scaler = StandardScaler().fit(struct_all[idx["train"]])

    out: Dict[str, np.ndarray] = {}
    for s in ("train", "val", "test"):
        rows = idx[s]
        emb = enc.encode([docs_all[i] for i in rows])
        rsv = res_svd.transform(res_vec.transform([res_all[i] for i in rows])).astype(np.float32)
        stc = np.nan_to_num(scaler.transform(struct_all[rows]).astype(np.float32))
        out[s] = np.hstack([emb, rsv, stc]).astype(np.float32)
    bundle = {"encoder": enc, "res_vec": res_vec, "res_svd": res_svd, "scaler": scaler,
              "dims": {"semantic": out["train"].shape[1] - comps - struct_all.shape[1],
                       "resource_svd": comps, "struct": struct_all.shape[1]}}
    LOG.info("hybrid features: %s -> %s", bundle["dims"], out["train"].shape)
    return out, bundle

t0 = time.time()
XH, HYBRID_BUNDLE = build_hybrid_features(EV_DEDUP, IDX)
RUN["hybrid_feature_seconds"] = round(time.time() - t0, 1)
RUN["semantic_encoder_mode"] = HYBRID_BUNDLE["encoder"].mode
mem_report("post-embedding")

13:40:22 | INFO    | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
13:40:22 | INFO    | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
13:40:22 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
13:40:22 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

13:40:22 | INFO    | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
13:40:22 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
13:40:22 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
13:40:22 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

13:40:22 | INFO    | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
13:40:22 | INFO    | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
13:40:22 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
13:40:22 | INFO    | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
13:40:22 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/README.md "HTTP/1.1 200 OK"
13:40:22 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b

...

13:40:26 | INFO    | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
13:40:27 | INFO    | semantic encoder: sentence_transformer (device=cuda, train_rows=67541)


Batches:   0%|          | 0/264 [00:00<?, ?it/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

13:46:53 | INFO    | hybrid features: {'semantic': 384, 'resource_svd': 192, 'struct': 78} -> (67541, 654)
13:46:53 | INFO    | [mem] post-embedding rss=4.60 GB


In [35]:
# --- masked-BCE multi-label head ------------------------------------------
class TorchMultiLabelHead:
    """MLP with masked BCE loss and macro-PR-AUC early stopping."""

    def __init__(self, in_dim: int, n_labels: int, cfg: Dict[str, Any]):
        self.in_dim, self.n_labels, self.cfg = in_dim, n_labels, cfg
        self.model = None
        self.history: List[Dict[str, float]] = []
        self.best_state = None

    def _build(self):
        h1, h2 = self.cfg["MLP_HIDDEN"]
        return nn.Sequential(
            nn.Linear(self.in_dim, h1), nn.BatchNorm1d(h1), nn.GELU(), nn.Dropout(0.30),
            nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(h2, self.n_labels),
        )

    def fit(self, Xtr, ytr, mtr, Xva, yva, mva):
        torch.manual_seed(self.cfg["RANDOM_SEED"])
        self.model = self._build().to(DEVICE)
        pos = (ytr * mtr).sum(0).astype(np.float64)
        neg = np.maximum(mtr.sum(0) - pos, 1.0)
        pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1.0), 1.0, 50.0),
                                  dtype=torch.float32, device=DEVICE)
        lossf = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.cfg["MLP_LR"], weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)

        Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
        ytr_t = torch.tensor(ytr, dtype=torch.float32)
        mtr_t = torch.tensor(mtr, dtype=torch.float32)
        ds = torch.utils.data.TensorDataset(Xtr_t, ytr_t, mtr_t)
        dl = torch.utils.data.DataLoader(ds, batch_size=self.cfg["MLP_BATCH"], shuffle=True,
                                         drop_last=len(ds) > self.cfg["MLP_BATCH"])
        best, best_epoch, patience = -1.0, -1, 6
        for epoch in range(self.cfg["MLP_EPOCHS"]):
            self.model.train(); tot = 0.0; nb = 0
            for xb, yb, mb in dl:
                xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
                opt.zero_grad()
                logits = self.model(xb)
                loss = (lossf(logits, yb) * mb).sum() / mb.sum().clamp(min=1.0)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 5.0)
                opt.step()
                tot += float(loss.item()); nb += 1
            pva = self.predict_proba(Xva)
            aps = []
            for j in range(self.n_labels):
                mm = mva[:, j].astype(bool)
                if mm.sum() > 10 and 0 < yva[mm, j].sum() < mm.sum():
                    try:
                        aps.append(average_precision_score(yva[mm, j], pva[mm, j]))
                    except Exception:
                        pass
            macro_ap = float(np.mean(aps)) if aps else 0.0
            sched.step(macro_ap)
            self.history.append({"epoch": epoch, "train_loss": tot / max(nb, 1), "val_macro_ap": macro_ap})
            LOG.info("epoch %02d | loss %.4f | val macro-PR-AUC %.4f", epoch, tot / max(nb, 1), macro_ap)
            if macro_ap > best + 1e-4:
                best, best_epoch = macro_ap, epoch
                self.best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
            elif epoch - best_epoch >= patience:
                LOG.info("early stopping at epoch %d (best %d, macro-PR-AUC %.4f)", epoch, best_epoch, best)
                break
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
        self.best_val_ap = best
        return self

    def predict_proba(self, X) -> np.ndarray:
        self.model.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(X), 2048):
                xb = torch.tensor(X[i:i + 2048], dtype=torch.float32, device=DEVICE)
                outs.append(torch.sigmoid(self.model(xb)).cpu().numpy())
        return np.vstack(outs).astype(np.float32)

def fit_advanced_model(XH, Y_TR, M_TR, Y_VA, M_VA):
    """Torch head when available; otherwise a dense logistic OvR over the same features."""
    if HAVE_TORCH:
        head = TorchMultiLabelHead(XH["train"].shape[1], len(TECHS), CONFIG)
        head.fit(XH["train"], Y_TR, M_TR, XH["val"], Y_VA, M_VA)
        return head, "hybrid_mlp"
    LOG.warning("torch unavailable - the advanced model falls back to dense logistic OvR")
    head = MaskedOvR(lambda: LogisticRegression(max_iter=300, C=1.0, class_weight="balanced"),
                     TECHS, "hybrid_logreg", seed=CONFIG["RANDOM_SEED"])
    head.fit(XH["train"], Y_TR, M_TR)
    return head, "hybrid_logreg"

t0 = time.time()
ADV_MODEL, ADV_NAME = fit_advanced_model(XH, Y_TR, M_TR, Y_VA, M_VA)
RUN["advanced_training_seconds"] = round(time.time() - t0, 1)
p_adv_va = ADV_MODEL.predict_proba(XH["val"])
BASELINES[ADV_NAME] = masked_metrics(Y_VA, p_adv_va, M_VA, DEFAULT_THRESHOLDS, TECHS)

model_tbl = pd.DataFrame([summarise(k, v) for k, v in BASELINES.items()])
RESULTS["validation_models"] = model_tbl.to_dict("records")
print(f"advanced model: {ADV_NAME} | semantic channel: {HYBRID_BUNDLE['encoder'].mode} | "
      f"{RUN['advanced_training_seconds']:.0f}s on {DEVICE}")
display(model_tbl)

13:46:54 | INFO    | epoch 00 | loss 0.6387 | val macro-PR-AUC 0.4645
13:46:56 | INFO    | epoch 01 | loss 0.4784 | val macro-PR-AUC 0.5200
13:46:57 | INFO    | epoch 02 | loss 0.4255 | val macro-PR-AUC 0.5479
13:46:58 | INFO    | epoch 03 | loss 0.3911 | val macro-PR-AUC 0.5604
13:46:59 | INFO    | epoch 04 | loss 0.3681 | val macro-PR-AUC 0.5743
13:47:00 | INFO    | epoch 05 | loss 0.3485 | val macro-PR-AUC 0.5799
13:47:01 | INFO    | epoch 06 | loss 0.3344 | val macro-PR-AUC 0.5878
13:47:03 | INFO    | epoch 07 | loss 0.3227 | val macro-PR-AUC 0.5838
13:47:04 | INFO    | epoch 08 | loss 0.3113 | val macro-PR-AUC 0.5947
13:47:06 | INFO    | epoch 09 | loss 0.3015 | val macro-PR-AUC 0.5987
13:47:07 | INFO    | epoch 10 | loss 0.2929 | val macro-PR-AUC 0.5878
13:47:08 | INFO    | epoch 11 | loss 0.2879 | val macro-PR-AUC 0.5904
13:47:10 | INFO    | epoch 12 | loss 0.2786 | val macro-PR-AUC 0.6049
13:47:11 | INFO    | epoch 13 | loss 0.2732 | val macro-PR-AUC 0.6072
13:47:12 | INFO    |

advanced model: hybrid_mlp | semantic channel: sentence_transformer | 43s on cuda


,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,macro_precision,macro_recall,macro_pr_auc,micro_pr_auc,hamming_loss,subset_accuracy
0,rules_full,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,1.0000
1,rules_reduced,0.2219,0.1412,0.1754,1.0000,0.1248,0.3077,0.1034,0.1820,0.2019,0.0771,0.0543
2,majority_class,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0919,0.3304,0.0881,0.0417
3,frequency_prior,0.3674,0.1318,0.3975,0.2299,0.9146,0.0845,0.3590,0.0919,0.3304,0.2775,0.0000
4,tfidf_logreg,0.4766,0.2987,0.5059,0.3903,0.6119,0.2420,0.4459,0.3079,0.4626,0.1184,0.0272
5,struct_gbdt,0.6038,0.4082,0.5500,0.7724,0.4956,0.5774,0.3626,0.4645,0.7230,0.0573,0.1363
6,hybrid_mlp,0.6743,0.5476,0.7140,0.5599,0.8475,0.4601,0.7543,0.6280,0.7859,0.0721,0.1246


## 19. Hybrid feature comparison — what information actually matters

Four channel-isolation runs with an identical model (masked OvR logistic regression, so the only thing that varies
is the input representation):

| Run | Input |
|---|---|
| `text_only` | redacted page text |
| `resource_only` | filtered resource-graph and URL tokens |
| `structure_only` | DOM/resource numeric features |
| `hybrid` | text + resource + structure |

This answers "what information is actually useful for technology fingerprinting?" and separates the contribution
of *what a page says* from *what it loads* from *how it is shaped*.

In [36]:
# --- channel ablation -----------------------------------------------------
CHANNEL_RUNS = ["text_only", "resource_only", "C_structure", "hybrid"]
CHANNEL_RESULTS: Dict[str, Dict[str, Any]] = {}

for regime in CHANNEL_RUNS:
    t0 = time.time()
    _, Xr = get_regime(regime)
    clf = MaskedOvR(logreg_factory, TECHS, f"chan_{regime}", max_rows=100_000,
                    seed=CONFIG["RANDOM_SEED"])
    clf.fit(Xr["train"], Y_TR, M_TR)
    pv = clf.predict_proba(Xr["val"])
    CHANNEL_RESULTS[regime] = masked_metrics(Y_VA, pv, M_VA, DEFAULT_THRESHOLDS, TECHS)
    CHANNEL_RESULTS[regime]["seconds"] = round(time.time() - t0, 1)
    CHANNEL_RESULTS[regime]["n_features"] = int(Xr["train"].shape[1])
    LOG.info("channel %s: micro-F1=%.4f macro-F1=%.4f (%.0fs, %d features)", regime,
             CHANNEL_RESULTS[regime]["micro_f1"], CHANNEL_RESULTS[regime]["macro_f1"],
             CHANNEL_RESULTS[regime]["seconds"], Xr["train"].shape[1])

# These four matrices are ~1 GB each at 100k pages and none is referenced again after this
# section; the A/B/C ablation that follows needs headroom to build two wider ones.
del Xr, clf, pv
free("X_text")
# "hybrid" is an alias of B_reduced, which S20/S23/S27 still need - do not evict it here.
evict_regimes("text_only", "resource_only")

chan_tbl = pd.DataFrame([{**summarise(REGIME_SPECS[r]["label"], CHANNEL_RESULTS[r]),
                          "n_features": CHANNEL_RESULTS[r]["n_features"],
                          "seconds": CHANNEL_RESULTS[r]["seconds"]} for r in CHANNEL_RUNS])
chan_tbl.to_csv(PATHS["reports"] / "channel_ablation.csv", index=False)
RESULTS["channel_ablation"] = chan_tbl.to_dict("records")
display(chan_tbl)

if CONFIG["MAKE_FIGURES"]:
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(CHANNEL_RUNS)); w = 0.38
    ax.bar(x - w/2, [CHANNEL_RESULTS[r]["micro_f1"] for r in CHANNEL_RUNS], w, label="micro-F1", color="#2b6cb0")
    ax.bar(x + w/2, [CHANNEL_RESULTS[r]["macro_f1"] for r in CHANNEL_RUNS], w, label="macro-F1", color="#dd9b3c")
    ax.set_xticks(x); ax.set_xticklabels([REGIME_SPECS[r]["label"] for r in CHANNEL_RUNS],
                                         rotation=18, ha="right", fontsize=8)
    ax.set_ylabel("F1 (validation)"); ax.legend(); ax.set_title("Which feature channel carries the signal?")
    savefig(fig, "05_channel_ablation")

13:47:41 | INFO    | [chan_text_only] fitted 5/39 labels (5s elapsed)
13:47:48 | INFO    | [chan_text_only] fitted 10/39 labels (12s elapsed)
13:47:57 | INFO    | [chan_text_only] fitted 15/39 labels (21s elapsed)
13:48:04 | INFO    | [chan_text_only] fitted 20/39 labels (28s elapsed)
13:48:10 | INFO    | [chan_text_only] fitted 25/39 labels (34s elapsed)
13:48:15 | INFO    | [chan_text_only] fitted 30/39 labels (38s elapsed)
13:48:24 | INFO    | [chan_text_only] fitted 35/39 labels (48s elapsed)
13:48:27 | INFO    | [chan_text_only] fitted 39/39 labels (51s elapsed)
13:48:28 | INFO    | channel text_only: micro-F1=0.4766 macro-F1=0.2987 (52s, 59970 features)
13:48:44 | INFO    | resource_only -> resource_only: resource=59871 | train (67541, 59871)
13:48:52 | INFO    | [chan_resource_only] fitted 5/39 labels (9s elapsed)
13:48:56 | INFO    | [chan_resource_only] fitted 10/39 labels (13s elapsed)
13:49:00 | INFO    | [chan_resource_only] fitted 15/39 labels (16s elapsed)
13:49:08 | INFO

,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,macro_precision,macro_recall,macro_pr_auc,micro_pr_auc,hamming_loss,subset_accuracy,n_features,seconds
0,text only (redacted),0.4766,0.2987,0.5059,0.3903,0.6119,0.2420,0.4459,0.3079,0.4626,0.1184,0.0272,59970,51.5
1,resource graph only (filtered),0.6607,0.5198,0.7023,0.5629,0.7996,0.4447,0.7316,0.6007,0.7702,0.0724,0.1428,59871,57.1
2,C - structure only,0.3956,0.2835,0.5550,0.2674,0.7594,0.2135,0.7995,0.3116,0.3641,0.2045,0.0006,78,41.0
3,hybrid (text + resource + structure),0.7235,0.5764,0.7420,0.6596,0.8010,0.5002,0.7300,0.6372,0.8247,0.0539,0.2142,119919,887.6


14:04:56 | INFO    | figure saved: 05_channel_ablation.png


## 20. Fingerprint leakage experiments (A / B / C)

The central experiment. The same model family is trained three times over the three regimes defined in S15b, and
the results are read as a *decomposition*, not a leaderboard:

```text
A  =  fingerprint echo  +  learnable signal  +  prior
B  =                       learnable signal  +  prior      (signature removed)
C  =                       structural signal +  prior      (all lexical content removed)

A - B  ->  how much of the apparent performance was the labeller talking to itself
B - C  ->  how much non-signature lexical evidence contributes
C - prior -> pure structural generalisation
```

A large `A - B` gap is not a failure. It is the measurement this project exists to make: it quantifies how much of
"technology detection accuracy" in the literature is circular. What matters is whether **B and C stay meaningfully
above the frequency prior** — that is the evidence that something was genuinely learned.

In [37]:
# --- A / B / C ablation ---------------------------------------------------
ABLATION_REGIMES = ["A_full", "B_reduced", "C_structure"]
ABLATION: Dict[str, Dict[str, Any]] = {}
ABLATION_MODELS: Dict[str, Any] = {}
ABLATION_FS: Dict[str, FeatureSpace] = {}

for regime in ABLATION_REGIMES:
    t0 = time.time()
    fs, Xr = get_regime(regime)
    clf = MaskedOvR(logreg_factory, TECHS, f"abl_{regime}", max_rows=120_000,
                    seed=CONFIG["RANDOM_SEED"])
    clf.fit(Xr["train"], Y_TR, M_TR)
    pv = clf.predict_proba(Xr["val"])
    res = masked_metrics(Y_VA, pv, M_VA, DEFAULT_THRESHOLDS, TECHS)
    res["seconds"] = round(time.time() - t0, 1)
    res["n_features"] = int(Xr["train"].shape[1])
    ABLATION[regime] = res
    ABLATION_MODELS[regime] = clf
    ABLATION_FS[regime] = fs
    LOG.info("%s | micro-F1 %.4f | macro-F1 %.4f | %d features | %.0fs",
             REGIME_SPECS[regime]["label"], res["micro_f1"], res["macro_f1"],
             res["n_features"], res["seconds"])

prior_micro = BASELINES["frequency_prior"]["micro_f1"]
abl_tbl = pd.DataFrame([{**summarise(REGIME_SPECS[r]["label"], ABLATION[r]),
                         "regime": r, "n_features": ABLATION[r]["n_features"]}
                        for r in ABLATION_REGIMES])
abl_tbl["micro_f1_above_prior"] = (abl_tbl["micro_f1"] - prior_micro).round(4)
abl_tbl.to_csv(PATHS["reports"] / "ablation_results.csv", index=False)

drop_ab = ABLATION["A_full"]["micro_f1"] - ABLATION["B_reduced"]["micro_f1"]
drop_bc = ABLATION["B_reduced"]["micro_f1"] - ABLATION["C_structure"]["micro_f1"]
RESULTS["ablation"] = {
    "table": abl_tbl.to_dict("records"),
    "A_minus_B_micro_f1": round(float(drop_ab), 4),
    "B_minus_C_micro_f1": round(float(drop_bc), 4),
    "A_relative_drop_pct": round(float(100 * drop_ab / max(ABLATION["A_full"]["micro_f1"], 1e-9)), 1),
    "prior_micro_f1": round(float(prior_micro), 4),
}
print(f"A -> B  micro-F1 drop : {drop_ab:+.4f}  ({RESULTS['ablation']['A_relative_drop_pct']:.1f}% of A)")
print(f"B -> C  micro-F1 drop : {drop_bc:+.4f}")
print(f"frequency prior        : {prior_micro:.4f}")
display(abl_tbl)

14:07:12 | INFO    | A_full -> A_full: text=60000, resource=60000, struct=78, fingerprint=140 | train (67541, 120218)
14:07:25 | INFO    | [abl_A_full] fitted 5/39 labels (13s elapsed)
14:07:39 | INFO    | [abl_A_full] fitted 10/39 labels (27s elapsed)
14:07:52 | INFO    | [abl_A_full] fitted 15/39 labels (41s elapsed)
14:08:06 | INFO    | [abl_A_full] fitted 20/39 labels (54s elapsed)
14:08:18 | INFO    | [abl_A_full] fitted 25/39 labels (67s elapsed)
14:08:32 | INFO    | [abl_A_full] fitted 30/39 labels (80s elapsed)
14:08:45 | INFO    | [abl_A_full] fitted 35/39 labels (94s elapsed)
14:08:55 | INFO    | [abl_A_full] fitted 39/39 labels (103s elapsed)
14:08:56 | INFO    | A - full evidence (upper bound) | micro-F1 0.9974 | macro-F1 0.9741 | 120218 features | 240s
14:11:04 | INFO    | [abl_B_reduced] fitted 5/39 labels (128s elapsed)
14:12:39 | INFO    | [abl_B_reduced] fitted 10/39 labels (223s elapsed)
14:14:38 | INFO    | [abl_B_reduced] fitted 15/39 labels (342s elapsed)
14:16:16 

A -> B  micro-F1 drop : +0.2739  (27.5% of A)
B -> C  micro-F1 drop : +0.3279
frequency prior        : 0.3674


,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,macro_precision,macro_recall,macro_pr_auc,micro_pr_auc,hamming_loss,subset_accuracy,regime,n_features,micro_f1_above_prior
0,A - full evidence (upper bound),0.9974,0.9741,0.9975,0.9962,0.9986,0.9610,0.9900,0.9936,0.9998,0.0005,0.9826,A_full,120218,0.6300
1,B - fingerprint-reduced,0.7235,0.5764,0.7420,0.6596,0.8010,0.5002,0.7300,0.6372,0.8247,0.0539,0.2142,B_reduced,119919,0.3561
2,C - structure only,0.3956,0.2835,0.5550,0.2674,0.7594,0.2135,0.7995,0.3116,0.3641,0.2045,0.0006,C_structure,78,0.0282


In [38]:
# --- per-technology ablation view ----------------------------------------
per_reg = {r: ABLATION[r]["per_technology"].set_index("technology") for r in ABLATION_REGIMES}
ABL_PER_TECH = pd.DataFrame({
    "category": [TECH_CATEGORY[t] for t in TECHS],
    "support": per_reg["A_full"]["support"],
    "f1_A_full": per_reg["A_full"]["f1"],
    "f1_B_reduced": per_reg["B_reduced"]["f1"],
    "f1_C_structure": per_reg["C_structure"]["f1"],
}, index=TECHS)
ABL_PER_TECH["retained_B_pct"] = (100 * ABL_PER_TECH["f1_B_reduced"] /
                                  ABL_PER_TECH["f1_A_full"].replace(0, np.nan)).round(1)
ABL_PER_TECH = ABL_PER_TECH.sort_values("f1_B_reduced", ascending=False)
ABL_PER_TECH.to_csv(PATHS["reports"] / "ablation_per_technology.csv")
RESULTS["ablation_per_technology"] = ABL_PER_TECH.reset_index().rename(
    columns={"index": "technology"}).to_dict("records")

if CONFIG["MAKE_FIGURES"] and len(ABL_PER_TECH):
    top = ABL_PER_TECH.head(24).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, max(6, 0.34 * len(top))))
    yy = np.arange(len(top)); h = 0.26
    ax.barh(yy + h, top["f1_A_full"], h, label="A full (upper bound)", color="#9fb6cc")
    ax.barh(yy,      top["f1_B_reduced"], h, label="B fingerprint-reduced", color="#2b6cb0")
    ax.barh(yy - h,  top["f1_C_structure"], h, label="C structure only", color="#dd9b3c")
    ax.set_yticks(yy); ax.set_yticklabels(top.index, fontsize=8)
    ax.set_xlabel("F1 (validation)"); ax.legend(loc="lower right")
    ax.set_title("Per-technology F1 across leakage regimes")
    savefig(fig, "06_ablation_per_technology")

print("Technologies that survive fingerprint removal best (high F1 in regime B):")
display(ABL_PER_TECH.head(12))
print("Technologies that collapse without their signature (pure fingerprint echo):")
display(ABL_PER_TECH.sort_values("retained_B_pct").head(12))

14:22:11 | INFO    | figure saved: 06_ablation_per_technology.png


Technologies that survive fingerprint removal best (high F1 in regime B):


,category,support,f1_A_full,f1_B_reduced,f1_C_structure,retained_B_pct
WordPress,CMS,5914,0.998306,0.982226,0.894784,98.4
Wix,site-builder,234,0.966527,0.939271,0.727273,97.2
Bootstrap,CSS/UI,3254,0.999232,0.901126,0.529458,90.2
Google Tag Manager,analytics,78,0.968153,0.891566,0.482085,92.1
Tailwind CSS,CSS/UI,1454,0.998280,0.882617,0.512891,88.4
jQuery,frontend,3218,0.999845,0.853035,0.752868,85.3
Font Awesome,CSS/UI,4164,0.999880,0.823013,0.657534,82.3
Google Fonts,CSS/UI,5191,0.999904,0.812893,0.687559,81.3
Joomla,CMS,180,0.978142,0.809412,0.130698,82.7
Next.js,meta-framework,528,0.982326,0.806818,0.525524,82.1


Technologies that collapse without their signature (pure fingerprint echo):


,category,support,f1_A_full,f1_B_reduced,f1_C_structure,retained_B_pct
Netlify,hosting,35,0.985507,0.146667,0.023705,14.9
Django,backend,33,0.869565,0.134078,0.018310,15.4
PayPal,payments,60,1.000000,0.165517,0.021207,16.6
Fastly,CDN,47,0.959184,0.164835,0.023105,17.2
Akamai,CDN,91,0.967742,0.238372,0.050297,24.6
Stripe,payments,114,0.978541,0.281690,0.053529,28.8
HubSpot,marketing,125,0.996016,0.306173,0.054861,30.7
Laravel,backend,95,0.974359,0.303951,0.041180,31.2
Express,backend,104,0.995215,0.387435,0.063362,38.9
Amazon CloudFront,CDN,423,0.998819,0.404898,0.141956,40.5


## 21. Threshold optimization

A single 0.5 cut-off is wrong for multi-label problems with 100:1 imbalance ratios: a technology present on 0.4%
of pages needs a far lower operating point than one present on 30%.

Thresholds are tuned **per technology on the validation set only**, then **frozen** before the test set is
touched. Both the default and tuned results are reported so the gain from tuning is visible and cannot be confused
with model improvement.

In [39]:
# --- tune on validation, then freeze --------------------------------------
PRIMARY_REGIME = "B_reduced"   # the honest operating condition used for the headline model

p_primary_va = ADV_MODEL.predict_proba(XH["val"])
THRESHOLDS = tune_thresholds(Y_VA, p_primary_va, M_VA)
THRESHOLDS_FROZEN = THRESHOLDS.copy()

default_va = masked_metrics(Y_VA, p_primary_va, M_VA, DEFAULT_THRESHOLDS, TECHS)
tuned_va   = masked_metrics(Y_VA, p_primary_va, M_VA, THRESHOLDS, TECHS)

thr_tbl = pd.DataFrame({
    "technology": TECHS,
    "category": [TECH_CATEGORY[t] for t in TECHS],
    "threshold": THRESHOLDS.round(3),
    "val_prevalence": [round(float(((Y_VA[:, j] == 1) & (M_VA[:, j] == 1)).sum() /
                                   max(1, M_VA[:, j].sum())), 4) for j in range(len(TECHS))],
    "val_f1_default": default_va["per_technology"]["f1"].round(4).values,
    "val_f1_tuned": tuned_va["per_technology"]["f1"].round(4).values,
}).sort_values("val_f1_tuned", ascending=False)
thr_tbl["f1_gain"] = (thr_tbl["val_f1_tuned"] - thr_tbl["val_f1_default"]).round(4)
thr_tbl.to_csv(PATHS["reports"] / "thresholds.csv", index=False)
save_json({t: float(v) for t, v in zip(TECHS, THRESHOLDS)}, PATHS["models"] / "thresholds.json")

RESULTS["threshold_tuning"] = {
    "default": summarise("default_0.5", default_va),
    "tuned": summarise("val_tuned", tuned_va),
    "mean_threshold": round(float(THRESHOLDS.mean()), 3),
    "table": thr_tbl.to_dict("records"),
}
print(f"validation micro-F1  default 0.5 : {default_va['micro_f1']:.4f}")
print(f"validation micro-F1  tuned       : {tuned_va['micro_f1']:.4f}  "
      f"(+{tuned_va['micro_f1'] - default_va['micro_f1']:.4f})")
print(f"validation macro-F1  default 0.5 : {default_va['macro_f1']:.4f}")
print(f"validation macro-F1  tuned       : {tuned_va['macro_f1']:.4f}  "
      f"(+{tuned_va['macro_f1'] - default_va['macro_f1']:.4f})")
print("THRESHOLDS ARE NOW FROZEN - the test set has not been touched.")
display(thr_tbl.head(20))

validation micro-F1  default 0.5 : 0.6743
validation micro-F1  tuned       : 0.7237  (+0.0493)
validation macro-F1  default 0.5 : 0.5476
validation macro-F1  tuned       : 0.6316  (+0.0840)
THRESHOLDS ARE NOW FROZEN - the test set has not been touched.


,technology,category,threshold,val_prevalence,val_f1_default,val_f1_tuned,f1_gain
34,Wix,site-builder,0.76,0.0162,0.9894,0.9957,0.0063
36,WordPress,CMS,0.55,0.4094,0.9835,0.9836,0.0001
3,Angular,frontend,0.89,0.0035,0.7244,0.9293,0.2049
21,Next.js,meta-framework,0.95,0.0365,0.8659,0.9213,0.0554
16,Joomla,CMS,0.93,0.0124,0.7785,0.9205,0.1420
28,Shopify,ecommerce,0.95,0.0058,0.7958,0.9080,0.1122
33,Webflow,site-builder,0.95,0.0035,0.8738,0.8936,0.0198
5,Bootstrap,CSS/UI,0.71,0.2450,0.8674,0.8871,0.0197
30,Tailwind CSS,CSS/UI,0.91,0.1005,0.8204,0.8822,0.0618
8,Drupal,CMS,0.87,0.0132,0.7953,0.8633,0.0680


## 22. Calibration

A probability of 0.95 should mean the technology is present about 95% of the time. Calibration is checked on
validation with reliability diagrams and **Expected Calibration Error** (10 equal-width bins, trusted cells only),
then isotonic regression is fitted per label — again on validation only — and the improvement reported.

Miscalibration here is expected and diagnostic: `pos_weight` deliberately distorts the output distribution to
fight class imbalance, so raw sigmoid outputs tend to be over-confident on rare technologies. That is a reasonable
trade for F1 and a bad one for anyone reading the number as a probability, which is why the isotonic layer is
fitted and shipped alongside the raw model.

In [40]:
# --- calibration ----------------------------------------------------------
def expected_calibration_error(y: np.ndarray, p: np.ndarray, bins: int = 10) -> Tuple[float, pd.DataFrame]:
    edges = np.linspace(0, 1, bins + 1)
    rows, ece, n = [], 0.0, len(y)
    for b in range(bins):
        lo, hi = edges[b], edges[b + 1]
        sel = (p >= lo) & (p < hi if b < bins - 1 else p <= hi)
        if not sel.any():
            rows.append({"bin_lo": lo, "bin_hi": hi, "n": 0, "confidence": np.nan, "accuracy": np.nan})
            continue
        conf, acc = float(p[sel].mean()), float(y[sel].mean())
        ece += (sel.sum() / n) * abs(acc - conf)
        rows.append({"bin_lo": lo, "bin_hi": hi, "n": int(sel.sum()),
                     "confidence": conf, "accuracy": acc})
    return float(ece), pd.DataFrame(rows)

class IsotonicCalibrator:
    """Per-label isotonic calibration fitted on validation trusted cells."""

    def __init__(self, labels: Sequence[str]):
        self.labels = list(labels); self.models_: List[Optional[IsotonicRegression]] = []

    def fit(self, y, p, m):
        self.models_ = []
        for j in range(len(self.labels)):
            sel = m[:, j].astype(bool)
            yj, pj = y[sel, j], p[sel, j]
            if sel.sum() < 50 or yj.sum() < 10 or yj.sum() == len(yj):
                self.models_.append(None); continue
            try:
                self.models_.append(IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(pj, yj))
            except Exception:
                self.models_.append(None)
        return self

    def transform(self, p: np.ndarray) -> np.ndarray:
        out = p.copy()
        for j, mdl in enumerate(self.models_):
            if mdl is not None:
                out[:, j] = np.clip(mdl.predict(p[:, j]), 0.0, 1.0)
        return out

ece_rows, diagrams = [], {}
for j, tech in enumerate(TECHS):
    sel = M_VA[:, j].astype(bool)
    if sel.sum() < 50 or Y_VA[sel, j].sum() < 10:
        continue
    e, table = expected_calibration_error(Y_VA[sel, j], p_primary_va[sel, j])
    ece_rows.append({"technology": tech, "ece": round(e, 4), "support": int(Y_VA[sel, j].sum()),
                     "mean_confidence": round(float(p_primary_va[sel, j].mean()), 4),
                     "actual_rate": round(float(Y_VA[sel, j].mean()), 4)})
    diagrams[tech] = table

CALIBRATOR = IsotonicCalibrator(TECHS).fit(Y_VA, p_primary_va, M_VA)
p_cal_va = CALIBRATOR.transform(p_primary_va)
ece_after = []
for j, tech in enumerate(TECHS):
    sel = M_VA[:, j].astype(bool)
    if sel.sum() < 50 or Y_VA[sel, j].sum() < 10:
        continue
    e, _ = expected_calibration_error(Y_VA[sel, j], p_cal_va[sel, j])
    ece_after.append({"technology": tech, "ece_calibrated": round(e, 4)})

CALIB_TBL = pd.DataFrame(ece_rows).merge(pd.DataFrame(ece_after), on="technology", how="left")
CALIB_TBL["improvement"] = (CALIB_TBL["ece"] - CALIB_TBL["ece_calibrated"]).round(4)
CALIB_TBL = CALIB_TBL.sort_values("ece", ascending=False)
CALIB_TBL.to_csv(PATHS["reports"] / "calibration.csv", index=False)
RESULTS["calibration"] = {
    "mean_ece_raw": round(float(CALIB_TBL["ece"].mean()), 4) if len(CALIB_TBL) else None,
    "mean_ece_isotonic": round(float(CALIB_TBL["ece_calibrated"].mean()), 4) if len(CALIB_TBL) else None,
    "table": CALIB_TBL.to_dict("records"),
}
print(f"mean ECE raw       : {RESULTS['calibration']['mean_ece_raw']}")
print(f"mean ECE isotonic  : {RESULTS['calibration']['mean_ece_isotonic']}")
display(CALIB_TBL.head(12))

if CONFIG["MAKE_FIGURES"] and diagrams:
    picks = list(CALIB_TBL.sort_values("support", ascending=False)["technology"][:6])
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, tech in zip(axes.ravel(), picks):
        d = diagrams[tech].dropna()
        ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
        ax.plot(d["confidence"], d["accuracy"], "o-", color="#2b6cb0", label="raw")
        j = TECHS.index(tech); sel = M_VA[:, j].astype(bool)
        _, dc = expected_calibration_error(Y_VA[sel, j], p_cal_va[sel, j])
        dc = dc.dropna()
        ax.plot(dc["confidence"], dc["accuracy"], "s-", color="#dd9b3c", alpha=0.8, label="isotonic")
        ax.set_title(tech, fontsize=10); ax.set_xlabel("predicted"); ax.set_ylabel("observed")
        ax.legend(fontsize=7)
    for ax in axes.ravel()[len(picks):]:
        ax.axis("off")
    fig.suptitle("Reliability diagrams (validation, trusted cells)")
    savefig(fig, "07_calibration")

mean ECE raw       : 0.0526
mean ECE isotonic  : 0.0


,technology,ece,support,mean_confidence,actual_rate,ece_calibrated,improvement
18,LiteSpeed,0.2134,1156,0.2950,0.0816,0.0,0.2134
4,Apache,0.2057,3241,0.4297,0.2239,0.0,0.2057
22,Nginx,0.1781,3881,0.4462,0.2682,0.0,0.1781
24,PHP,0.1600,3860,0.4501,0.2901,0.0,0.1600
12,Google Analytics,0.1406,3941,0.4190,0.2783,0.0,0.1406
2,Amazon CloudFront,0.1149,423,0.1447,0.0298,0.0,0.1149
6,Cloudflare,0.1075,2645,0.2903,0.1828,0.0,0.1075
0,ASP.NET,0.0860,523,0.1221,0.0361,0.0,0.0860
11,Font Awesome,0.0821,4164,0.3945,0.3124,0.0,0.0821
19,Microsoft IIS,0.0797,370,0.1052,0.0256,0.0,0.0797


14:22:15 | INFO    | figure saved: 07_calibration.png


## 23. Final test evaluation

First and only contact with the test set. Everything below uses artefacts frozen upstream: the model selected on
validation, thresholds tuned on validation, and the calibrator fitted on validation. No decision after this point
feeds back into any model.

In [41]:
# --- test-set evaluation --------------------------------------------------
TEST_RESULTS: Dict[str, Dict[str, Any]] = {}

p_adv_te = ADV_MODEL.predict_proba(XH["test"])
TEST_RESULTS[f"{ADV_NAME}_tuned"] = masked_metrics(Y_TE, p_adv_te, M_TE, THRESHOLDS_FROZEN, TECHS)
TEST_RESULTS[f"{ADV_NAME}_default05"] = masked_metrics(Y_TE, p_adv_te, M_TE, DEFAULT_THRESHOLDS, TECHS)

p_adv_te_cal = CALIBRATOR.transform(p_adv_te)
TEST_RESULTS[f"{ADV_NAME}_calibrated"] = masked_metrics(Y_TE, p_adv_te_cal, M_TE, THRESHOLDS_FROZEN, TECHS)

for regime in ABLATION_REGIMES:
    _, Xr = get_regime(regime)
    pt = ABLATION_MODELS[regime].predict_proba(Xr["test"])
    thr = tune_thresholds(Y_VA, ABLATION_MODELS[regime].predict_proba(Xr["val"]), M_VA)
    TEST_RESULTS[f"logreg_{regime}"] = masked_metrics(Y_TE, pt, M_TE, thr, TECHS)
    if regime != "B_reduced":        # B_reduced is reused by the temporal section
        evict_regimes(regime)
del Xr

TEST_RESULTS["rules_full"] = masked_metrics(Y_TE, rule_predictions(HITS_TE, TECHS, False),
                                            M_TE, DEFAULT_THRESHOLDS, TECHS)
TEST_RESULTS["rules_reduced"] = masked_metrics(Y_TE, rule_predictions(HITS_TE, TECHS, True),
                                               M_TE, DEFAULT_THRESHOLDS, TECHS)
p_freq_te = frequency_baseline(Y_TR, M_TR, len(IDX["test"]))
TEST_RESULTS["majority_class"] = masked_metrics(Y_TE, p_freq_te, M_TE, DEFAULT_THRESHOLDS, TECHS)
TEST_RESULTS["frequency_prior"] = masked_metrics(Y_TE, p_freq_te, M_TE, THR_PRIOR, TECHS)

test_tbl = pd.DataFrame([summarise(k, v) for k, v in TEST_RESULTS.items()])
test_tbl.to_csv(PATHS["reports"] / "classification_results.csv", index=False)
RESULTS["test_results"] = test_tbl.to_dict("records")

PRIMARY_KEY = f"{ADV_NAME}_tuned"
PER_TECH_TEST = TEST_RESULTS[PRIMARY_KEY]["per_technology"].copy()
PER_TECH_TEST["category"] = PER_TECH_TEST["technology"].map(TECH_CATEGORY)
PER_TECH_TEST = PER_TECH_TEST.sort_values("f1", ascending=False)
PER_TECH_TEST.to_csv(PATHS["reports"] / "per_technology_results.csv", index=False)
RESULTS["per_technology_test"] = PER_TECH_TEST.to_dict("records")

print("=" * 78)
print(f"FINAL TEST RESULTS  ({len(IDX['test']):,} pages / {len(test_domains):,} unseen domains)")
print("=" * 78)
display(test_tbl)
print("\nPer-technology (primary model, frozen thresholds):")
display(PER_TECH_TEST[["technology", "category", "precision", "recall", "f1",
                       "pr_auc", "support", "threshold"]].round(4))

14:22:19 | INFO    | evicted cached features for regime A_full
14:22:19 | INFO    | [mem] post-evict rss=5.64 GB
14:22:23 | INFO    | evicted cached features for regime C_structure
14:22:23 | INFO    | [mem] post-evict rss=5.46 GB


FINAL TEST RESULTS  (14,473 pages / 13,001 unseen domains)


,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,macro_precision,macro_recall,macro_pr_auc,micro_pr_auc,hamming_loss,subset_accuracy
0,hybrid_mlp_tuned,0.7189,0.6059,0.7303,0.6672,0.7793,0.6188,0.6145,0.6212,0.7894,0.0539,0.1973
1,hybrid_mlp_default05,0.6752,0.5506,0.7138,0.5612,0.8473,0.4645,0.7467,0.6212,0.7894,0.0721,0.1280
2,hybrid_mlp_calibrated,0.6760,0.4414,0.6281,0.8614,0.5563,0.7861,0.3717,0.5978,0.8312,0.0471,0.1900
3,logreg_A_full,0.9987,0.9863,0.9987,0.9985,0.9990,0.9839,0.9889,0.9945,0.9997,0.0002,0.9904
4,logreg_B_reduced,0.7362,0.6215,0.7489,0.6910,0.7878,0.6304,0.6304,0.6322,0.8262,0.0499,0.2443
5,logreg_C_structure,0.5309,0.3710,0.5762,0.4123,0.7452,0.3059,0.5426,0.3180,0.3811,0.1165,0.0074
6,rules_full,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,1.0000
7,rules_reduced,0.2162,0.1446,0.1718,1.0000,0.1212,0.3077,0.1077,0.1869,0.1989,0.0777,0.0566
8,majority_class,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0923,0.3366,0.0884,0.0453
9,frequency_prior,0.3689,0.1323,0.3996,0.2310,0.9149,0.0850,0.3590,0.0923,0.3366,0.2768,0.0000



Per-technology (primary model, frozen thresholds):


,technology,category,precision,recall,f1,pr_auc,support,threshold
34,Wix,site-builder,0.9966,0.9966,0.9966,0.9999,291,0.76
36,WordPress,CMS,0.9854,0.9787,0.9820,0.9973,6202,0.55
21,Next.js,meta-framework,0.9354,0.9128,0.9239,0.9613,539,0.95
33,Webflow,site-builder,1.0000,0.8519,0.9200,0.8886,54,0.95
16,Joomla,CMS,0.8961,0.9020,0.8990,0.9659,153,0.93
28,Shopify,ecommerce,0.9333,0.8333,0.8805,0.8764,84,0.95
30,Tailwind CSS,CSS/UI,0.9036,0.8379,0.8695,0.9384,1376,0.91
5,Bootstrap,CSS/UI,0.8971,0.8358,0.8654,0.9420,3149,0.71
8,Drupal,CMS,0.8861,0.8434,0.8642,0.8929,166,0.87
37,jQuery,frontend,0.8106,0.8562,0.8328,0.9166,3193,0.56


In [42]:
# --- test figures ---------------------------------------------------------
if CONFIG["MAKE_FIGURES"]:
    fig, ax = plt.subplots(figsize=(10, 6))
    order = test_tbl.sort_values("micro_f1")
    yy = np.arange(len(order))
    ax.barh(yy, order["micro_f1"], 0.4, label="micro-F1", color="#2b6cb0")
    ax.barh(yy + 0.4, order["macro_f1"], 0.4, label="macro-F1", color="#dd9b3c")
    ax.set_yticks(yy + 0.2); ax.set_yticklabels(order["model"], fontsize=8)
    ax.legend(); ax.set_title("Test-set performance by model")
    ax.text(0.02, 0.02, "rules_full is circular with the labelling function",
            transform=ax.transAxes, fontsize=7, style="italic")
    savefig(fig, "08_test_models")

    d = PER_TECH_TEST.dropna(subset=["f1"])
    fig, axes = plt.subplots(1, 2, figsize=(15, max(6, 0.32 * len(d))))
    dd = d.iloc[::-1]
    axes[0].barh(dd["technology"], dd["f1"], color="#2b6cb0")
    axes[0].set_xlabel("F1"); axes[0].set_title("Per-technology F1 (test)")
    axes[0].tick_params(labelsize=7)
    axes[1].scatter(d["support"], d["f1"], s=28, c="#2b6cb0")
    for _, r in d.iterrows():
        if r["support"] > 0:
            axes[1].annotate(r["technology"], (r["support"], r["f1"]), fontsize=6,
                             xytext=(3, 2), textcoords="offset points")
    axes[1].set_xscale("log"); axes[1].set_xlabel("test support (log)"); axes[1].set_ylabel("F1")
    axes[1].set_title("Does rarity explain difficulty?")
    savefig(fig, "09_per_technology_test")

14:22:25 | INFO    | figure saved: 08_test_models.png
14:22:27 | INFO    | figure saved: 09_per_technology_test.png


## 24. Error analysis

Errors are grouped by their likely *cause*, not just counted. For each technology the analysis separates:

* **False positives** — predicted present, labelled absent. Often a genuine detection the labeller missed
  (evidence stripped from the archive), which means the "error" may be a label error.
* **False negatives** — labelled present, predicted absent. Usually the fingerprint-reduced regime removing the
  only visible marker.
* **Confusion pairs** — technologies whose errors correlate, computed as the co-occurrence of FP(A) with true(B).
  This surfaces exactly the pairs called out in the brief: React/Next.js, Vue/Nuxt, WordPress/generic PHP.

In [43]:
# --- error analysis -------------------------------------------------------
pred_te = (p_adv_te >= THRESHOLDS_FROZEN[None, :]).astype(np.int8)
mask_te = M_TE.astype(bool)

err_rows = []
for j, tech in enumerate(TECHS):
    mm = mask_te[:, j]
    yj, pj = Y_TE[mm, j], pred_te[mm, j]
    tp = int(((yj == 1) & (pj == 1)).sum()); fp = int(((yj == 0) & (pj == 1)).sum())
    fn = int(((yj == 1) & (pj == 0)).sum()); tn = int(((yj == 0) & (pj == 0)).sum())
    err_rows.append({"technology": tech, "category": TECH_CATEGORY[tech],
                     "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                     "fp_rate": round(fp / max(fp + tn, 1), 4),
                     "fn_rate": round(fn / max(fn + tp, 1), 4),
                     "n_active_rules": int(LABEL_QUALITY.set_index("technology")
                                           .loc[tech, "n_active_rules"]) if tech in
                                           set(LABEL_QUALITY["technology"]) else 0,
                     "dominant_rule_share": float(LABEL_QUALITY.set_index("technology")
                                                  .loc[tech, "dominant_rule_share"]) if tech in
                                                  set(LABEL_QUALITY["technology"]) else np.nan})
ERROR_TBL = pd.DataFrame(err_rows).sort_values("fn_rate", ascending=False)

def diagnose(row) -> str:
    """Attribute the dominant error mode to a plausible cause."""
    causes = []
    if row["fn_rate"] > 0.5 and row["dominant_rule_share"] > 0.85:
        causes.append("single-rule label: signature removal leaves no learnable trace")
    if row["tp"] + row["fn"] < 60:
        causes.append("class imbalance / low support")
    if row["fp_rate"] > 0.05:
        causes.append("ambiguous fingerprint or ecosystem co-occurrence")
    if row["category"] in ("server", "CDN", "hosting", "backend") and row["fn_rate"] > 0.4:
        causes.append("header-derived label with no client-side correlate")
    if row["fn_rate"] > 0.7 and not causes:
        causes.append("insufficient page evidence")
    return "; ".join(causes) or "no dominant failure mode"

ERROR_TBL["likely_cause"] = ERROR_TBL.apply(diagnose, axis=1)
ERROR_TBL.to_csv(PATHS["reports"] / "error_analysis.csv", index=False)
RESULTS["error_analysis"] = ERROR_TBL.to_dict("records")
display(ERROR_TBL.head(18))

# --- confusion between technologies --------------------------------------
conf_pairs = []
for a in range(len(TECHS)):
    fp_a = mask_te[:, a] & (Y_TE[:, a] == 0) & (pred_te[:, a] == 1)
    if fp_a.sum() < 5:
        continue
    for b in range(len(TECHS)):
        if a == b:
            continue
        both = fp_a & mask_te[:, b] & (Y_TE[:, b] == 1)
        rate = both.sum() / max(fp_a.sum(), 1)
        base = float(((Y_TE[:, b] == 1) & mask_te[:, b]).sum() / max(mask_te[:, b].sum(), 1))
        if rate > 0.25 and rate > 2 * base:
            conf_pairs.append({"predicted": TECHS[a], "actually_present": TECHS[b],
                               "fp_count": int(fp_a.sum()), "share_of_fps": round(float(rate), 3),
                               "base_rate": round(base, 4),
                               "lift": round(float(rate / max(base, 1e-6)), 2)})
CONFUSION = pd.DataFrame(conf_pairs).sort_values("lift", ascending=False) if conf_pairs \
            else pd.DataFrame(columns=["predicted", "actually_present", "fp_count",
                                       "share_of_fps", "base_rate", "lift"])
CONFUSION.to_csv(PATHS["reports"] / "technology_confusion.csv", index=False)
RESULTS["confusion_pairs"] = CONFUSION.head(25).to_dict("records")
print("Technology confusion (false positives that land on pages using a related technology):")
display(CONFUSION.head(15))

# --- example errors, host-level only (no page content is exposed) ---------
ex_rows = []
test_df = EV_DEDUP.iloc[IDX["test"]].reset_index(drop=True)
for j, tech in enumerate(TECHS[:12]):
    for kind, sel in (("false_positive", mask_te[:, j] & (Y_TE[:, j] == 0) & (pred_te[:, j] == 1)),
                      ("false_negative", mask_te[:, j] & (Y_TE[:, j] == 1) & (pred_te[:, j] == 0))):
        idxs = np.where(sel)[0][:2]
        for i in idxs:
            ex_rows.append({"technology": tech, "error": kind,
                            "domain": test_df.loc[i, "domain"],
                            "probability": round(float(p_adv_te[i, j]), 3),
                            "scripts": int(test_df.loc[i, "n_scripts"]),
                            "elements": int(test_df.loc[i, "n_elements"]),
                            "ext_domains": int(test_df.loc[i, "n_external_domains"])})
ERROR_EXAMPLES = pd.DataFrame(ex_rows)
ERROR_EXAMPLES.to_csv(PATHS["reports"] / "error_examples.csv", index=False)
display(ERROR_EXAMPLES.head(16))

,technology,category,tp,fp,fn,tn,fp_rate,fn_rate,n_active_rules,dominant_rule_share,likely_cause
7,Django,backend,2,6,33,14411,0.0004,0.9429,3,1.342,single-rule label: signature removal leaves no...
25,PayPal,payments,5,24,74,14370,0.0017,0.9367,2,1.000,single-rule label: signature removal leaves no...
20,Netlify,hosting,4,17,29,14423,0.0012,0.8788,2,1.000,single-rule label: signature removal leaves no...
17,Laravel,backend,14,42,60,13585,0.0031,0.8108,3,7.165,single-rule label: signature removal leaves no...
15,HubSpot,marketing,28,75,90,14276,0.0052,0.7627,2,1.000,single-rule label: signature removal leaves no...
1,Akamai,CDN,31,57,78,14278,0.0040,0.7156,2,1.000,single-rule label: signature removal leaves no...
19,Microsoft IIS,server,108,174,228,13963,0.0123,0.6786,1,1.000,single-rule label: signature removal leaves no...
29,Stripe,payments,43,142,77,14211,0.0099,0.6417,2,1.000,single-rule label: signature removal leaves no...
10,Fastly,CDN,14,37,21,14270,0.0026,0.6000,3,4.739,single-rule label: signature removal leaves no...
0,ASP.NET,backend,211,176,307,13772,0.0126,0.5927,5,0.679,header-derived label with no client-side corre...


Technology confusion (false positives that land on pages using a related technology):


,predicted,actually_present,fp_count,share_of_fps,base_rate,lift
18,React,Nuxt.js,6,0.833,0.0084,98.80
5,HubSpot,Webflow,75,0.333,0.0037,89.34
2,Express,Angular,52,0.327,0.0042,77.79
19,React,Vue.js,6,0.833,0.0124,67.21
16,React,Express,6,0.500,0.0077,64.61
22,Shopify,Express,5,0.400,0.0077,51.69
12,Next.js,Nuxt.js,34,0.294,0.0084,34.87
14,Next.js,Vue.js,34,0.294,0.0124,23.72
25,Vercel,Next.js,159,0.849,0.0373,22.79
20,Shopify,Amazon CloudFront,5,0.400,0.0301,13.27


,technology,error,domain,probability,scripts,elements,ext_domains
0,ASP.NET,false_positive,128idc.com,0.955,17,277,1
1,ASP.NET,false_positive,banguonm.co.kr,0.979,10,188,0
2,ASP.NET,false_negative,achairslife.com,0.547,1,36,0
3,ASP.NET,false_negative,3677u.com,0.900,15,267,3
4,Akamai,false_positive,axel.market,0.998,3,249,0
5,Akamai,false_positive,centrepompidou.fr,0.938,282,797,2
6,Akamai,false_negative,iveco.com,0.837,8,687,0
7,Akamai,false_negative,caracol.com.co,0.749,20,645,8
8,Amazon CloudFront,false_positive,lalame.net,0.946,57,722,9
9,Amazon CloudFront,false_positive,majordalim.com,0.956,12,625,2


## 25. Technology co-occurrence

Which technologies travel together? Co-occurrence is computed on **observed trusted labels**, using three
complementary statistics because each answers a different question:

* **Jaccard** — overlap as a fraction of the union. Symmetric, penalises prevalence mismatch.
* **Conditional probability** P(B|A) — asymmetric. "Given Next.js, how often Cloudflare?" is not the same question
  as the reverse, and the asymmetry is the interesting part.
* **Lift** — P(A,B) / (P(A)P(B)). Corrects for base rates, so a pairing with ubiquitous Nginx does not look
  meaningful just because Nginx is everywhere.

Only pairs where both labels are trusted on the same page contribute, so the masking discipline carries through.
Note that implied labels (S10) were recorded at MEDIUM confidence and are therefore *absent* from this analysis —
the React/Next.js relationship measured here is the directly observed one, not one manufactured by implication.

In [44]:
# --- co-occurrence --------------------------------------------------------
def cooccurrence(y: np.ndarray, m: np.ndarray, labels: Sequence[str]):
    k = len(labels)
    jac = np.zeros((k, k)); cond = np.zeros((k, k)); lift = np.ones((k, k))
    pos = (y == 1) & (m == 1)
    for a in range(k):
        for b in range(k):
            both_trusted = m[:, a].astype(bool) & m[:, b].astype(bool)
            n = both_trusted.sum()
            if n == 0:
                continue
            A = pos[both_trusted, a]; B = pos[both_trusted, b]
            inter = float((A & B).sum())
            union = float((A | B).sum())
            jac[a, b] = inter / union if union else 0.0
            cond[a, b] = inter / A.sum() if A.sum() else 0.0
            pa, pb = A.mean(), B.mean()
            lift[a, b] = (inter / n) / (pa * pb) if pa > 0 and pb > 0 else 0.0
    idx = list(labels)
    return (pd.DataFrame(jac, index=idx, columns=idx),
            pd.DataFrame(cond, index=idx, columns=idx),
            pd.DataFrame(lift, index=idx, columns=idx))

JACCARD, CONDITIONAL, LIFT = cooccurrence(Y, M, TECHS)
JACCARD.round(4).to_csv(PATHS["reports"] / "technology_cooccurrence.csv")
CONDITIONAL.round(4).to_csv(PATHS["reports"] / "technology_cooccurrence_conditional.csv")

pairs = []
for a in range(len(TECHS)):
    for b in range(a + 1, len(TECHS)):
        if JACCARD.iloc[a, b] > 0:
            pairs.append({"tech_a": TECHS[a], "tech_b": TECHS[b],
                          "jaccard": round(float(JACCARD.iloc[a, b]), 4),
                          "p_b_given_a": round(float(CONDITIONAL.iloc[a, b]), 4),
                          "p_a_given_b": round(float(CONDITIONAL.iloc[b, a]), 4),
                          "lift": round(float(LIFT.iloc[a, b]), 2)})
PAIRS = pd.DataFrame(pairs)
TOP_PAIRS = PAIRS.sort_values("jaccard", ascending=False).head(25) if len(PAIRS) else PAIRS
TOP_LIFT = PAIRS[PAIRS["jaccard"] > 0.01].sort_values("lift", ascending=False).head(25) if len(PAIRS) else PAIRS
RESULTS["cooccurrence_top_jaccard"] = TOP_PAIRS.to_dict("records")
RESULTS["cooccurrence_top_lift"] = TOP_LIFT.to_dict("records")

print("Strongest technology pairs by Jaccard overlap:")
display(TOP_PAIRS.head(15))
print("Strongest by lift (base-rate corrected - the genuinely surprising pairings):")
display(TOP_LIFT.head(15))

if CONFIG["MAKE_FIGURES"] and len(TECHS) > 2:
    top_t = LABEL_QUALITY[LABEL_QUALITY["technology"].isin(TECHS)].head(22)["technology"].tolist()
    sub = JACCARD.loc[top_t, top_t]
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(sub.values, cmap="viridis", vmin=0, vmax=min(1.0, float(np.nanmax(sub.values)) or 1.0))
    ax.set_xticks(range(len(top_t))); ax.set_xticklabels(top_t, rotation=90, fontsize=7)
    ax.set_yticks(range(len(top_t))); ax.set_yticklabels(top_t, fontsize=7)
    fig.colorbar(im, ax=ax, label="Jaccard")
    ax.set_title("Technology co-occurrence (trusted labels)")
    savefig(fig, "10_cooccurrence_matrix")

if HAVE_NETWORKX and CONFIG["MAKE_FIGURES"] and len(PAIRS):
    G = nx.Graph()
    for t in TECHS:
        G.add_node(t, category=TECH_CATEGORY[t])
    for _, r in PAIRS[PAIRS["jaccard"] >= max(0.05, PAIRS["jaccard"].quantile(0.92))].iterrows():
        G.add_edge(r["tech_a"], r["tech_b"], weight=float(r["jaccard"]))
    G.remove_nodes_from([n for n in list(G.nodes) if G.degree(n) == 0])
    if len(G):
        cats = sorted({TECH_CATEGORY[n] for n in G.nodes})
        cmap = {c: plt.cm.tab20(i / max(len(cats) - 1, 1)) for i, c in enumerate(cats)}
        pos_layout = nx.spring_layout(G, seed=CONFIG["RANDOM_SEED"], k=0.7, weight="weight")
        fig, ax = plt.subplots(figsize=(12, 9))
        nx.draw_networkx_edges(G, pos_layout, ax=ax, alpha=0.35,
                               width=[2.5 * G[u][v]["weight"] for u, v in G.edges])
        nx.draw_networkx_nodes(G, pos_layout, ax=ax, node_size=420,
                               node_color=[cmap[TECH_CATEGORY[n]] for n in G.nodes])
        nx.draw_networkx_labels(G, pos_layout, ax=ax, font_size=7)
        ax.set_title("Technology ecosystem graph (edges = strong co-occurrence)")
        ax.axis("off")
        savefig(fig, "11_cooccurrence_network")
        RESULTS["ecosystem_graph"] = {"nodes": G.number_of_nodes(), "edges": G.number_of_edges()}

Strongest technology pairs by Jaccard overlap:


,tech_a,tech_b,jaccard,p_b_given_a,p_a_given_b,lift
508,Nuxt.js,Vue.js,0.5370,1.0000,0.5370,83.41
15,ASP.NET,Microsoft IIS,0.4950,0.5601,0.8097,22.84
362,Google Fonts,WordPress,0.4269,0.6302,0.5696,1.49
476,Next.js,React,0.4081,1.0000,0.4081,248.81
312,Font Awesome,jQuery,0.3679,0.7250,0.4275,1.93
288,Font Awesome,Google Fonts,0.3644,0.5864,0.4905,1.55
311,Font Awesome,WordPress,0.3586,0.6144,0.4628,1.47
152,Bootstrap,Font Awesome,0.2971,0.5430,0.3961,1.73
363,Google Fonts,jQuery,0.2946,0.5172,0.4063,1.40
178,Bootstrap,jQuery,0.2878,0.5616,0.3712,1.49


Strongest by lift (base-rate corrected - the genuinely surprising pairings):


,tech_a,tech_b,jaccard,p_b_given_a,p_a_given_b,lift
476,Next.js,React,0.4081,1.0000,0.4081,248.81
508,Nuxt.js,Vue.js,0.5370,1.0000,0.5370,83.41
101,Angular,Express,0.1853,0.4874,0.2302,61.27
15,ASP.NET,Microsoft IIS,0.4950,0.5601,0.8097,22.84
481,Next.js,Vercel,0.2478,0.2652,0.7907,21.59
404,HubSpot,Webflow,0.0495,0.0691,0.1481,18.99
531,PayPal,Stripe,0.0302,0.0789,0.0465,9.84
264,Express,React,0.0255,0.0385,0.0703,9.57
462,Netlify,Nuxt.js,0.0146,0.0615,0.0188,7.42
574,Tailwind CSS,Vercel,0.0829,0.0859,0.7057,6.99


14:22:34 | INFO    | figure saved: 10_cooccurrence_matrix.png
14:22:34 | INFO    | figure saved: 11_cooccurrence_network.png


## 26. Technology stack clustering

Pages are clustered on their observed technology vectors to find recurring **stack archetypes**. Two views:

* **K-means** over binary technology vectors, with k chosen by silhouette over a small sweep.
* **Hierarchical clustering** of the *technologies themselves* (on co-occurrence distance), which reveals
  ecosystem families independently of how common each stack is.

Clusters are named automatically from their most over-represented technologies (lift against the global rate), so
the labels describe what actually distinguishes each cluster rather than what is merely common inside it.

In [45]:
# --- stack clustering -----------------------------------------------------
STACK_MATRIX = ((Y == 1) & (M == 1)).astype(np.float32)
rows_with_tech = np.where(STACK_MATRIX.sum(axis=1) > 0)[0]
CLUSTER_TABLE = pd.DataFrame()
if len(rows_with_tech) >= 50 and len(TECHS) >= 3:
    Xc = STACK_MATRIX[rows_with_tech]
    sample = Xc if len(Xc) <= 40000 else Xc[np.random.default_rng(CONFIG["RANDOM_SEED"])
                                            .choice(len(Xc), 40000, replace=False)]
    best_k, best_score, best_km = None, -1.0, None
    from sklearn.metrics import silhouette_score
    for k in [c for c in (4, 6, 8, 10, 12) if c < len(sample)]:
        km = KMeans(n_clusters=k, n_init=4, random_state=CONFIG["RANDOM_SEED"]).fit(sample)
        try:
            sub = sample if len(sample) <= 5000 else sample[:5000]
            score = silhouette_score(sub, km.predict(sub))
        except Exception:
            score = -1.0
        LOG.info("k=%d silhouette=%.4f", k, score)
        if score > best_score:
            best_k, best_score, best_km = k, score, km
    labels_c = best_km.predict(Xc)
    global_rate = STACK_MATRIX.mean(axis=0)
    rows = []
    for c in range(best_k):
        sel = labels_c == c
        rate = Xc[sel].mean(axis=0)
        lift = rate / np.maximum(global_rate, 1e-6)
        top = np.argsort(-(lift * (rate > 0.15)))[:4]
        name = " + ".join(TECHS[i] for i in top if rate[i] > 0.15) or "sparse / low-evidence"
        rows.append({"cluster": c, "pages": int(sel.sum()),
                     "share": round(float(sel.mean()), 4), "archetype": name,
                     "mean_stack_size": round(float(Xc[sel].sum(axis=1).mean()), 2),
                     "top_technologies": ", ".join(
                         f"{TECHS[i]} ({rate[i]:.0%})" for i in np.argsort(-rate)[:6] if rate[i] > 0.05)})
    CLUSTER_TABLE = pd.DataFrame(rows).sort_values("pages", ascending=False)
    CLUSTER_TABLE.to_csv(PATHS["reports"] / "stack_clusters.csv", index=False)
    RESULTS["stack_clusters"] = {"k": best_k, "silhouette": round(float(best_score), 4),
                                 "clusters": CLUSTER_TABLE.to_dict("records")}
    print(f"k={best_k} selected by silhouette ({best_score:.3f})")
    display(CLUSTER_TABLE)
else:
    LOG.warning("not enough labelled pages for stack clustering")

# hierarchical view over technologies
if len(TECHS) >= 4 and CONFIG["MAKE_FIGURES"]:
    dist = 1.0 - JACCARD.loc[TECHS, TECHS].values
    np.fill_diagonal(dist, 0.0)
    dist = np.clip((dist + dist.T) / 2, 0, 1)
    hc = AgglomerativeClustering(n_clusters=min(6, len(TECHS) - 1), metric="precomputed",
                                 linkage="average").fit(dist)
    fam = pd.DataFrame({"technology": TECHS, "family": hc.labels_,
                        "category": [TECH_CATEGORY[t] for t in TECHS]}).sort_values("family")
    fam.to_csv(PATHS["reports"] / "technology_families.csv", index=False)
    RESULTS["technology_families"] = fam.to_dict("records")
    print("Hierarchical technology families (from co-occurrence distance):")
    display(fam)

14:22:35 | INFO    | k=4 silhouette=0.1286
14:22:35 | INFO    | k=6 silhouette=0.1280
14:22:36 | INFO    | k=8 silhouette=0.1359
14:22:37 | INFO    | k=10 silhouette=0.1365
14:22:38 | INFO    | k=12 silhouette=0.1454


k=12 selected by silhouette (0.145)


,cluster,pages,share,archetype,mean_stack_size,top_technologies
1,1,13609,0.1450,Nginx + jQuery + PHP + Google Analytics,2.45,"Nginx (100%), PHP (25%), Google Analytics (22%..."
2,2,9836,0.1048,Cloudflare + WordPress + Google Analytics + Go...,3.64,"WordPress (100%), Cloudflare (74%), Google Fon..."
9,9,9813,0.1046,Apache + WooCommerce + PHP + WordPress,4.72,"WordPress (95%), Apache (83%), Google Fonts (7..."
4,4,9685,0.1032,Wix + ASP.NET + Bootstrap + jQuery,1.92,"Bootstrap (31%), Google Analytics (19%), jQuer..."
10,10,8524,0.0908,Apache + WordPress + PHP + Google Analytics,2.28,"Apache (100%), WordPress (27%), PHP (16%), Goo..."
6,6,8088,0.0862,Cloudflare + Tailwind CSS + Google Analytics +...,2.55,"Cloudflare (100%), Google Fonts (25%), Google ..."
3,3,7440,0.0793,Nginx + WooCommerce + Font Awesome + Google Fonts,4.60,"WordPress (94%), Nginx (92%), Google Fonts (83..."
7,7,7100,0.0757,LiteSpeed + WooCommerce + WordPress + PHP,4.52,"LiteSpeed (100%), WordPress (93%), Google Font..."
0,0,5796,0.0618,ASP.NET + Bootstrap + Font Awesome + jQuery,4.80,"Font Awesome (86%), Bootstrap (73%), Google Fo..."
5,5,5078,0.0541,jQuery + Apache + PHP + Bootstrap,4.38,"jQuery (90%), Apache (81%), PHP (71%), Bootstr..."


Hierarchical technology families (from co-occurrence distance):


,technology,family,category
0,ASP.NET,0,backend
1,Akamai,0,CDN
2,Amazon CloudFront,0,CDN
3,Angular,0,frontend
4,Apache,0,server
5,Bootstrap,0,CSS/UI
6,Cloudflare,0,CDN
8,Drupal,0,CMS
14,Google Tag Manager,0,analytics
9,Express,0,backend


## 27. Temporal generalization

A second, harder generalisation test. The domain-disjoint split asks "does this work on sites you have not seen?"
The temporal split asks "does this work on **a web you have not seen**?"

```text
oldest crawls   -> TRAIN
middle crawl    -> VALIDATION
newest crawl    -> TEST
```

Each temporal crawl is ingested with the same pipeline at `TEMPORAL_PAGES_PER_CRAWL` scale. Domain disjointness is
*additionally* enforced across the temporal splits, so a site captured in both 2021 and 2025 cannot leak — without
that guard the experiment would silently measure domain memorisation again, and would look far better than it is.

The comparison to make is `temporal micro-F1` against `domain-disjoint micro-F1` on the same model family. A gap
between them is **technology drift**: fingerprints that changed, frameworks that did not exist in the training
period, and vendors that altered their emitted markers.

In [46]:
# --- temporal corpora -----------------------------------------------------
TEMPORAL_OK = False
TEMPORAL_FRAMES: Dict[str, pd.DataFrame] = {}

if CONFIG["ENABLE_TEMPORAL_EVALUATION"] and len(TEMPORAL_CRAWLS) >= 3:
    for cid in TEMPORAL_CRAWLS:
        try:
            d = run_ingestion(cid, int(CONFIG["TEMPORAL_PAGES_PER_CRAWL"]), tag="temporal")
            df_t = materialise_json(load_evidence(d))
            df_t["crawl_year"] = crawl_year(cid)
            TEMPORAL_FRAMES[cid] = df_t
            LOG.info("temporal corpus %s: %d pages / %d domains", cid, len(df_t), df_t["domain"].nunique())
        except Exception as exc:
            LOG.warning("temporal ingestion failed for %s: %s", cid, exc)
    TEMPORAL_OK = len(TEMPORAL_FRAMES) >= 3
else:
    LOG.warning("temporal evaluation disabled or too few crawls available")

if TEMPORAL_OK:
    T_ALL = pd.concat(TEMPORAL_FRAMES.values(), ignore_index=True)
    T_ALL, t_dedup = deduplicate(T_ALL)
    T_ALL = T_ALL.reset_index(drop=True)
    RESULTS["temporal_dedup"] = t_dedup
    print(f"temporal corpus: {len(T_ALL):,} pages, {T_ALL['domain'].nunique():,} domains, "
          f"crawls {sorted(T_ALL['crawl_id'].unique())}")
    display(T_ALL.groupby(["crawl_id", "crawl_year"]).size().rename("pages").reset_index())

14:22:39 | INFO    | CC-MAIN-2013-48: 51900 warc files listed
14:22:39 | INFO    | [CC-MAIN-2013-48] 12/12 archive files remaining (target 8000 pages, 0 already cached)
14:23:46 | INFO    | [CC-MAIN-2013-48] CC-MAIN-20131204131726-00067-ip-10-33-133-15.ec2.internal.warc.gz -> +2054 rows (total kept 8001, seen 18842, 67s)
14:23:47 | INFO    | [CC-MAIN-2013-48] ingestion complete: 8003 new rows, 8003 total, 4873 domains, 1.1 min
14:23:47 | INFO    | [mem] post-ingest rss=5.94 GB
14:23:48 | INFO    | loaded 8003 evidence rows from 4 shards
14:23:49 | INFO    | temporal corpus CC-MAIN-2013-48: 8003 pages / 4873 domains
14:23:49 | INFO    | CC-MAIN-2017-51: 80000 warc files listed
14:23:49 | INFO    | [CC-MAIN-2017-51] 12/12 archive files remaining (target 8000 pages, 0 already cached)
14:24:52 | INFO    | [CC-MAIN-2017-51] CC-MAIN-20171212060515-20171212080515-00575.warc.gz -> +2012 rows (total kept 8000, seen 13016, 63s)
14:24:53 | INFO    | [CC-MAIN-2017-51] ingestion complete: 8002 new 

simhash:   0%|          | 0/31798 [00:00<?, ?it/s]

temporal corpus: 30,588 pages, 24,335 domains, crawls ['CC-MAIN-2013-48', 'CC-MAIN-2017-51', 'CC-MAIN-2022-49', 'CC-MAIN-2026-30']


,crawl_id,crawl_year,pages
0,CC-MAIN-2013-48,2013,7632
1,CC-MAIN-2017-51,2017,7711
2,CC-MAIN-2022-49,2022,7661
3,CC-MAIN-2026-30,2026,7584


In [47]:
# --- temporal split, training and evaluation ------------------------------
TEMPORAL_SUMMARY = pd.DataFrame()
if TEMPORAL_OK:
    conf_t, hits_t = label_corpus(T_ALL, CONFIG["APPLY_IMPLICATIONS"])
    # Same treatment as the main corpus: rule matching is done, so the dict-heavy evidence
    # columns can go. tag_counts is retained here because the temporal frame has no entry in
    # STRUCT_CACHE and struct_matrix must fall back to computing from it.
    T_ALL, _ = precompute_documents(T_ALL)
    _old_t = T_ALL
    T_ALL = slim_frame(T_ALL, keep_extra=("tag_counts_json",))
    del _old_t
    gc.collect()
    mem_report("post-temporal-slim")
    y_t_all, m_t_all, techs_t_all = build_label_matrices(
        conf_t, CONFIG["POSITIVE_CONFIDENCE"], CONFIG["MASK_UNCERTAIN"])
    cols_t = [techs_t_all.index(t) for t in TECHS]
    Yt, Mt = y_t_all[:, cols_t], m_t_all[:, cols_t]

    years = sorted(T_ALL["crawl_year"].dropna().unique())
    newest, middle = years[-1], years[-2]
    role = np.where(T_ALL["crawl_year"].values == newest, "test",
                    np.where(T_ALL["crawl_year"].values == middle, "val", "train"))
    # Enforce domain disjointness ACROSS TIME, otherwise this silently becomes another
    # memorisation test. Overlapping domains are removed from the EARLIER split, never the
    # later one: evaluation data is the scarce resource here, and dropping test rows to
    # protect training rows would shrink the very thing being measured.
    test_doms_t = set(T_ALL.loc[role == "test", "domain"])
    val_doms_t = set(T_ALL.loc[role == "val", "domain"]) - test_doms_t
    keep_rows = np.array([
        (r == "test") or
        (r == "val" and d in val_doms_t) or
        (r == "train" and d not in test_doms_t and d not in val_doms_t)
        for r, d in zip(role, T_ALL["domain"])])
    leaked = int((~keep_rows).sum())

    T_IDX = {s: np.where(keep_rows & (role == s))[0] for s in ("train", "val", "test")}
    print(f"temporal split (train<= {middle - 1}, val={middle}, test={newest}); "
          f"{leaked:,} rows dropped to keep domains disjoint across time")
    for s in ("train", "val", "test"):
        print(f"  {s:<6} {len(T_IDX[s]):>7,} pages  {T_ALL.iloc[T_IDX[s]]['domain'].nunique():>6,} domains")

    if min(len(v) for v in T_IDX.values()) < 50:
        LOG.warning("a temporal split is too small to evaluate")
        TEMPORAL_OK = False

if TEMPORAL_OK:
    fs_t = FeatureSpace(PRIMARY_REGIME, CONFIG).fit(T_ALL.iloc[T_IDX["train"]], hits_t[T_IDX["train"]])
    Xt = {s: fs_t.transform(T_ALL.iloc[T_IDX[s]], hits_t[T_IDX[s]]) for s in ("train", "val", "test")}
    clf_t = MaskedOvR(logreg_factory, TECHS, "temporal", max_rows=120_000, seed=CONFIG["RANDOM_SEED"])
    clf_t.fit(Xt["train"], Yt[T_IDX["train"]], Mt[T_IDX["train"]])
    pv_t = clf_t.predict_proba(Xt["val"])
    thr_t = tune_thresholds(Yt[T_IDX["val"]], pv_t, Mt[T_IDX["val"]])
    pt_t = clf_t.predict_proba(Xt["test"])
    temporal_res = masked_metrics(Yt[T_IDX["test"]], pt_t, Mt[T_IDX["test"]], thr_t, TECHS)

    # Same model family, same regime, but the domain-disjoint split -> a fair contrast.
    _, Xb = get_regime(PRIMARY_REGIME)
    pb_va = ABLATION_MODELS[PRIMARY_REGIME].predict_proba(Xb["val"])
    thr_b = tune_thresholds(Y_VA, pb_va, M_VA)
    domain_res = masked_metrics(Y_TE, ABLATION_MODELS[PRIMARY_REGIME].predict_proba(Xb["test"]),
                                M_TE, thr_b, TECHS)

    TEMPORAL_SUMMARY = pd.DataFrame([
        {**summarise("domain-disjoint (same crawl)", domain_res), "protocol": "domain split"},
        {**summarise(f"temporal (train<={middle - 1} -> test {newest})", temporal_res),
         "protocol": "temporal split"},
    ])
    TEMPORAL_SUMMARY.to_csv(PATHS["reports"] / "temporal_results.csv", index=False)
    drift = domain_res["micro_f1"] - temporal_res["micro_f1"]
    RESULTS["temporal"] = {
        "table": TEMPORAL_SUMMARY.to_dict("records"),
        "micro_f1_drift": round(float(drift), 4),
        "relative_drift_pct": round(float(100 * drift / max(domain_res["micro_f1"], 1e-9)), 1),
        "train_years": [int(y) for y in years[:-2]], "val_year": int(middle), "test_year": int(newest),
        "rows_dropped_for_disjointness": leaked,
    }
    print(f"\nmicro-F1 drift domain-disjoint -> temporal: {drift:+.4f} "
          f"({RESULTS['temporal']['relative_drift_pct']:+.1f}%)")
    display(TEMPORAL_SUMMARY)

    per_t = temporal_res["per_technology"].set_index("technology")["f1"]
    per_d = domain_res["per_technology"].set_index("technology")["f1"]
    DRIFT_TBL = pd.DataFrame({"domain_f1": per_d, "temporal_f1": per_t})
    DRIFT_TBL["drift"] = (DRIFT_TBL["domain_f1"] - DRIFT_TBL["temporal_f1"]).round(4)
    DRIFT_TBL = DRIFT_TBL.dropna().sort_values("drift", ascending=False)
    RESULTS["temporal_per_technology"] = DRIFT_TBL.reset_index().to_dict("records")
    print("Largest per-technology drift (positive = worse across time):")
    display(DRIFT_TBL.head(12))

labelling:   0%|          | 0/30588 [00:00<?, ?it/s]

14:30:11 | INFO    | precompute: 20000/30588 rows (82s)
14:30:54 | INFO    | precompute complete: 4 document variants + struct (30588, 78) (126s)
14:30:54 | INFO    | released 25 evidence columns (39 retained)
14:30:55 | INFO    | [mem] post-temporal-slim rss=6.55 GB


temporal split (train<= 2021, val=2022, test=2026); 993 rows dropped to keep domains disjoint across time
  train   14,606 pages  10,589 domains
  val      7,405 pages   6,717 domains
  test     7,584 pages   7,029 domains


14:31:59 | INFO    | [temporal] fitted 5/39 labels (31s elapsed)
14:32:25 | INFO    | [temporal] fitted 10/39 labels (57s elapsed)
14:33:17 | INFO    | [temporal] fitted 20/39 labels (108s elapsed)
14:33:44 | INFO    | [temporal] fitted 25/39 labels (136s elapsed)
14:34:01 | INFO    | [temporal] fitted 30/39 labels (153s elapsed)
14:34:21 | INFO    | [temporal] fitted 39/39 labels (173s elapsed)



micro-F1 drift domain-disjoint -> temporal: +0.1667 (+22.6%)


,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,macro_precision,macro_recall,macro_pr_auc,micro_pr_auc,hamming_loss,subset_accuracy,protocol
0,domain-disjoint (same crawl),0.7362,0.6215,0.7489,0.6910,0.7878,0.6304,0.6304,0.6322,0.8262,0.0499,0.2443,domain split
1,temporal (train<=2021 -> test 2026),0.5695,0.2442,0.5903,0.4714,0.7192,0.2154,0.3141,0.2315,0.5644,0.0808,0.1291,temporal split


Largest per-technology drift (positive = worse across time):


,domain_f1,temporal_f1,drift
technology,,,
Wix,0.981450,0.000000,0.9815
Google Tag Manager,0.966887,0.000000,0.9669
Tailwind CSS,0.891922,0.000000,0.8919
Shopify,0.887500,0.000000,0.8875
Next.js,0.875000,0.000000,0.8750
Webflow,0.859813,0.000000,0.8598
Angular,0.724409,0.000000,0.7244
Ruby on Rails,0.774194,0.202703,0.5715
Cloudflare,0.813415,0.251526,0.5619


## 28. Technology adoption over time

Prevalence of each technology per crawl year, measured on the temporal corpora.

**This is a sample statistic, not a census.** Common Crawl's frontier changes between crawls, its seed and
host-selection policies have shifted over the years, and the per-domain cap reshapes the mix further. A rise in
measured prevalence can reflect genuine adoption *or* a change in what the crawler chose to fetch. Confidence
intervals below are binomial on the sample and account only for sampling noise — not for the far larger
composition effect. Treat the direction of large, consistent multi-year trends as informative and treat everything
else as noise.

In [48]:
# --- adoption trends ------------------------------------------------------
ADOPTION = pd.DataFrame()
if TEMPORAL_OK:
    rows = []
    for year in sorted(T_ALL["crawl_year"].dropna().unique()):
        sel = (T_ALL["crawl_year"].values == year)
        n = int(sel.sum())
        for j, tech in enumerate(TECHS):
            trusted = int(Mt[sel, j].sum())
            pos = int(((Yt[sel, j] == 1) & (Mt[sel, j] == 1)).sum())
            p = pos / max(trusted, 1)
            se = math.sqrt(max(p * (1 - p), 0) / max(trusted, 1))
            rows.append({"year": int(year), "technology": tech, "category": TECH_CATEGORY[tech],
                         "pages": n, "trusted": trusted, "positives": pos,
                         "prevalence": round(p, 5), "ci95": round(1.96 * se, 5)})
    ADOPTION = pd.DataFrame(rows)
    ADOPTION.to_csv(PATHS["reports"] / "technology_adoption.csv", index=False)

    pivot = ADOPTION.pivot(index="year", columns="technology", values="prevalence").fillna(0)
    years_sorted = sorted(pivot.index)
    if len(years_sorted) >= 2:
        first, last = years_sorted[0], years_sorted[-1]
        change = pd.DataFrame({
            "prevalence_first": pivot.loc[first], "prevalence_last": pivot.loc[last]})
        change["abs_change"] = (change["prevalence_last"] - change["prevalence_first"]).round(5)
        change["rel_change_pct"] = (100 * change["abs_change"] /
                                    change["prevalence_first"].replace(0, np.nan)).round(1)
        change = change.sort_values("abs_change", ascending=False)
        RESULTS["adoption"] = {"first_year": int(first), "last_year": int(last),
                               "rising": change.head(8).reset_index().to_dict("records"),
                               "declining": change.tail(8).reset_index().to_dict("records")}
        print(f"Prevalence change {first} -> {last} (this Common Crawl-derived sample only):")
        display(change.head(10)); display(change.tail(10))

    if CONFIG["MAKE_FIGURES"] and len(pivot) >= 2:
        interesting = change.reindex(change["abs_change"].abs().sort_values(ascending=False).index).head(10).index
        fig, ax = plt.subplots(figsize=(11, 6))
        for tech in interesting:
            ax.plot(pivot.index, pivot[tech], marker="o", label=tech)
        ax.set_xlabel("crawl year"); ax.set_ylabel("prevalence in sample")
        ax.set_title("Technology adoption over time (Common Crawl sample, not a census)")
        ax.legend(fontsize=7, ncol=2)
        savefig(fig, "12_adoption_trends")
else:
    print("temporal corpora unavailable - adoption analysis skipped")

Prevalence change 2013 -> 2026 (this Common Crawl-derived sample only):


,prevalence_first,prevalence_last,abs_change,rel_change_pct
technology,,,,
Font Awesome,0.01364,0.24629,0.23265,1705.6
Google Fonts,0.09591,0.29294,0.19703,205.4
Bootstrap,0.02107,0.17329,0.15222,722.4
Nginx,0.16012,0.29589,0.13577,84.8
WordPress,0.21730,0.33118,0.11388,52.4
Google Analytics,0.05566,0.16404,0.10838,194.7
Cloudflare,0.01769,0.10897,0.09128,516.0
LiteSpeed,0.01166,0.07720,0.06554,562.1
Tailwind CSS,0.00000,0.05182,0.05182,NaN


,prevalence_first,prevalence_last,abs_change,rel_change_pct
technology,,,,
Webflow,0.00000,0.00053,0.00053,NaN
Wix,0.00000,0.00040,0.00040,NaN
Django,0.00144,0.00145,0.00001,0.7
PayPal,0.00996,0.00409,-0.00587,-58.9
Drupal,0.01949,0.01242,-0.00707,-36.3
Ruby on Rails,0.02022,0.00634,-0.01388,-68.6
PHP,0.37198,0.33443,-0.03755,-10.1
ASP.NET,0.14366,0.05598,-0.08768,-61.0
Microsoft IIS,0.14492,0.04338,-0.10154,-70.1


14:34:26 | INFO    | figure saved: 12_adoption_trends.png


## 29. Technology transition analysis

For domains captured in more than one crawl year, how often does a stack change? Transitions are counted only for
`(domain, technology)` pairs where **both** observations are trusted, which is the key defence against the failure
mode that makes this analysis easy to get wrong.

The confounds are severe and are reported alongside the counts rather than in a footnote:

| Confound | Effect | Mitigation here |
|---|---|---|
| Different page sampled per crawl | A site's homepage in 2021 vs. a deep article in 2025 have different evidence | Aggregate to domain level with an "any trusted page" rule |
| Evidence simply not captured | Looks like removal | Only trusted->trusted pairs counted; abstentions excluded |
| Registry drift | A rule added later manufactures adoption | One registry version applied to every crawl |
| Small overlap | Wild ratios from tiny denominators | Minimum-support gate, and the raw overlap count is printed |

Even with all of that, a "transition" here means *observable evidence changed*, which is weaker than *the site
migrated*. The distinction is stated in the report.

In [49]:
# --- transitions ----------------------------------------------------------
TRANSITIONS = pd.DataFrame()
if TEMPORAL_OK:
    tdf = T_ALL[["domain", "crawl_year"]].copy()
    for j, tech in enumerate(TECHS):
        tdf[f"y_{j}"] = np.where(Mt[:, j] == 1, Yt[:, j].astype(float), np.nan)
    agg = tdf.groupby(["domain", "crawl_year"]).agg("max").reset_index()
    years_sorted = sorted(agg["crawl_year"].dropna().unique())
    first_y, last_y = years_sorted[0], years_sorted[-1]
    a = agg[agg["crawl_year"] == first_y].set_index("domain")
    b = agg[agg["crawl_year"] == last_y].set_index("domain")
    common = a.index.intersection(b.index)
    RESULTS["transition_overlap_domains"] = int(len(common))
    print(f"domains observed in both {first_y} and {last_y}: {len(common):,}")

    if len(common) >= 30:
        rows = []
        for j, tech in enumerate(TECHS):
            col = f"y_{j}"
            va, vb = a.loc[common, col], b.loc[common, col]
            both = va.notna() & vb.notna()
            n = int(both.sum())
            if n < 15:
                continue
            adopted = int(((va == 0) & (vb == 1) & both).sum())
            dropped = int(((va == 1) & (vb == 0) & both).sum())
            retained = int(((va == 1) & (vb == 1) & both).sum())
            rows.append({"technology": tech, "category": TECH_CATEGORY[tech],
                         "domains_compared": n, "present_first": int(((va == 1) & both).sum()),
                         "retained": retained, "adopted": adopted, "dropped": dropped,
                         "retention_rate": round(retained / max(retained + dropped, 1), 3),
                         "adoption_rate": round(adopted / max(int(((va == 0) & both).sum()), 1), 4)})
        TRANSITIONS = pd.DataFrame(rows).sort_values("adopted", ascending=False)
        TRANSITIONS.to_csv(PATHS["reports"] / "technology_transitions.csv", index=False)
        RESULTS["transitions"] = TRANSITIONS.to_dict("records")
        print(f"Observed evidence transitions {first_y} -> {last_y} "
              f"(evidence change, NOT confirmed migration):")
        display(TRANSITIONS.head(18))

        # directed pairs: dropped A while adopting B on the same domain
        pair_rows = []
        for ja, ta in enumerate(TECHS):
            lost = (a.loc[common, f"y_{ja}"] == 1) & (b.loc[common, f"y_{ja}"] == 0)
            if lost.sum() < 8:
                continue
            for jb, tb in enumerate(TECHS):
                if ja == jb:
                    continue
                gained = (a.loc[common, f"y_{jb}"] == 0) & (b.loc[common, f"y_{jb}"] == 1)
                both = int((lost & gained).sum())
                if both >= 5:
                    pair_rows.append({"from": ta, "to": tb, "domains": both,
                                      "share_of_departures": round(float(both / lost.sum()), 3)})
        PAIR_TRANS = pd.DataFrame(pair_rows).sort_values("domains", ascending=False) \
                     if pair_rows else pd.DataFrame()
        if len(PAIR_TRANS):
            RESULTS["transition_pairs"] = PAIR_TRANS.head(20).to_dict("records")
            print("Directed evidence transitions (A disappears while B appears on the same domain):")
            display(PAIR_TRANS.head(15))
    else:
        print("insufficient cross-crawl domain overlap for transition analysis - "
              "increase TEMPORAL_PAGES_PER_CRAWL to raise the chance of recapturing the same sites")
else:
    print("temporal corpora unavailable - transition analysis skipped")

domains observed in both 2013 and 2026: 72
Observed evidence transitions 2013 -> 2026 (evidence change, NOT confirmed migration):


,technology,category,domains_compared,present_first,retained,adopted,dropped,retention_rate,adoption_rate
5,Bootstrap,CSS/UI,67,1,0,22,1,0.000,0.3333
11,Font Awesome,CSS/UI,68,0,0,16,0,0.000,0.2353
13,Google Fonts,CSS/UI,64,6,1,12,5,0.167,0.2069
22,Nginx,server,72,16,5,11,11,0.312,0.1964
37,jQuery,frontend,47,19,8,10,11,0.421,0.3571
12,Google Analytics,analytics,47,3,2,8,1,0.667,0.1818
36,WordPress,CMS,72,15,9,8,6,0.600,0.1404
24,PHP,backend,65,17,6,5,11,0.353,0.1042
14,Google Tag Manager,analytics,51,0,0,5,0,0.000,0.0980
19,Microsoft IIS,server,72,6,3,5,3,0.500,0.0758


Directed evidence transitions (A disappears while B appears on the same domain):


,from,to,domains,share_of_departures
2,Apache,Nginx,9,0.429
0,Apache,Bootstrap,8,0.381
1,Apache,Font Awesome,7,0.333
3,Nginx,Google Fonts,5,0.455


## 30. Model explainability

Three complementary views:

1. **Global linear weights** (regime B logistic models) — the n-grams and resource tokens that push a technology's
   score up. Because these come from the *redacted* regime, they show what the model found *after* the obvious
   signature was removed, which is far more interesting than the signature itself.
2. **Channel ablation attribution** for the hybrid network — each feature block is zeroed in turn and the change in
   predicted probability measured. This is model-agnostic, needs no extra dependency, and maps cleanly onto the
   three information sources.
3. **Per-prediction evidence categories** — for a single page, which evidence families supported the call.

Per the safety scope, explanations report **evidence categories, never raw strings** extracted from crawled pages.

In [50]:
# --- global explainability ------------------------------------------------
def top_features_linear(clf: MaskedOvR, fs: FeatureSpace, tech: str, k: int = 12) -> pd.DataFrame:
    j = TECHS.index(tech)
    mdl = clf.models_[j]
    if mdl is None or not hasattr(mdl, "coef_"):
        return pd.DataFrame(columns=["feature", "weight", "channel"])
    names: List[str] = []
    if fs.text_vec is not None:
        names += [f"text:{t}" for t in fs.text_vec.feature_names]
    if fs.res_vec is not None:
        names += [f"res:{t}" for t in fs.res_vec.feature_names]
    if fs.scaler is not None:
        names += [f"struct:{c}" for c in STRUCT_COLUMNS]
    if fs.rule_cols_ is not None:
        names += [f"rule:{RULE_IDS[i]}" for i in fs.rule_cols_]
    coef = mdl.coef_.ravel()
    if len(names) != len(coef):
        names = [f"f{i}" for i in range(len(coef))]
    order = np.argsort(-coef)[:k]
    return pd.DataFrame({"feature": [names[i] for i in order],
                         "weight": coef[order].round(4),
                         "channel": [names[i].split(":")[0] for i in order]})

focus = PER_TECH_TEST.dropna(subset=["f1"]).head(6)["technology"].tolist()
EXPLAIN = {}
for tech in focus:
    EXPLAIN[tech] = top_features_linear(ABLATION_MODELS["B_reduced"], ABLATION_FS["B_reduced"], tech)
    print(f"\n--- {tech}: strongest positive features in the FINGERPRINT-REDUCED regime ---")
    display(EXPLAIN[tech])
RESULTS["explainability_linear"] = {t: df.to_dict("records") for t, df in EXPLAIN.items()}


--- Wix: strongest positive features in the FINGERPRINT-REDUCED regime ---


,feature,weight,channel
0,res:l_thunderbolt,6.8571,res
1,res:l_wixui,5.6601,res
2,res:l_rb,5.6518,res
3,res:s_production,4.2075,res
4,res:s_react,4.0997,res
5,res:s_bundle,3.9339,res
6,text:built on,3.5628,text
7,res:l_min,3.2106,res
8,res:l_chunk,2.8713,res
9,struct:tagfrac_div,2.7847,struct



--- WordPress: strongest positive features in the FINGERPRINT-REDUCED regime ---


,feature,weight,channel
0,res:c_menu_item,12.6855,res
1,res:c_menu_item_type_post_type,11.1175,res
2,res:c_menu_item_object_page,10.9847,res
3,res:l_11,9.3884,res
4,struct:n_resource_domains,9.2355,struct
5,res:c_menu_item_type_custom,7.6273,res
6,res:c_menu_item_object_custom,7.5459,res
7,res:c_menu_item_type_taxonomy,6.7885,res
8,res:l_com,6.5185,res
9,res:c_menu_item_has_children,6.2323,res



--- Next.js: strongest positive features in the FINGERPRINT-REDUCED regime ---


...

,feature,weight,channel
0,res:c_w_inline_block,9.9901,res
1,res:s_webflow,9.4909,res
2,res:s_schunk,6.7196,res
3,res:l_webflow,6.6785,res
4,res:l_files,5.4447,res
5,res:l_website,5.3266,res
6,res:l_prod,5.2196,res
7,res:l_shared,4.9358,res
8,struct:tagfrac_div,4.6625,struct
9,res:s_webfont,4.6589,res



--- Joomla: strongest positive features in the FINGERPRINT-REDUCED regime ---


,feature,weight,channel
0,res:l_joomla,12.5980,res
1,res:c_custom,11.3523,res
2,res:c_parent,10.3118,res
3,res:c_moduletable,10.1469,res
4,res:c_deeper,9.3719,res
5,res:l_system,9.0120,res
6,res:l_css,8.2336,res
7,res:c_pathway,7.6454,res
8,res:l_general,7.2431,res
9,res:l_alert,7.1065,res



--- Shopify: strongest positive features in the FINGERPRINT-REDUCED regime ---


,feature,weight,channel
0,res:l_monorail,10.4051,res
1,res:l_edge,10.0673,res
2,res:s_perf,9.9302,res
3,res:p_products,9.8256,res
4,res:s_kit,9.7230,res
5,res:s_standard,9.0362,res
6,res:s_actions,8.9366,res
7,res:l_backwards,7.4873,res
8,res:l_accelerated,7.4873,res
9,res:l_compat,7.3183,res


In [51]:
# --- channel attribution for the hybrid model -----------------------------
def channel_attribution(model, X: np.ndarray, dims: Dict[str, int], n: int = 800) -> pd.DataFrame:
    """Zero each feature block and measure the mean absolute probability shift."""
    rng = np.random.default_rng(CONFIG["RANDOM_SEED"])
    rows = rng.choice(len(X), min(n, len(X)), replace=False)
    Xs = X[rows].copy()
    base = model.predict_proba(Xs)
    bounds, start = {}, 0
    for name in ("semantic", "resource_svd", "struct"):
        d = int(dims[name]); bounds[name] = (start, start + d); start += d
    out = []
    for name, (lo, hi) in bounds.items():
        Xm = Xs.copy(); Xm[:, lo:hi] = 0.0
        p = model.predict_proba(Xm)
        delta = np.abs(p - base)
        for j, tech in enumerate(TECHS):
            out.append({"technology": tech, "channel": name,
                        "mean_abs_prob_shift": round(float(delta[:, j].mean()), 4)})
    df = pd.DataFrame(out)
    piv = df.pivot(index="technology", columns="channel", values="mean_abs_prob_shift")
    piv["dominant_channel"] = piv.idxmax(axis=1)
    return piv.sort_values("semantic", ascending=False)

ATTRIB = channel_attribution(ADV_MODEL, XH["test"], HYBRID_BUNDLE["dims"])
ATTRIB.to_csv(PATHS["reports"] / "channel_attribution.csv")
RESULTS["channel_attribution"] = ATTRIB.reset_index().to_dict("records")
print("Which feature block each technology's prediction actually depends on:")
display(ATTRIB.head(20))

if CONFIG["MAKE_FIGURES"]:
    fig, ax = plt.subplots(figsize=(10, max(5, 0.3 * len(ATTRIB))))
    chans = ["semantic", "resource_svd", "struct"]
    bottom = np.zeros(len(ATTRIB))
    colors = ["#2b6cb0", "#dd9b3c", "#4a5568"]
    for c, col in zip(chans, colors):
        ax.barh(ATTRIB.index, ATTRIB[c], left=bottom, label=c, color=col)
        bottom += ATTRIB[c].to_numpy()
    ax.set_xlabel("mean |probability shift| when the block is zeroed")
    ax.legend(); ax.tick_params(labelsize=7)
    ax.set_title("Channel attribution for the hybrid model")
    savefig(fig, "13_channel_attribution")

Which feature block each technology's prediction actually depends on:


channel,resource_svd,semantic,struct,dominant_channel
technology,,,,
Cloudflare,0.1577,0.0991,0.0933,resource_svd
LiteSpeed,0.1491,0.0957,0.1069,resource_svd
Nginx,0.1121,0.0893,0.0724,resource_svd
Apache,0.1305,0.0852,0.1028,resource_svd
PHP,0.1410,0.0778,0.1024,resource_svd
Google Analytics,0.0840,0.0676,0.2387,struct
Amazon CloudFront,0.0992,0.0600,0.0772,resource_svd
Google Fonts,0.1289,0.0536,0.1962,struct
jQuery,0.1104,0.0489,0.3039,struct


14:34:28 | INFO    | figure saved: 13_channel_attribution.png


## 31. Final model training

Two models are kept, and the distinction between them matters:

* **`frozen`** — trained on TRAIN only. This is the model every test number in section 23 came from. Its
  validation-tuned thresholds never saw the test set, so it is the scientifically valid estimate.
* **`shipped`** — the same architecture and the *same train-fitted feature transforms*, refit on TRAIN + VAL so
  the deployed artefact uses all available supervision. Its test score is also reported, with the caveat attached:
  thresholds were tuned on data this model has now trained on, so its test figure carries a small optimistic bias
  and should not be quoted as the headline result.

Feature transforms are deliberately **not** refit on train+val. Refitting the TF-IDF vocabulary and scaler would
change the input dimensionality and silently invalidate the frozen thresholds and the calibrator.

In [52]:
# --- refit on train + val for deployment ----------------------------------
X_TRVAL = np.vstack([XH["train"], XH["val"]])
Y_TRVAL = np.vstack([Y_TR, Y_VA])
M_TRVAL = np.vstack([M_TR, M_VA])

t0 = time.time()
if HAVE_TORCH:
    FINAL_MODEL = TorchMultiLabelHead(X_TRVAL.shape[1], len(TECHS), CONFIG)
    # early stopping still needs a held-out signal; the val slice is inside training here, so we
    # cap epochs at the epoch count the frozen run selected rather than peeking further.
    best_epoch = max(3, int(np.argmax([h["val_macro_ap"] for h in ADV_MODEL.history])) + 1) \
                 if getattr(ADV_MODEL, "history", None) else CONFIG["MLP_EPOCHS"]
    cfg_final = dict(CONFIG); cfg_final["MLP_EPOCHS"] = best_epoch
    FINAL_MODEL.cfg = cfg_final
    FINAL_MODEL.fit(X_TRVAL, Y_TRVAL, M_TRVAL, XH["val"], Y_VA, M_VA)
else:
    FINAL_MODEL = MaskedOvR(lambda: LogisticRegression(max_iter=300, C=1.0, class_weight="balanced"),
                            TECHS, "final_logreg", seed=CONFIG["RANDOM_SEED"])
    FINAL_MODEL.fit(X_TRVAL, Y_TRVAL, M_TRVAL)
RUN["final_training_seconds"] = round(time.time() - t0, 1)

p_final_te = FINAL_MODEL.predict_proba(XH["test"])
final_res = masked_metrics(Y_TE, p_final_te, M_TE, THRESHOLDS_FROZEN, TECHS)
RESULTS["final_model"] = {
    "frozen_test": summarise("frozen (train only) - HEADLINE", TEST_RESULTS[PRIMARY_KEY]),
    "shipped_test": summarise("shipped (train+val) - mildly optimistic", final_res),
    "training_seconds": RUN["final_training_seconds"],
}
print(f"frozen  model test micro-F1 : {TEST_RESULTS[PRIMARY_KEY]['micro_f1']:.4f}   <- headline result")
print(f"shipped model test micro-F1 : {final_res['micro_f1']:.4f}   (thresholds tuned on data it trained on)")
display(pd.DataFrame([RESULTS["final_model"]["frozen_test"], RESULTS["final_model"]["shipped_test"]]))

14:34:29 | INFO    | epoch 00 | loss 0.6157 | val macro-PR-AUC 0.4991
14:34:31 | INFO    | epoch 01 | loss 0.4644 | val macro-PR-AUC 0.5686
14:34:32 | INFO    | epoch 02 | loss 0.4150 | val macro-PR-AUC 0.6134
14:34:34 | INFO    | epoch 03 | loss 0.3850 | val macro-PR-AUC 0.6301
14:34:36 | INFO    | epoch 04 | loss 0.3644 | val macro-PR-AUC 0.6484
14:34:38 | INFO    | epoch 05 | loss 0.3481 | val macro-PR-AUC 0.6718
14:34:41 | INFO    | epoch 06 | loss 0.3341 | val macro-PR-AUC 0.6840
14:34:42 | INFO    | epoch 07 | loss 0.3223 | val macro-PR-AUC 0.7011
14:34:43 | INFO    | epoch 08 | loss 0.3115 | val macro-PR-AUC 0.7093
14:34:45 | INFO    | epoch 09 | loss 0.3040 | val macro-PR-AUC 0.7264
14:34:46 | INFO    | epoch 10 | loss 0.2952 | val macro-PR-AUC 0.7204
14:34:48 | INFO    | epoch 11 | loss 0.2855 | val macro-PR-AUC 0.7410
14:34:49 | INFO    | epoch 12 | loss 0.2814 | val macro-PR-AUC 0.7588
14:34:50 | INFO    | epoch 13 | loss 0.2758 | val macro-PR-AUC 0.7662
14:34:53 | INFO    |

frozen  model test micro-F1 : 0.7189   <- headline result
shipped model test micro-F1 : 0.7148   (thresholds tuned on data it trained on)


,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,macro_precision,macro_recall,macro_pr_auc,micro_pr_auc,hamming_loss,subset_accuracy
0,frozen (train only) - HEADLINE,0.7189,0.6059,0.7303,0.6672,0.7793,0.6188,0.6145,0.6212,0.7894,0.0539,0.1973
1,shipped (train+val) - mildly optimistic,0.7148,0.6007,0.7289,0.6621,0.7766,0.5797,0.6420,0.6160,0.7668,0.0548,0.2039


## 32. Model saving

Everything needed to reproduce a prediction is serialised: the fitted transforms, the label space, the frozen
thresholds, the isotonic calibrator, the registry version and the full configuration. The bundle is
self-describing, so inference does not depend on any notebook state.

In [53]:
# --- persist artefacts ----------------------------------------------------
MODEL_DIR = PATHS["models"] / "best_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_VERSION = f"{CONFIG['EXPERIMENT_TAG']}-{REGISTRY_VERSION}-{time.strftime('%Y%m%d')}"

def unwrap_filtered_tfidf(ft: Optional[FilteredTfidf]) -> Optional[Dict[str, Any]]:
    """Store the sklearn vectorizer plus its keep-mask instead of the wrapper object.

    The bundle must be loadable in a plain Python process that has never seen this notebook,
    so no notebook-defined class is allowed inside it.
    """
    if ft is None:
        return None
    return {"vectorizer": ft.vec, "keep": ft.keep_, "n_dropped": ft.n_dropped_}

bundle = {
    "model_version": MODEL_VERSION,
    "registry_version": REGISTRY_VERSION,
    "labels": TECHS,
    "categories": {t: TECH_CATEGORY[t] for t in TECHS},
    "thresholds": {t: float(v) for t, v in zip(TECHS, THRESHOLDS_FROZEN)},
    "regime": PRIMARY_REGIME,
    "semantic_mode": HYBRID_BUNDLE["encoder"].mode,
    "dims": HYBRID_BUNDLE["dims"],
    "struct_columns": STRUCT_COLUMNS,
    "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CONFIG.items()},
    "model_kind": "torch_mlp" if HAVE_TORCH else "sklearn_ovr",
    # transforms (the sentence-transformer itself is reloaded by name, not pickled)
    "res_vec": unwrap_filtered_tfidf(HYBRID_BUNDLE["res_vec"]),
    "res_svd": HYBRID_BUNDLE["res_svd"],
    "scaler": HYBRID_BUNDLE["scaler"],
    "semantic_tfidf": unwrap_filtered_tfidf(HYBRID_BUNDLE["encoder"].tfidf),
    "semantic_svd": HYBRID_BUNDLE["encoder"].svd,
    "calibrator_models": CALIBRATOR.models_,   # list of sklearn IsotonicRegression / None
}
joblib.dump(bundle, MODEL_DIR / "feature_bundle.joblib", compress=3)

if HAVE_TORCH:
    torch.save({"state_dict": FINAL_MODEL.model.state_dict(),
                "in_dim": X_TRVAL.shape[1], "n_labels": len(TECHS),
                "hidden": list(CONFIG["MLP_HIDDEN"])}, MODEL_DIR / "model.pt")
else:
    # again, no notebook-defined class inside the artefact: store the per-label estimators
    joblib.dump({"estimators": FINAL_MODEL.models_, "priors": FINAL_MODEL.priors_,
                 "labels": FINAL_MODEL.labels}, MODEL_DIR / "model.joblib", compress=3)

mlb = MultiLabelBinarizer(classes=TECHS)
mlb.fit([TECHS])
joblib.dump(mlb, PATHS["models"] / "label_encoder.joblib")
save_json({t: float(v) for t, v in zip(TECHS, THRESHOLDS_FROZEN)}, PATHS["models"] / "thresholds.json")
save_json({"technologies": TECHS, "categories": {t: TECH_CATEGORY[t] for t in TECHS},
           "registry_version": REGISTRY_VERSION,
           "fingerprints": {t: {"category": v["category"],
                                "rules": [{k2: r[k2] for k2 in ("kind", "pattern", "strength", "sufficient")}
                                          for r in v["rules"]]}
                            for t, v in TECHNOLOGY_FINGERPRINTS.items()}},
          PATHS["models"] / "fingerprint_registry.json")

for split, rows in IDX.items():
    cols = ["url", "host", "domain", "crawl_id", "split"] + STRUCT_NUMERIC
    out = EV_DEDUP.iloc[rows][cols].copy()
    for j, t in enumerate(TECHS):
        out[f"label__{t}"] = Y[rows, j]
        out[f"mask__{t}"] = M[rows, j]
    name = {"train": "train", "val": "validation", "test": "test"}[split]
    out.to_parquet(PATHS["datasets"] / f"{name}.parquet", index=False)
    LOG.info("dataset written: %s.parquet (%d rows)", name, len(out))

print(f"model version : {MODEL_VERSION}")
for p in sorted(MODEL_DIR.glob("*")) + sorted(PATHS["datasets"].glob("*.parquet")):
    print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size / 1e6:.2f} MB)")

14:35:17 | INFO    | dataset written: train.parquet (67541 rows)
14:35:18 | INFO    | dataset written: validation.parquet (14473 rows)
14:35:18 | INFO    | dataset written: test.parquet (14473 rows)


model version : v1-1.0.0-20260817
  models/best_model/feature_bundle.joblib  (42.48 MB)
  models/best_model/model.pt  (1.92 MB)
  datasets/test.parquet  (1.87 MB)
  datasets/train.parquet  (8.12 MB)
  datasets/validation.parquet  (1.88 MB)


## 33. HTML inference

The prediction path reuses `extract_evidence_from_html` — the same function that processed every training page —
so there is no train/serve skew. Input is HTML supplied by the user; **no URL is ever fetched**.

Two caveats are surfaced directly in the output rather than buried:

1. **Header-derived technologies** (Nginx, Cloudflare, Express, ASP.NET, Vercel...) have no HTTP response headers
   available when HTML is pasted. Predictions for these rest entirely on whatever structural correlates the model
   learned, and are flagged `header-dependent`.
2. **Rule hits vs. model output** are shown side by side. Where they disagree the disagreement is informative: the
   rules are precise and brittle, the model is fuzzy and generalising.

In [54]:
# --- inference path -------------------------------------------------------
HEADER_DEPENDENT = {t for t, spec in TECHNOLOGY_FINGERPRINTS.items()
                    if all(r["kind"] in ("header", "cookie") for r in spec["rules"])}

def evidence_to_frame(rec: Dict[str, Any]) -> pd.DataFrame:
    df = pd.DataFrame([rec], columns=EVIDENCE_COLUMNS)
    for col in ["inline_js", "title", "text_sample", "url", "host", "domain", "url_ext"]:
        df[col] = df[col].fillna("").astype(str)
    for col in EVIDENCE_COLUMNS:
        if col.startswith(("n_", "url_len", "url_path", "url_has", "dom_depth",
                           "html_bytes", "text_len", "resource_entropy", "avg_class")):
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return materialise_json(df)

def hybrid_features_for(df: pd.DataFrame) -> np.ndarray:
    enc = HYBRID_BUNDLE["encoder"]
    emb = enc.encode(text_field(df, redacted=True))
    rsv = HYBRID_BUNDLE["res_svd"].transform(
        HYBRID_BUNDLE["res_vec"].transform(resource_field(df, filtered=True))).astype(np.float32)
    stc = np.nan_to_num(HYBRID_BUNDLE["scaler"].transform(struct_matrix(df)).astype(np.float32))
    return np.hstack([emb, rsv, stc]).astype(np.float32)

def predict_technologies(html: str, url: str = "https://example.invalid/pasted",
                         model=None, calibrate: bool = True,
                         min_probability: float = 0.05) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Predict a technology stack from user-supplied HTML. Never contacts the network."""
    model = model or FINAL_MODEL
    rec = extract_evidence_from_html(html, url, headers=[], source_mode="user_html")
    if rec is None:
        raise ValueError("could not parse the supplied HTML")
    df = evidence_to_frame(rec)
    X = hybrid_features_for(df)
    prob = model.predict_proba(X)
    if calibrate:
        prob = CALIBRATOR.transform(prob)
    hits = match_rules(df.iloc[0])
    rule_conf = {t: score_technology(rs)[0] for t, rs in hits.items()}
    rule_families = {t: sorted({r.kind for r in rs}) for t, rs in hits.items()}

    rows = []
    for j, tech in enumerate(TECHS):
        p = float(prob[0, j])
        thr = float(THRESHOLDS_FROZEN[j])
        rows.append({
            "technology": tech,
            "category": TECH_CATEGORY[tech],
            "probability": round(p, 4),
            "predicted": bool(p >= thr),
            "threshold": round(thr, 3),
            "rule_evidence": rule_conf.get(tech, "abstain"),
            "evidence_families": ", ".join(rule_families.get(tech, [])) or "-",
            "header_dependent": tech in HEADER_DEPENDENT,
        })
    out = pd.DataFrame(rows)
    out = out[(out["predicted"]) | (out["probability"] >= min_probability) |
              (out["rule_evidence"] != "abstain")].sort_values("probability", ascending=False)
    meta = {
        "model_version": MODEL_VERSION, "registry_version": REGISTRY_VERSION,
        "regime": PRIMARY_REGIME, "semantic_mode": HYBRID_BUNDLE["encoder"].mode,
        "page_structure": {"elements": int(rec["n_elements"]), "dom_depth": int(rec["dom_depth"]),
                           "scripts": int(rec["n_scripts"]), "stylesheets": int(rec["n_stylesheets"]),
                           "external_resource_domains": int(rec["n_external_domains"]),
                           "html_kb": round(rec["html_bytes"] / 1024, 1)},
        "note": "HTTP headers are unavailable for pasted HTML; header-derived technologies are flagged.",
    }
    return out, meta

DEMO_HTML = """<!doctype html><html lang=en><head><meta charset=utf-8>
<title>Acme - checkout</title><meta name=viewport content="width=device-width,initial-scale=1">
<link rel=stylesheet href="https://fonts.googleapis.com/css2?family=Inter"></head>
<body><div id="__next"><div class="flex items-center justify-between px-4 py-2 bg-white rounded-lg shadow-md">
<h1 class="text-xl font-semibold">Checkout</h1><p class="mt-2 text-gray-700">Pay securely.</p></div></div>
<script id="__NEXT_DATA__" type="application/json">{"props":{"pageProps":{}}}</script>
<script src="/_next/static/chunks/main-abc.js"></script>
<script src="https://js.stripe.com/v3/"></script>
<script async src="https://www.googletagmanager.com/gtag/js?id=G-DEMO"></script>
</body></html>"""

demo_pred, demo_meta = predict_technologies(DEMO_HTML)
print(json.dumps(demo_meta, indent=2))
display(demo_pred.head(15))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "model_version": "v1-1.0.0-20260817",
  "registry_version": "1.0.0",
  "regime": "B_reduced",
  "semantic_mode": "sentence_transformer",
  "page_structure": {
    "elements": 15,
    "dom_depth": 4,
    "scripts": 4,
    "stylesheets": 1,
    "external_resource_domains": 3,
    "html_kb": 0.7
  },
  "note": "HTTP headers are unavailable for pasted HTML; header-derived technologies are flagged."
}


,technology,category,probability,predicted,threshold,rule_evidence,evidence_families,header_dependent
30,Tailwind CSS,CSS/UI,0.9221,True,0.91,high,custom,False
22,Nginx,server,0.2109,False,0.54,abstain,-,True
4,Apache,server,0.1845,False,0.59,abstain,-,True
12,Google Analytics,analytics,0.1721,False,0.56,high,script_src,False
24,PHP,backend,0.1709,False,0.52,abstain,-,False
13,Google Fonts,CSS/UI,0.1646,False,0.48,high,"link_href, resource_domain",False
37,jQuery,frontend,0.1538,False,0.56,abstain,-,False
11,Font Awesome,CSS/UI,0.1452,False,0.64,abstain,-,False
19,Microsoft IIS,server,0.0852,False,0.94,abstain,-,True
6,Cloudflare,CDN,0.0799,False,0.80,abstain,-,False


## 34. Gradio inference interface

Paste HTML or upload an `.html` file and get a predicted stack. The interface has no URL input by design — adding
one would turn a passive analysis tool into an active scanner, which is outside this project's ethical scope.

In [55]:
# --- Gradio app -----------------------------------------------------------
def _read_upload(file_obj) -> str:
    if file_obj is None:
        return ""
    path = getattr(file_obj, "name", file_obj)
    try:
        with open(path, "rb") as fh:
            return fh.read().decode("utf-8", "replace")
    except Exception as exc:
        return f"<!-- could not read upload: {exc} -->"

def gradio_predict(html_text: str, uploaded, min_prob: float, use_calibration: bool):
    html = (html_text or "").strip() or _read_upload(uploaded)
    if not html or len(html) < 30:
        return pd.DataFrame([{"message": "Paste some HTML or upload an .html file."}]), "", ""
    try:
        table, meta = predict_technologies(html, model=FINAL_MODEL, calibrate=use_calibration,
                                           min_probability=float(min_prob))
    except Exception as exc:
        return pd.DataFrame([{"error": str(exc)}]), "", ""
    view = table.copy()
    view["probability"] = (view["probability"] * 100).round(1).astype(str) + "%"
    view["confidence"] = np.where(table["probability"] >= 0.8, "high",
                          np.where(table["probability"] >= 0.5, "medium", "low"))
    view = view[["technology", "category", "probability", "confidence", "predicted",
                 "rule_evidence", "evidence_families", "header_dependent"]]
    detected = table[table["predicted"]]
    summary = (f"**{len(detected)} technologies predicted** across "
               f"{detected['category'].nunique()} categories.\n\n"
               f"Stack: {', '.join(detected['technology'].head(12)) or 'none above threshold'}")
    struct = meta["page_structure"]
    info = (f"model `{meta['model_version']}` | registry `{meta['registry_version']}` | "
            f"regime `{meta['regime']}` | semantic channel `{meta['semantic_mode']}`\n\n"
            f"page: {struct['html_kb']} KB, {struct['elements']} elements, depth {struct['dom_depth']}, "
            f"{struct['scripts']} scripts, {struct['external_resource_domains']} external resource domains\n\n"
            f"_{meta['note']}_")
    return view, summary, info

def build_interface():
    import gradio as gr
    with gr.Blocks(title="Web Technology Fingerprinting", theme=gr.themes.Soft()) as demo:
        gr.Markdown(
            "# Web Technology Fingerprinting\n"
            "### Multi-label technology-stack detection learned from Common Crawl\n"
            "Paste a page's HTML (or upload a file) to predict its technology stack. "
            "**This tool never visits a URL** - it analyses only the HTML you provide. "
            "Technologies normally identified from HTTP response headers are flagged, since "
            "those headers are not present in pasted markup.")
        with gr.Row():
            with gr.Column(scale=3):
                html_in = gr.Textbox(label="Paste HTML", lines=16,
                                     placeholder="<!doctype html><html>...")
                file_in = gr.File(label="...or upload an .html file",
                                  file_types=[".html", ".htm", ".txt"])
                with gr.Row():
                    min_prob = gr.Slider(0.0, 0.5, value=0.05, step=0.01,
                                         label="minimum probability to display")
                    calib = gr.Checkbox(value=True, label="apply isotonic calibration")
                btn = gr.Button("Detect technologies", variant="primary")
                gr.Examples([[DEMO_HTML]], inputs=[html_in], label="Example page")
            with gr.Column(scale=4):
                summary_out = gr.Markdown()
                table_out = gr.Dataframe(label="Detected technologies", wrap=True)
                info_out = gr.Markdown()
        btn.click(gradio_predict, [html_in, file_in, min_prob, calib],
                  [table_out, summary_out, info_out])
        gr.Markdown(
            "---\n**Scope.** Trained on publicly archived Common Crawl pages. Passive analysis only: "
            "no scanning, no live requests, no authentication, no vulnerability testing. "
            "Predictions describe *observable evidence*, which is not the same as the underlying "
            "infrastructure - absence of evidence is not evidence of absence.")
    return demo

DEMO_APP = None
if CONFIG["LAUNCH_GRADIO"]:
    try:
        DEMO_APP = build_interface()
        DEMO_APP.launch(share=CONFIG["GRADIO_SHARE"], debug=False, quiet=True)
    except Exception as exc:
        LOG.warning("Gradio launch failed (%s) - use predict_technologies(html) directly", exc)
else:
    print("Gradio launch disabled (CONFIG['LAUNCH_GRADIO']=False). "
          "Call predict_technologies(html) directly.")

14:35:18 | INFO    | HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
14:35:18 | INFO    | HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
14:35:19 | INFO    | HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
14:35:19 | INFO    | HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
14:35:19 | INFO    | HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
14:35:19 | INFO    | HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"
14:35:19 | INFO    | HTTP Request: HEAD https://b03906f31460b8fa1e.gradio.live "HTTP/1.1 200 OK"


* Running on public URL: https://b03906f31460b8fa1e.gradio.live


## 35. Automated research report

`final_report.md` is generated from the `RESULTS` dictionary populated throughout the run, so it always reflects
the numbers this run actually produced. It answers each research question with measured values and carries the
limitations and safety sections with it.

In [56]:
# --- report generation ----------------------------------------------------
def _f(x, nd: int = 4, pct: bool = False) -> str:
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return "n/a"
        return f"{100 * float(x):.1f}%" if pct else f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def df_to_md(df: pd.DataFrame, max_rows: int = 25) -> str:
    if df is None or not len(df):
        return "_no data_\n"
    d = df.head(max_rows).copy()
    for c in d.columns:
        if d[c].dtype.kind == "f":
            d[c] = d[c].round(4)
    header = "| " + " | ".join(str(c) for c in d.columns) + " |"
    sep = "| " + " | ".join("---" for _ in d.columns) + " |"
    rows = ["| " + " | ".join(str(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join([header, sep, *rows]) + "\n"

def generate_report() -> str:
    primary = TEST_RESULTS[PRIMARY_KEY]
    abl = RESULTS.get("ablation", {})
    per = PER_TECH_TEST.dropna(subset=["f1"])
    easiest = per.head(5)[["technology", "f1", "support"]]
    hardest = per.tail(5)[["technology", "f1", "support"]]
    temporal = RESULTS.get("temporal")
    synthetic = bool(CONFIG["OFFLINE_DEMO"])

    L = []
    A = L.append
    A("# Learning Web Technology Fingerprints from Common Crawl")
    A("## Multi-Label Stack Detection, Fingerprint Ablation, and Temporal Analysis\n")
    A(f"*Generated {time.strftime('%Y-%m-%d %H:%M')} | model `{MODEL_VERSION}` | "
      f"registry `{REGISTRY_VERSION}` | crawl `{CRAWL_ID}`*\n")
    if synthetic:
        A("> **WARNING: this run used the synthetic offline corpus.** Every number below validates the "
          "pipeline, not the web. Set `CONFIG['OFFLINE_DEMO']=False` for research results.\n")

    A("## 1. Summary\n")
    A(f"A multi-label classifier was trained to predict {len(TECHS)} web technologies from page evidence "
      f"archived in Common Crawl. The corpus contains **{len(EV_DEDUP):,} deduplicated pages** across "
      f"**{EV_DEDUP['domain'].nunique():,} registrable domains**, split domain-disjointly into "
      f"{len(IDX['train']):,} / {len(IDX['val']):,} / {len(IDX['test']):,} pages.\n")
    A(f"On unseen domains the primary model reaches **micro-F1 {_f(primary['micro_f1'])}**, "
      f"**macro-F1 {_f(primary['macro_f1'])}**, micro-PR-AUC {_f(primary['micro_pr_auc'])}, "
      f"Hamming loss {_f(primary['hamming_loss'])}, subset accuracy {_f(primary['subset_accuracy'])}.\n")

    A("## 2. Data and method\n")
    A(f"- Source: `{CONFIG['SOURCE_MODE'].upper()}` records streamed from Common Crawl `{CRAWL_ID}`, "
      f"{CONFIG['MAX_ARCHIVE_FILES']} stride-sampled archive files, max "
      f"{CONFIG['MAX_PAGES_PER_DOMAIN']} pages/domain.")
    A(f"- Labels: {len(TECHNOLOGY_FINGERPRINTS)} technologies x "
      f"{sum(len(v['rules']) for v in TECHNOLOGY_FINGERPRINTS.values())} rules, confidence-aware weak "
      f"supervision. Only HIGH becomes a positive; MEDIUM/LOW cells are masked out of training and metrics "
      f"({_f(1 - float(M.mean()), pct=True)} of cells masked).")
    A(f"- {len(RESULTS.get('dropped_technologies', []))} technologies fell below "
      f"MIN_TECH_SUPPORT={MIN_SUPPORT} and were excluded: "
      f"{', '.join(RESULTS.get('dropped_technologies', [])) or 'none'}.")
    A(f"- Deduplication removed {RESULTS['deduplication']['removed_total']:,} records "
      f"({RESULTS['deduplication']['removed_pct']}%) across URL, normalised-URL, content-hash and "
      f"SimHash near-duplicate passes.")
    A(f"- Model: {ADV_NAME} over a hybrid representation "
      f"(semantic `{HYBRID_BUNDLE['encoder'].mode}` + resource-graph SVD + structural features), "
      f"trained with masked BCE on {DEVICE}.\n")

    A("## 3. Research questions\n")
    A("### RQ1 - Can ML identify web technologies from Common Crawl evidence?\n")
    A(f"Yes, with the qualification that performance varies enormously by technology. The primary model "
      f"scores micro-F1 {_f(primary['micro_f1'])} / macro-F1 {_f(primary['macro_f1'])} on "
      f"{len(test_domains):,} unseen domains, against a frequency-prior baseline of "
      f"{_f(TEST_RESULTS['frequency_prior']['micro_f1'])}. The macro/micro gap shows the aggregate is carried "
      f"by common technologies while rare ones remain hard.\n")

    A("### RQ2 - How does ML compare with direct fingerprint rules?\n")
    A(f"Complete rules score micro-F1 {_f(TEST_RESULTS['rules_full']['micro_f1'])} — but this is "
      f"**circular**: those rules generated the labels, so the figure measures self-consistency, not detection. "
      f"The meaningful comparison strips the decisive rules from both sides: rules-reduced reaches "
      f"{_f(TEST_RESULTS['rules_reduced']['micro_f1'])} while the learned model in the fingerprint-reduced "
      f"regime reaches {_f(TEST_RESULTS['logreg_B_reduced']['micro_f1'])}. "
      f"The practical difference is that rules degrade to silence when a signature is absent, whereas the model "
      f"still emits a graded probability from indirect evidence.\n")

    A("### RQ3 - How much performance disappears when obvious fingerprints are removed?\n")
    A(f"Removing the label-generating signatures costs **{_f(abl.get('A_minus_B_micro_f1'))} micro-F1** "
      f"({abl.get('A_relative_drop_pct')}% of regime A). Removing all lexical content costs a further "
      f"{_f(abl.get('B_minus_C_micro_f1'))}. Structure-only (regime C) still scores "
      f"{_f(ABLATION['C_structure']['micro_f1'])} against a prior of {_f(abl.get('prior_micro_f1'))}, "
      f"which is the clearest evidence in this project that something genuinely generalisable was learned "
      f"rather than a regex being memorised.\n")
    A(df_to_md(pd.DataFrame(abl.get("table", []))))

    A("### RQ4 - Does the model generalise to unseen domains?\n")
    A(f"This is the default evaluation protocol: all {len(test_domains):,} test domains are disjoint from "
      f"train and validation, verified by assertion. Reported test performance is therefore already an "
      f"unseen-domain estimate.\n")

    A("### RQ5 - Does it generalise across time?\n")
    if temporal:
        A(f"Trained on crawls up to {temporal['val_year'] - 1} and tested on {temporal['test_year']}, "
          f"micro-F1 falls by **{_f(temporal['micro_f1_drift'])}** "
          f"({temporal['relative_drift_pct']:+.1f}%) relative to the same model family evaluated "
          f"domain-disjointly. {temporal['rows_dropped_for_disjointness']:,} rows were dropped to keep "
          f"domains disjoint across time, without which this would silently re-measure memorisation.\n")
        A(df_to_md(pd.DataFrame(temporal.get("table", []))))
    else:
        A("Temporal evaluation did not run (disabled, or too few crawls ingested).\n")

    A("### RQ6 - Which technologies are easiest and hardest to identify?\n")
    A("Easiest:\n"); A(df_to_md(easiest))
    A("Hardest:\n"); A(df_to_md(hardest))
    A("Difficulty tracks three things: how many independent evidence families a technology emits, whether its "
      "label derives from HTTP headers (invisible in the reduced regime), and its support.\n")

    A("### RQ7 - Which technologies most commonly co-occur?\n")
    A(df_to_md(pd.DataFrame(RESULTS.get("cooccurrence_top_jaccard", []))[
        ["tech_a", "tech_b", "jaccard", "p_b_given_a", "lift"]] if RESULTS.get("cooccurrence_top_jaccard")
        else pd.DataFrame(), 15))
    if RESULTS.get("stack_clusters"):
        A(f"\nClustering pages on their technology vectors yields "
          f"{RESULTS['stack_clusters']['k']} stack archetypes "
          f"(silhouette {RESULTS['stack_clusters']['silhouette']}):\n")
        A(df_to_md(pd.DataFrame(RESULTS["stack_clusters"]["clusters"])[
            ["archetype", "pages", "share", "mean_stack_size"]]))

    A("## 4. Full results\n")
    A("### Test-set model comparison\n"); A(df_to_md(test_tbl))
    A("### Per-technology (primary model)\n")
    A(df_to_md(PER_TECH_TEST[["technology", "category", "precision", "recall", "f1",
                              "pr_auc", "support", "threshold"]], 60))
    A("### Threshold tuning\n")
    A(f"Validation micro-F1 at 0.5: {_f(RESULTS['threshold_tuning']['default']['micro_f1'])}; "
      f"with per-technology tuned thresholds (mean {RESULTS['threshold_tuning']['mean_threshold']}): "
      f"{_f(RESULTS['threshold_tuning']['tuned']['micro_f1'])}. Thresholds were frozen before test.\n")
    A("### Calibration\n")
    A(f"Mean ECE {RESULTS['calibration']['mean_ece_raw']} raw, "
      f"{RESULTS['calibration']['mean_ece_isotonic']} after isotonic calibration fitted on validation.\n")
    A("### Error analysis\n")
    A(df_to_md(ERROR_TBL[["technology", "tp", "fp", "fn", "fn_rate", "likely_cause"]], 20))
    if len(CONFUSION):
        A("### Technology confusion\n"); A(df_to_md(CONFUSION, 12))
    if len(ADOPTION):
        A("### Adoption over time\n")
        A("Prevalence within this Common Crawl-derived sample. Not a census of the web.\n")
        if RESULTS.get("adoption"):
            A(df_to_md(pd.DataFrame(RESULTS["adoption"]["rising"]), 8))
    if len(TRANSITIONS):
        A("### Observed evidence transitions\n"); A(df_to_md(TRANSITIONS, 12))

    A("## 5. Limitations\n")
    A(LIMITATIONS_MD)
    A("## 6. Safety and ethical scope\n")
    A(SAFETY_MD)
    return "\n".join(L)

LIMITATIONS_MD = r"""
**Sampling.** Common Crawl is a large but non-uniform sample of the public web. Its host selection, per-host page
caps and frontier policy change between crawls, and this project adds its own per-domain cap and archive-file
stride sample on top. Every prevalence figure describes *this sample*, never the Internet. A statement of the form
"X% of the web uses React" cannot be supported by this data; "React evidence appeared on X% of pages in this
sample" can.

**Weak supervision.** Labels come from rules, so the model's ceiling is the rules' accuracy. Where a rule is wrong,
the model is trained to be wrong in the same direction, and the metrics — computed against those same labels —
cannot see it. The confidence ladder and trust mask reduce but do not remove this.

**Negatives are unverified.** A cell is a trusted negative when *no* evidence was found, which conflates "not
present" with "not visible". Recall is therefore optimistic for technologies whose markers are easy to strip or
proxy away, and the true false-negative rate is higher than the tables report.

**Fingerprint ambiguity.** Some signals are shared across technologies (`XSRF-TOKEN` spans several frameworks;
`#app` and `#root` are conventions, not evidence). These are marked `weak` and rarely produce a HIGH label alone,
but they still contribute to confusion between related technologies.

**Fingerprint redaction is approximate.** Regime B removes the matched signature, not everything correlated with
it. Permalink shapes, asset-path conventions and template idioms survive redaction, so regime B retains some
residual leakage. Regime C is the clean condition and should be read as the conservative bound.

**No JavaScript execution.** Common Crawl captures pre-hydration HTML. Any technology that only manifests after
client-side rendering is systematically invisible — the single largest source of false negatives here, and the
reason SPA-framework recall in particular should be read as a floor.

**Incomplete captures.** Truncated responses, 200-with-error-body pages, bot-mitigation interstitials and
region-varying content all appear in the corpus. Pages over the byte cap are truncated, so evidence late in a very
large document can be missed.

**Header availability asymmetry.** Server, CDN and backend labels derive from HTTP response headers present during
training but absent at inference time for pasted HTML. Those predictions rest entirely on learned structural
correlates and are flagged in the interface.

**Temporal drift.** Vendors change their emitted markers. A model trained on older crawls sees fingerprints that
have since changed, and the registry itself is written against present-day conventions, which may under-detect
older deployments and slightly distort the adoption trends in both directions.

**Rare technologies.** Anything below the support threshold was dropped rather than modelled badly. Their absence
from the results is a modelling decision, not a claim about their prevalence.

**Evidence is not infrastructure.** Everything here measures *observable client-side evidence*. A Cloudflare header
means traffic passed through Cloudflare, not that the origin is hosted there; a WordPress marker on one page does
not make the whole domain WordPress. Observed evidence and underlying infrastructure are different things, and only
the former is measured.
"""

SAFETY_MD = r"""
**Data provenance.** All training data comes from Common Crawl, a public archive of already-published web pages,
retrieved over HTTPS from its official distribution endpoint. No other host was contacted for page data.

**No active reconnaissance.** This project performs no live crawling, no port scanning, no host discovery, no
service enumeration and no probing of any kind. Nothing in the notebook sends a request to a third-party website.

**No exploitation or access control interaction.** No vulnerability testing, no authentication, no credential
testing, no brute forcing, no access-control bypass. Software *versions* are deliberately not parsed or stored,
which removes the natural bridge from fingerprinting to vulnerability mapping.

**No secret discovery.** The extractor does not search for API keys, tokens or credentials. `Set-Cookie` values
are discarded at parse time and only cookie *names* are retained; request headers are never captured; only a
whitelist of response headers is stored.

**Data minimisation.** Only bounded derived evidence is persisted — counts, tag histograms, resource domains,
truncated text and truncated inline-script samples. Raw HTML is never written to disk.

**Inference is passive by construction.** The interface accepts HTML the user supplies and has no URL field. This
is a deliberate design constraint: adding one would convert a passive analysis tool into an active scanner.

**Responsible interpretation.** Outputs describe observable evidence with calibrated uncertainty. They are not
assertions about a site's infrastructure and should not be used to target any system.
"""

REPORT = generate_report()
(PATHS["reports"] / "final_report.md").write_text(REPORT, encoding="utf-8")
print(f"report written: {PATHS['reports'] / 'final_report.md'} ({len(REPORT):,} chars)")
print("\n" + REPORT[:2500] + "\n...[truncated]...")

14:35:19 | INFO    | HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


report written: /content/drive/MyDrive/common_crawl_technology_fingerprinting/reports/final_report.md (21,801 chars)

# Learning Web Technology Fingerprints from Common Crawl
## Multi-Label Stack Detection, Fingerprint Ablation, and Temporal Analysis

*Generated 2026-08-17 14:35 | model `v1-1.0.0-20260817` | registry `1.0.0` | crawl `CC-MAIN-2026-30`*

## 1. Summary

A multi-label classifier was trained to predict 39 web technologies from page evidence archived in Common Crawl. The corpus contains **96,487 deduplicated pages** across **86,554 registrable domains**, split domain-disjointly into 67,541 / 14,473 / 14,473 pages.

On unseen domains the primary model reaches **micro-F1 0.7189**, **macro-F1 0.6059**, micro-PR-AUC 0.7894, Hamming loss 0.0539, subset accuracy 0.1973.

## 2. Data and method

- Source: `WARC` records streamed from Common Crawl `CC-MAIN-2026-30`, 12 stride-sampled archive files, max 3 pages/domain.
- Labels: 45 technologies x 148 rules, confidence-aware weak super

## 36. Reproducibility metadata

Everything needed to re-run or audit this experiment, written to `experiment_summary.json`.

In [57]:
# --- experiment record ----------------------------------------------------
import platform
def _ver(mod: str) -> Optional[str]:
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return None

EXPERIMENT_SUMMARY = {
    "project": CONFIG["PROJECT_NAME"],
    "model_version": MODEL_VERSION,
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "duration_minutes": round((time.time() - RUN["started_at"]) / 60, 1),
    "common_crawl": {
        "primary_crawl": CRAWL_ID,
        "temporal_crawls": TEMPORAL_CRAWLS,
        "source_mode": CONFIG["SOURCE_MODE"],
        "archive_files": CONFIG["MAX_ARCHIVE_FILES"],
        "n_available_indexes": RUN.get("n_available_crawls"),
        "synthetic_offline_corpus": bool(CONFIG["OFFLINE_DEMO"]),
    },
    "dataset": {
        "pages_ingested": RUN.get("ingested_pages"),
        "pages_after_dedup": int(len(EV_DEDUP)),
        "domains": int(EV_DEDUP["domain"].nunique()),
        "deduplication": RESULTS.get("deduplication"),
        "split": RESULTS.get("split_stats"),
        "technologies_modelled": TECHS,
        "technologies_dropped": RESULTS.get("dropped_technologies"),
        "masked_cell_fraction": round(float(1 - M.mean()), 4),
    },
    "labels": {"registry_version": REGISTRY_VERSION,
               "n_technologies": len(TECHNOLOGY_FINGERPRINTS),
               "n_rules": len(COMPILED_RULES),
               "positive_confidence": CONFIG["POSITIVE_CONFIDENCE"],
               "implications_applied": CONFIG["APPLY_IMPLICATIONS"],
               "implication_confidence": CONFIG["IMPLICATION_CONFIDENCE"]},
    "model": {"architecture": ADV_NAME, "regime": PRIMARY_REGIME,
              "semantic_channel": HYBRID_BUNDLE["encoder"].mode,
              "embedding_model": CONFIG["EMBEDDING_MODEL"],
              "feature_dims": HYBRID_BUNDLE["dims"],
              "hidden": list(CONFIG["MLP_HIDDEN"]), "epochs": CONFIG["MLP_EPOCHS"],
              "batch_size": CONFIG["MLP_BATCH"], "lr": CONFIG["MLP_LR"],
              "training_seconds": RUN.get("advanced_training_seconds")},
    "results": {"test": RESULTS.get("test_results"), "ablation": RESULTS.get("ablation"),
                "temporal": RESULTS.get("temporal"), "calibration_ece": RESULTS.get("calibration", {}).get("mean_ece_raw")},
    "environment": {"python": sys.version.split()[0], "platform": platform.platform(),
                    "device": DEVICE, "hardware": HW,
                    "packages": {m: _ver(m) for m in
                                 ["numpy", "pandas", "sklearn", "scipy", "torch", "pyarrow",
                                  "lxml", "warcio", "gradio", "sentence_transformers",
                                  "matplotlib", "networkx", "joblib"]}},
    "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CONFIG.items()},
    "random_seed": CONFIG["RANDOM_SEED"],
}
save_json(EXPERIMENT_SUMMARY, PATHS["reports"] / "experiment_summary.json")
save_json(EXPERIMENT_SUMMARY, ROOT / "experiment_summary.json")
print(json.dumps({k: EXPERIMENT_SUMMARY[k] for k in
                  ("project", "model_version", "duration_minutes", "common_crawl")}, indent=2))
print("\nartefacts:")
for sub in ("datasets", "models", "reports", "figures"):
    files = sorted(PATHS[sub].rglob("*"))
    print(f"  {sub}/ ({len([f for f in files if f.is_file()])} files)")
    for f in [f for f in files if f.is_file()][:12]:
        print(f"      {f.relative_to(PATHS[sub])}")

{
  "project": "common_crawl_technology_fingerprinting",
  "model_version": "v1-1.0.0-20260817",
  "duration_minutes": 79.7,
  "common_crawl": {
    "primary_crawl": "CC-MAIN-2026-30",
    "temporal_crawls": [
      "CC-MAIN-2013-48",
      "CC-MAIN-2017-51",
      "CC-MAIN-2022-49",
      "CC-MAIN-2026-30"
    ],
    "source_mode": "warc",
    "archive_files": 12,
    "n_available_indexes": 126,
    "synthetic_offline_corpus": false
  }
}

artefacts:
  datasets/ (3 files)
      test.parquet
      train.parquet
      validation.parquet
  models/ (5 files)
      best_model/feature_bundle.joblib
      best_model/model.pt
      fingerprint_registry.json
      label_encoder.joblib
      thresholds.json
  reports/ (23 files)
      ablation_per_technology.csv
      ablation_results.csv
      calibration.csv
      channel_ablation.csv
      channel_attribution.csv
      classification_results.csv
      deduplication.json
      error_analysis.csv
      error_examples.csv
      experiment_summa

## 37. Limitations

The full limitations discussion is generated into `reports/final_report.md` (and printed below) so it travels with
the results rather than living only in the notebook. In short:

* Common Crawl is a **sample**, not a census, and its composition shifts between crawls.
* Labels are **weak supervision**; the model inherits the rules' errors and the metrics cannot see them.
* Trusted negatives really mean "no visible evidence", so recall estimates are optimistic.
* Regime B redaction is approximate; regime C is the clean bound.
* **No JavaScript is executed**, so runtime-injected technologies are systematically invisible.
* Header-derived labels have no header available at inference time for pasted HTML.
* Observed evidence is not the same thing as underlying infrastructure.

In [58]:
print(LIMITATIONS_MD)
print("\n" + "=" * 78)
print(SAFETY_MD)


**Sampling.** Common Crawl is a large but non-uniform sample of the public web. Its host selection, per-host page
caps and frontier policy change between crawls, and this project adds its own per-domain cap and archive-file
stride sample on top. Every prevalence figure describes *this sample*, never the Internet. A statement of the form
"X% of the web uses React" cannot be supported by this data; "React evidence appeared on X% of pages in this
sample" can.

**Weak supervision.** Labels come from rules, so the model's ceiling is the rules' accuracy. Where a rule is wrong,
the model is trained to be wrong in the same direction, and the metrics — computed against those same labels —
cannot see it. The confidence ladder and trust mask reduce but do not remove this.

**Negatives are unverified.** A cell is a trusted negative when *no* evidence was found, which conflates "not
present" with "not visible". Recall is therefore optimistic for technologies whose markers are easy to strip or
prox

## 38. Final conclusions

### What was built

An end-to-end, fully automated pipeline from Common Crawl discovery to a served technology-stack predictor:
broad-web streaming ingestion, a versioned fingerprint registry, confidence-aware weak supervision with a trust
mask, four-level deduplication, a domain-disjoint split, rule and ML baselines, a hybrid neural multi-label model,
three-regime leakage ablations, per-label threshold tuning and calibration, error and co-occurrence analysis,
stack clustering, temporal generalisation, adoption and transition analysis, and an HTML-only inference interface.

### What was learned

The result worth carrying away is the **decomposition in section 20**. Technology detection benchmarks that train
and evaluate on rule-generated labels while feeding the model those same rules' signals are measuring their own
regex. Separating regime A from B from C turns that circularity from a hidden flaw into a reported quantity, and
what remains in regimes B and C — detection above the prior with the signature removed — is the part that
represents genuine learning.

The secondary finding is that **structure carries real signal**. Regime C has no URLs, no domains, no text and no
headers, only DOM shape and resource-graph statistics, and it still separates several technologies well above
chance. Frameworks impose a measurable morphology on the documents they emit.

### Honest framing

This system detects *evidence*, not infrastructure. It cannot see through JavaScript rendering, its negatives are
unverified, and its prevalence figures describe a Common Crawl sample rather than the web. Those constraints are
in the report, in the interface, and in the model card — not because they are disclaimers, but because they are
the difference between a measurement and a claim.

---

### Explicit delta against the prior Common Crawl retrieval project

| Dimension | Previous project (retrieval) | This project (fingerprinting) |
|---|---|---|
| Task | Information retrieval + single-label page categorisation | **Multi-label technology-stack detection** |
| Unit of prediction | Document relevance / one category per page | **~40 independent technology probabilities per page** |
| Input to the system | A user query | **HTML evidence — no query exists anywhere in this notebook** |
| CC access pattern | CDX/operator lookup, WARC byte-range fetch, query-driven sampling | **Sequential WARC/WAT streaming, stride-sampled files, per-domain caps for breadth** |
| Sampling goal | Documents relevant to a query | **Domain-representative coverage of the public web** |
| Labels | Hand-designed content categories (`general_web`, `government_public`, ...) | **Registry-driven fingerprint rules with a confidence ladder and trust mask** |
| Core algorithms | BM25, TF-IDF retrieval, semantic retrieval, ranking fusion | **Masked-BCE multi-label network, masked one-vs-rest, rule baselines** |
| Evaluation | NDCG, P@K, MRR (ranking metrics) | **micro/macro P-R-F1, PR-AUC, Hamming loss, subset accuracy, per-label ECE** |
| Split protocol | Document-level | **Domain-disjoint, plus an independent temporal protocol** |
| Signature research contribution | Ranking quality | **Fingerprint-leakage ablation (A/B/C) quantifying label circularity** |
| Analyses | Query intent, ranking quality | **Co-occurrence, stack clustering, adoption trends, transitions, calibration** |
| Interface | Search UI with query safety guards | **HTML-only inference UI that deliberately cannot visit a URL** |

No component of the retrieval architecture is reused: there is no query parser, no BM25, no relevance ranking, no
NDCG/MRR, and no search interface anywhere in this notebook.

In [59]:
# --- run summary ----------------------------------------------------------
print("=" * 78)
print("RUN COMPLETE")
print("=" * 78)
print(f"crawl              : {CRAWL_ID}"
      f"{'  [SYNTHETIC OFFLINE CORPUS]' if CONFIG['OFFLINE_DEMO'] else ''}")
print(f"pages / domains    : {len(EV_DEDUP):,} / {EV_DEDUP['domain'].nunique():,}")
print(f"technologies       : {len(TECHS)} modelled of {len(TECHNOLOGY_FINGERPRINTS)} in registry")
print(f"test micro-F1      : {TEST_RESULTS[PRIMARY_KEY]['micro_f1']:.4f} "
      f"(macro {TEST_RESULTS[PRIMARY_KEY]['macro_f1']:.4f}) on {len(test_domains):,} unseen domains")
print(f"fingerprint echo   : {RESULTS['ablation']['A_minus_B_micro_f1']:+.4f} micro-F1 "
      f"({RESULTS['ablation']['A_relative_drop_pct']}% of regime A)")
print(f"structure-only     : {ABLATION['C_structure']['micro_f1']:.4f} vs prior "
      f"{RESULTS['ablation']['prior_micro_f1']:.4f}")
if RESULTS.get("temporal"):
    print(f"temporal drift     : {RESULTS['temporal']['micro_f1_drift']:+.4f} micro-F1")
print(f"runtime            : {(time.time() - RUN['started_at']) / 60:.1f} min on {DEVICE}")
print(f"artefacts          : {ROOT}")
print("=" * 78)

RUN COMPLETE
crawl              : CC-MAIN-2026-30
pages / domains    : 96,487 / 86,554
technologies       : 39 modelled of 45 in registry
test micro-F1      : 0.7189 (macro 0.6059) on 13,001 unseen domains
fingerprint echo   : +0.2739 micro-F1 (27.5% of regime A)
structure-only     : 0.3956 vs prior 0.3674
temporal drift     : +0.1667 micro-F1
runtime            : 79.7 min on cuda
artefacts          : /content/drive/MyDrive/common_crawl_technology_fingerprinting
